<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/GES_Stage6C_Cell_6C_4H0_Alternative_Outcome_Secondary_Drift_Materialization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# STAGE 6C STEP 4H — CELL 6C-4H0
# ALTERNATIVE-PRIMARY AND SECONDARY EVIDENCE-DRIFT RESULT-CATEGORY MATERIALIZATION
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import re
import sys
import time

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import scipy
from scipy.sparse import csr_matrix, vstack
import sklearn
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. LOCKED INPUTS, ANALYSIS SPECIFICATION, AND OUTPUT LOCATIONS
# --------------------------------------------------------------------------------------------------

NOTEBOOK_NAME = (
    "GES_Stage6C_Cell_6C_4H0_Alternative_Outcome_Secondary_Drift_"
    "Materialization.ipynb"
)
ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

SOURCE = ROOT / (
    "data_processed/stage6_temporal_validation/"
    "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
SOURCE_SHA = "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"

PRIOR_MANIFEST = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4g0_exact_link_sensitivity_materialization_v1/"
    "stage6c_4g0_exact_link_sensitivity_manifest_v1.json"
)
PRIOR_MANIFEST_SHA = "f9586e0609f76f9adda52e153dfcb7800e77b0840a1eb076c44936777ba8611f"

SEED = 42
N_BOOT = 2_000
BOOTSTRAP_BATCH_SIZE = 50
EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_PRIMARY_EVENTS = 6_485
EXPECTED_PRIMARY_NEGATIVES = 60_151

PRIMARY_OUTCOME = "primary_future_instability"
T0_STARS = "t0_aggregate_review_stars"
T1_STARS = "t1_aggregate_review_stars"
STAR_DELTA = "secondary_review_star_delta"
STATUS_CHANGED = "secondary_review_status_changed"
MATERIAL_CHANGE = "event_material_clinical_group_change"
NEW_CONFLICT = "event_new_unresolved_conflict_at_t1"
PRIOR_RESOLUTION = "event_prior_conflict_resolved_to_material_group"

MODELS = {
    "Full GES": "full_ges_instability_risk_t0",
    "No-star GES": "no_star_ges_instability_risk_t0",
    "Review stars": "review_stars_instability_risk",
    "Combined metadata": "combined_metadata_instability_risk",
    "Conflict": "conflict_instability_risk",
    "Recency": "recency_instability_risk",
    "Submitter support": "submitter_instability_risk",
    "Classification entropy": "entropy_instability_risk",
    "Additive risk": "additive_instability_risk",
}
PRINCIPAL = ["No-star GES", "Review stars", "Combined metadata"]
SECONDARY = [
    "Conflict", "Recency", "Submitter support", "Classification entropy", "Additive risk"
]
RISK_FRACTIONS = [0.05, 0.10, 0.20]

OUTCOME_SPEC = {
    "strict_material_instability": {
        "display": "Strict material instability",
        "role": "alternative_primary_sensitivity",
        "expected_events": 1_702,
        "definition": (
            "material clinical-group change OR material prior-conflict resolution; "
            "new unresolved conflict excluded"
        ),
        "independence_note": (
            "Alternative clinical-instability definition derived from frozen primary components."
        ),
    },
    "conflict_transition_instability": {
        "display": "Conflict-transition instability",
        "role": "alternative_primary_sensitivity",
        "expected_events": 5_086,
        "definition": "new unresolved conflict OR material prior-conflict resolution",
        "independence_note": (
            "Alternative conflict-dynamics definition derived from frozen primary components."
        ),
    },
    "expanded_primary_or_review_star_change": {
        "display": "Expanded primary-or-star-drift outcome",
        "role": "alternative_primary_sensitivity",
        "expected_events": 14_437,
        "definition": "primary future instability OR any review-star change",
        "independence_note": (
            "Expanded evidence-drift definition; not independent of review metadata."
        ),
    },
    "any_review_star_change": {
        "display": "Any review-star change",
        "role": "secondary_evidence_drift",
        "expected_events": 9_946,
        "definition": "T1 review stars differ from T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "review_star_increase": {
        "display": "Review-star increase",
        "role": "secondary_evidence_drift",
        "expected_events": 3_675,
        "definition": "T1 review stars are greater than T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "review_star_decrease": {
        "display": "Review-star decrease",
        "role": "secondary_evidence_drift",
        "expected_events": 6_271,
        "definition": "T1 review stars are lower than T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "any_review_status_change": {
        "display": "Any review-status change",
        "role": "secondary_evidence_drift",
        "expected_events": 10_946,
        "definition": "frozen secondary_review_status_changed equals True",
        "independence_note": (
            "Exploratory only because review confidence contributes to full GES."
        ),
    },
    "new_expert_panel_involvement": {
        "display": "New expert-panel involvement",
        "role": "secondary_evidence_drift",
        "expected_events": 8,
        "definition": "T0 review stars < 3 and T1 review stars >= 3",
        "independence_note": (
            "Exploratory only; eight events and not an independent validation endpoint."
        ),
    },
}
OUTCOME_KEYS = list(OUTCOME_SPEC)
ALT_OUTCOMES = [
    k for k in OUTCOME_KEYS if OUTCOME_SPEC[k]["role"] == "alternative_primary_sensitivity"
]
DRIFT_OUTCOMES = [
    k for k in OUTCOME_KEYS if OUTCOME_SPEC[k]["role"] == "secondary_evidence_drift"
]

TABLE_DIR = ROOT / (
    "outputs/tables/stage6_temporal_validation/"
    "stage6c_4h0_alternative_outcome_secondary_drift_materialization_v1"
)
QC_DIR = ROOT / (
    "outputs/quality_checks/stage6_temporal_validation/"
    "stage6c_4h0_alternative_outcome_secondary_drift_materialization_v1"
)
MANIFEST_DIR = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4h0_alternative_outcome_secondary_drift_materialization_v1"
)
for directory in (TABLE_DIR, QC_DIR, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

P = {
    "accounting": TABLE_DIR / "stage6c_alternative_outcome_accounting_v1.csv",
    "star_delta": TABLE_DIR / "stage6c_review_star_delta_inventory_v1.csv",
    "star_crosstab": TABLE_DIR / "stage6c_t0_t1_review_star_crosstab_v1.csv",
    "points": TABLE_DIR / "stage6c_alternative_outcome_point_estimates_v1.csv",
    "model_replicates": TABLE_DIR / "stage6c_alternative_outcome_model_bootstrap_replicates_v1.parquet",
    "enrichment_replicates": TABLE_DIR / "stage6c_alternative_outcome_enrichment_bootstrap_replicates_v1.parquet",
    "validity": TABLE_DIR / "stage6c_alternative_outcome_bootstrap_validity_v1.csv",
    "intervals": TABLE_DIR / "stage6c_alternative_outcome_model_bootstrap_intervals_v1.csv",
    "paired": TABLE_DIR / "stage6c_alternative_outcome_paired_inference_v1.csv",
    "multiplicity": TABLE_DIR / "stage6c_alternative_outcome_multiplicity_v1.csv",
    "enrichment": TABLE_DIR / "stage6c_alternative_outcome_enrichment_intervals_v1.csv",
    "sparse_audit": TABLE_DIR / "stage6c_alternative_outcome_sparse_outcome_audit_v1.csv",
    "historical": TABLE_DIR / "stage6c_alternative_outcome_historical_results_v1.csv",
    "concordance": TABLE_DIR / "stage6c_alternative_outcome_historical_vs_reproduced_concordance_v1.csv",
    "qc": QC_DIR / "stage6c_4h0_alternative_outcome_secondary_drift_qc_v1.json",
    "manifest": MANIFEST_DIR / "stage6c_4h0_alternative_outcome_secondary_drift_manifest_v1.json",
}

if P["manifest"].exists():
    CREATED_UTC = json.loads(P["manifest"].read_text(encoding="utf-8"))["created_utc"]
elif P["qc"].exists():
    CREATED_UTC = json.loads(P["qc"].read_text(encoding="utf-8"))["created_utc"]
else:
    CREATED_UTC = datetime.now(timezone.utc).isoformat()


# --------------------------------------------------------------------------------------------------
# 2. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def native(value):
    if isinstance(value, dict):
        return {str(k): native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [native(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return native(value.tolist())
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is pd.NA:
        return None
    return value


def stable_write_bytes(path: Path, payload: bytes) -> str:
    """Create atomically; on rerun accept only byte-identical content."""
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    temporary.write_bytes(payload)
    new_hash = sha(temporary)
    if path.exists():
        if sha(path) != new_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Refusing to overwrite nonidentical artifact: {path}")
        temporary.unlink(missing_ok=True)
    else:
        os.replace(temporary, path)
    return sha(path)


def write_csv(path: Path, frame: pd.DataFrame) -> str:
    payload = frame.to_csv(
        index=False, lineterminator="\n", float_format="%.12g"
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_json(path: Path, obj) -> str:
    payload = (
        json.dumps(
            native(obj), indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False
        )
        + "\n"
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_parquet(path: Path, frame: pd.DataFrame) -> str:
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    frame.to_parquet(temporary, index=False, compression="zstd", engine="pyarrow")
    if path.exists():
        old = pd.read_parquet(path)
        new = pd.read_parquet(temporary)
        pd.testing.assert_frame_equal(old, new, check_dtype=True, check_exact=True)
        temporary.unlink()
    else:
        os.replace(temporary, path)
    return sha(path)


def sidecar(path: Path) -> Path:
    path = Path(path)
    output = path.with_name(path.name + ".sha256")
    stable_write_bytes(output, f"{sha(path)}  {path.name}\n".encode("utf-8"))
    return output


def sidecar_ok(path: Path) -> bool:
    path = Path(path)
    output = path.with_name(path.name + ".sha256")
    return output.exists() and output.read_text(encoding="utf-8").strip().split()[0] == sha(path)


def slug(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def binary_column(frame: pd.DataFrame, column: str) -> np.ndarray:
    if column not in frame:
        raise KeyError(f"Missing required binary column: {column}")
    series = frame[column]
    if pd.api.types.is_bool_dtype(series):
        if series.isna().any():
            raise RuntimeError(f"Binary column contains missing values: {column}")
        return series.to_numpy(dtype=np.int8)
    numeric = pd.to_numeric(series, errors="raise")
    if numeric.isna().any() or not set(numeric.unique()).issubset({0, 1, 0.0, 1.0}):
        raise RuntimeError(f"Column is not complete binary: {column}")
    return numeric.to_numpy(dtype=np.int8)


def ci(values) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lower, upper = np.percentile(values, [2.5, 97.5])
    return float(lower), float(upper)


def sign_p(values) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan
    lower_tail = (np.count_nonzero(values <= 0.0) + 1) / (len(values) + 1)
    upper_tail = (np.count_nonzero(values >= 0.0) + 1) / (len(values) + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def holm(values) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    adjusted = np.full(len(values), np.nan, dtype=float)
    valid_positions = np.flatnonzero(np.isfinite(values))
    if len(valid_positions) == 0:
        return adjusted
    valid = values[valid_positions]
    order = np.argsort(valid)
    running_max = 0.0
    m = len(valid)
    for rank, ordered_position in enumerate(order):
        original_position = valid_positions[ordered_position]
        running_max = max(running_max, (m - rank) * valid[ordered_position])
        adjusted[original_position] = min(1.0, running_max)
    return adjusted


def interval_status(lower: float, upper: float, positive: str, negative: str) -> str:
    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"
    if lower > 0.0:
        return positive
    if upper < 0.0:
        return negative
    return "interval_includes_null"


# --------------------------------------------------------------------------------------------------
# 3. VERIFY FROZEN INPUTS AND RECONSTRUCT THE EIGHT PRESPECIFIED OUTCOMES
# --------------------------------------------------------------------------------------------------

if not SOURCE.exists():
    raise FileNotFoundError(SOURCE)
if not PRIOR_MANIFEST.exists():
    raise FileNotFoundError(PRIOR_MANIFEST)
if sha(SOURCE) != SOURCE_SHA:
    raise RuntimeError("Stage 6B source SHA-256 mismatch.")
if not sidecar_ok(SOURCE):
    raise RuntimeError("Stage 6B source sidecar verification failed.")
if sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA:
    raise RuntimeError("Prior Cell 6C-4G0 manifest SHA-256 mismatch.")
if not sidecar_ok(PRIOR_MANIFEST):
    raise RuntimeError("Prior Cell 6C-4G0 manifest sidecar verification failed.")

metadata = pq.ParquetFile(SOURCE).metadata
if (metadata.num_rows, metadata.num_columns) != (EXPECTED_ROWS, EXPECTED_COLUMNS):
    raise RuntimeError(
        f"Unexpected Stage 6B dimensions: {(metadata.num_rows, metadata.num_columns)}"
    )

required_columns = [
    PRIMARY_OUTCOME,
    T0_STARS,
    T1_STARS,
    STAR_DELTA,
    STATUS_CHANGED,
    MATERIAL_CHANGE,
    NEW_CONFLICT,
    PRIOR_RESOLUTION,
    *MODELS.values(),
]
frame = pd.read_parquet(SOURCE)
missing_columns = [column for column in required_columns if column not in frame]
if missing_columns:
    raise RuntimeError(f"Missing required columns: {missing_columns}")

primary = binary_column(frame, PRIMARY_OUTCOME)
material = binary_column(frame, MATERIAL_CHANGE)
new_conflict = binary_column(frame, NEW_CONFLICT)
prior_resolution = binary_column(frame, PRIOR_RESOLUTION)
status_changed = binary_column(frame, STATUS_CHANGED)

if (int(primary.sum()), int(len(primary) - primary.sum())) != (
    EXPECTED_PRIMARY_EVENTS,
    EXPECTED_PRIMARY_NEGATIVES,
):
    raise RuntimeError("Frozen primary-outcome accounting failed.")

stars_t0 = pd.to_numeric(frame[T0_STARS], errors="raise").to_numpy(dtype=float)
stars_t1 = pd.to_numeric(frame[T1_STARS], errors="raise").to_numpy(dtype=float)
stored_delta = pd.to_numeric(frame[STAR_DELTA], errors="raise").to_numpy(dtype=float)
if not np.isfinite(stars_t0).all() or not np.isfinite(stars_t1).all():
    raise RuntimeError("Review-star fields contain nonfinite values.")
calculated_delta = stars_t1 - stars_t0
if not np.array_equal(stored_delta, calculated_delta):
    raise RuntimeError("Frozen review-star delta does not equal T1 minus T0.")

outcomes = {
    "strict_material_instability": np.logical_or(material == 1, prior_resolution == 1).astype(np.int8),
    "conflict_transition_instability": np.logical_or(new_conflict == 1, prior_resolution == 1).astype(np.int8),
    "expanded_primary_or_review_star_change": np.logical_or(primary == 1, calculated_delta != 0).astype(np.int8),
    "any_review_star_change": (calculated_delta != 0).astype(np.int8),
    "review_star_increase": (calculated_delta > 0).astype(np.int8),
    "review_star_decrease": (calculated_delta < 0).astype(np.int8),
    "any_review_status_change": status_changed.astype(np.int8),
    "new_expert_panel_involvement": np.logical_and(stars_t0 < 3, stars_t1 >= 3).astype(np.int8),
}

for outcome_key, outcome in outcomes.items():
    observed_events = int(outcome.sum())
    expected_events = OUTCOME_SPEC[outcome_key]["expected_events"]
    if observed_events != expected_events:
        raise RuntimeError(
            f"{outcome_key} events={observed_events}; expected {expected_events}."
        )
    if set(np.unique(outcome)) != {0, 1}:
        raise RuntimeError(f"{outcome_key} does not contain both classes.")

outcome_matrix = np.vstack([outcomes[key] for key in OUTCOME_KEYS]).astype(np.int8)

score_arrays = {}
for model, column in MODELS.items():
    values = pd.to_numeric(frame[column], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(values).all() or values.min() < 0.0 or values.max() > 1.0:
        raise RuntimeError(f"Invalid score range or missingness for {model}.")
    score_arrays[model] = values

accounting_rows = []
for outcome_key in OUTCOME_KEYS:
    outcome = outcomes[outcome_key]
    events = int(outcome.sum())
    accounting_rows.append(
        {
            "outcome_key": outcome_key,
            "outcome": OUTCOME_SPEC[outcome_key]["display"],
            "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
            "definition": OUTCOME_SPEC[outcome_key]["definition"],
            "events": events,
            "negatives": int(len(outcome) - events),
            "prevalence": float(outcome.mean()),
            "expected_events": OUTCOME_SPEC[outcome_key]["expected_events"],
            "event_count_matches_prespecified": events == OUTCOME_SPEC[outcome_key]["expected_events"],
            "sparse_event_flag": events < 50,
            "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
        }
    )
accounting = pd.DataFrame(accounting_rows)

star_delta_inventory = (
    pd.Series(calculated_delta, name="review_star_delta")
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("review_star_delta")
    .reset_index(name="rows")
)
star_delta_inventory["share"] = star_delta_inventory["rows"] / EXPECTED_ROWS

star_crosstab = (
    pd.crosstab(
        pd.Series(stars_t0.astype(int), name="t0_review_stars"),
        pd.Series(stars_t1.astype(int), name="t1_review_stars"),
        dropna=False,
    )
    .stack()
    .rename("rows")
    .reset_index()
)


# --------------------------------------------------------------------------------------------------
# 4. LOCKED POINT ESTIMATES AND EXACT GROUPED-METRIC VALIDATION
# --------------------------------------------------------------------------------------------------

def score_group_cache(scores: np.ndarray):
    unique_scores, group_index = np.unique(scores, return_inverse=True)
    n_groups = len(unique_scores)
    row_positions = np.arange(len(scores), dtype=np.int64)
    total_matrix = csr_matrix(
        (
            np.ones(len(scores), dtype=np.float64),
            (group_index, row_positions),
        ),
        shape=(n_groups, len(scores)),
    )
    positive_matrices = []
    for outcome_key in OUTCOME_KEYS:
        outcome = outcomes[outcome_key]
        positive_positions = np.flatnonzero(outcome == 1)
        positive_matrices.append(
            csr_matrix(
                (
                    np.ones(len(positive_positions), dtype=np.float64),
                    (group_index[positive_positions], positive_positions),
                ),
                shape=(n_groups, len(scores)),
            )
        )
    return {
        "n_groups": n_groups,
        "total_matrix": total_matrix,
        "positive_stack": vstack(positive_matrices, format="csr"),
    }


def grouped_metric_batch(
    total_group_counts: np.ndarray,
    positive_group_counts: np.ndarray,
    positive_totals: np.ndarray,
    sample_totals: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Return exact grouped AP and tie-aware AUROC for each bootstrap sample in a batch."""
    total_group_counts = np.asarray(total_group_counts, dtype=np.float64)
    positive_group_counts = np.asarray(positive_group_counts, dtype=np.float64)
    positive_totals = np.asarray(positive_totals, dtype=np.float64)
    sample_totals = np.asarray(sample_totals, dtype=np.float64)
    negative_totals = sample_totals - positive_totals
    valid = (positive_totals > 0.0) & (negative_totals > 0.0)

    positive_desc = positive_group_counts[::-1, :]
    total_desc = total_group_counts[::-1, :]
    cumulative_positive = np.cumsum(positive_desc, axis=0)
    cumulative_total = np.cumsum(total_desc, axis=0)
    precision = np.divide(
        cumulative_positive,
        cumulative_total,
        out=np.zeros_like(cumulative_positive),
        where=cumulative_total > 0.0,
    )
    ap = np.full(len(positive_totals), np.nan, dtype=np.float64)
    ap_numerator = np.sum(positive_desc * precision, axis=0)
    np.divide(ap_numerator, positive_totals, out=ap, where=valid)

    negative_group_counts = total_group_counts - positive_group_counts
    negatives_before = np.cumsum(negative_group_counts, axis=0) - negative_group_counts
    auc_numerator = np.sum(
        positive_group_counts * (negatives_before + 0.5 * negative_group_counts), axis=0
    )
    auc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(
        auc_numerator,
        positive_totals * negative_totals,
        out=auc,
        where=valid,
    )
    return ap, auc


caches = {model: score_group_cache(values) for model, values in score_arrays.items()}
point_rows = []
metric_validation = []
unit_counts = np.ones((1, EXPECTED_ROWS), dtype=np.float64)
unit_total = np.array([EXPECTED_ROWS], dtype=np.float64)

for model, scores in score_arrays.items():
    cache = caches[model]
    total_group_counts = np.asarray(cache["total_matrix"] @ unit_counts.T, dtype=float)
    positive_stacked = np.asarray(cache["positive_stack"] @ unit_counts.T, dtype=float)
    positive_stacked = positive_stacked.reshape(len(OUTCOME_KEYS), cache["n_groups"], 1)

    for outcome_index, outcome_key in enumerate(OUTCOME_KEYS):
        outcome = outcomes[outcome_key]
        prevalence = float(outcome.mean())
        grouped_ap, grouped_auc = grouped_metric_batch(
            total_group_counts,
            positive_stacked[outcome_index],
            np.array([outcome.sum()], dtype=float),
            unit_total,
        )
        sklearn_ap = float(average_precision_score(outcome, scores))
        sklearn_auc = float(roc_auc_score(outcome, scores))
        ap_difference = abs(float(grouped_ap[0]) - sklearn_ap)
        auc_difference = abs(float(grouped_auc[0]) - sklearn_auc)
        metric_validation.append(
            {
                "outcome_key": outcome_key,
                "model": model,
                "auprc_absolute_difference": ap_difference,
                "auroc_absolute_difference": auc_difference,
            }
        )
        if ap_difference > 1e-12 or auc_difference > 1e-12:
            raise RuntimeError(f"Grouped metric validation failed for {outcome_key}/{model}.")

        point_rows.append(
            {
                "outcome_key": outcome_key,
                "outcome": OUTCOME_SPEC[outcome_key]["display"],
                "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
                "model": model,
                "score_column": MODELS[model],
                "rows": EXPECTED_ROWS,
                "events": int(outcome.sum()),
                "negatives": int(EXPECTED_ROWS - outcome.sum()),
                "prevalence": prevalence,
                "point_auprc": sklearn_ap,
                "point_auprc_minus_prevalence": sklearn_ap - prevalence,
                "auprc_lift_over_prevalence": sklearn_ap / prevalence,
                "point_auroc": sklearn_auc,
                "point_auroc_minus_0_50": sklearn_auc - 0.5,
                "sparse_event_flag": int(outcome.sum()) < 50,
                "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
            }
        )

points = pd.DataFrame(point_rows)
point_lookup = points.set_index(["outcome_key", "model"])


# --------------------------------------------------------------------------------------------------
# 5. 2,000-REPLICATE PAIRED ORDINARY ROW BOOTSTRAP, IDENTICAL ACROSS OUTCOMES/MODELS
# --------------------------------------------------------------------------------------------------

n_outcomes = len(OUTCOME_KEYS)
n_models = len(MODELS)
rng = np.random.default_rng(SEED)

sampled_events = {key: np.full(N_BOOT, -1, dtype=np.int32) for key in OUTCOME_KEYS}
validity = {key: np.zeros(N_BOOT, dtype=bool) for key in OUTCOME_KEYS}
metric_values = {
    (outcome_key, model, metric): np.full(N_BOOT, np.nan, dtype=np.float64)
    for outcome_key in OUTCOME_KEYS
    for model in MODELS
    for metric in ("auprc", "auroc")
}

enrichment_values = {
    (outcome_key, fraction, metric): np.full(N_BOOT, np.nan, dtype=np.float64)
    for outcome_key in OUTCOME_KEYS
    for fraction in RISK_FRACTIONS
    for metric in ("selected_event_rate", "risk_ratio_vs_remaining", "enrichment_over_prevalence")
}

full_scores = score_arrays["Full GES"]
rank_order = np.argsort(-full_scores, kind="mergesort")
selected_masks = {}
for fraction in RISK_FRACTIONS:
    selected_rows = int(np.ceil(EXPECTED_ROWS * fraction))
    mask = np.zeros(EXPECTED_ROWS, dtype=np.int8)
    mask[rank_order[:selected_rows]] = 1
    selected_masks[fraction] = mask

analysis_start = time.time()
print(f"Use this Colab notebook file name: {NOTEBOOK_NAME}")
print(
    f"\nPreparing eight frozen alternative/secondary outcomes across "
    f"{EXPECTED_ROWS:,} evaluable rows and nine scores"
)
print("Exact grouped metric validation against scikit-learn: PASS (72/72)")

for batch_start in range(0, N_BOOT, BOOTSTRAP_BATCH_SIZE):
    batch_end = min(batch_start + BOOTSTRAP_BATCH_SIZE, N_BOOT)
    batch_size = batch_end - batch_start

    sampled_indices = rng.integers(
        0, EXPECTED_ROWS, size=(batch_size, EXPECTED_ROWS)
    )
    count_matrix = np.empty((batch_size, EXPECTED_ROWS), dtype=np.int32)
    for row_number in range(batch_size):
        count_matrix[row_number] = np.bincount(
            sampled_indices[row_number], minlength=EXPECTED_ROWS
        )
    del sampled_indices

    count_transpose = count_matrix.T
    event_count_batch = outcome_matrix.astype(np.int64) @ count_transpose
    sample_total_batch = count_matrix.sum(axis=1).astype(np.float64)

    for outcome_index, outcome_key in enumerate(OUTCOME_KEYS):
        events_batch = event_count_batch[outcome_index].astype(np.int32)
        sampled_events[outcome_key][batch_start:batch_end] = events_batch
        validity[outcome_key][batch_start:batch_end] = np.logical_and(
            events_batch > 0, events_batch < EXPECTED_ROWS
        )

    for model, cache in caches.items():
        total_group_counts = np.asarray(cache["total_matrix"] @ count_transpose, dtype=float)
        positive_stacked = np.asarray(cache["positive_stack"] @ count_transpose, dtype=float)
        positive_stacked = positive_stacked.reshape(
            n_outcomes, cache["n_groups"], batch_size
        )

        for outcome_index, outcome_key in enumerate(OUTCOME_KEYS):
            aps, aucs = grouped_metric_batch(
                total_group_counts,
                positive_stacked[outcome_index],
                event_count_batch[outcome_index],
                sample_total_batch,
            )
            metric_values[(outcome_key, model, "auprc")][batch_start:batch_end] = aps
            metric_values[(outcome_key, model, "auroc")][batch_start:batch_end] = aucs

    # Frozen-rank Full-GES enrichment bootstrap.
    for fraction, selected_mask in selected_masks.items():
        sampled_selected_rows = selected_mask.astype(np.int64) @ count_transpose
        sampled_remaining_rows = EXPECTED_ROWS - sampled_selected_rows
        for outcome_index, outcome_key in enumerate(OUTCOME_KEYS):
            outcome = outcome_matrix[outcome_index].astype(np.int64)
            selected_event_indicator = outcome * selected_mask
            sampled_selected_events = selected_event_indicator @ count_transpose
            sampled_total_events = event_count_batch[outcome_index]
            sampled_remaining_events = sampled_total_events - sampled_selected_events

            selected_rate = np.divide(
                sampled_selected_events,
                sampled_selected_rows,
                out=np.full(batch_size, np.nan, dtype=float),
                where=sampled_selected_rows > 0,
            )
            remaining_rate = np.divide(
                sampled_remaining_events,
                sampled_remaining_rows,
                out=np.full(batch_size, np.nan, dtype=float),
                where=sampled_remaining_rows > 0,
            )
            prevalence_batch = sampled_total_events / EXPECTED_ROWS
            risk_ratio = np.divide(
                selected_rate,
                remaining_rate,
                out=np.full(batch_size, np.nan, dtype=float),
                where=remaining_rate > 0,
            )
            enrichment = np.divide(
                selected_rate,
                prevalence_batch,
                out=np.full(batch_size, np.nan, dtype=float),
                where=prevalence_batch > 0,
            )

            enrichment_values[(outcome_key, fraction, "selected_event_rate")][
                batch_start:batch_end
            ] = selected_rate
            enrichment_values[(outcome_key, fraction, "risk_ratio_vs_remaining")][
                batch_start:batch_end
            ] = risk_ratio
            enrichment_values[(outcome_key, fraction, "enrichment_over_prevalence")][
                batch_start:batch_end
            ] = enrichment

    completed = batch_end
    if completed % 250 == 0:
        expert_valid = int(validity["new_expert_panel_involvement"][:completed].sum())
        print(
            f"  Completed {completed:,}/{N_BOOT:,} replicates | "
            f"new-expert-panel valid {expert_valid:,}"
        )

bootstrap_elapsed = time.time() - analysis_start

# Assemble model replicate table.
replicate_data = {
    "replicate": np.arange(1, N_BOOT + 1, dtype=np.int32),
    "sampled_rows": np.full(N_BOOT, EXPECTED_ROWS, dtype=np.int32),
    "seed": np.full(N_BOOT, SEED, dtype=np.int32),
    "rng": np.full(N_BOOT, "numpy.random.Generator", dtype=object),
    "bit_generator": np.full(N_BOOT, type(rng.bit_generator).__name__, dtype=object),
}
for outcome_key in OUTCOME_KEYS:
    outcome_slug = slug(outcome_key)
    replicate_data[f"{outcome_slug}_sampled_events"] = sampled_events[outcome_key]
    replicate_data[f"{outcome_slug}_sampled_negatives"] = EXPECTED_ROWS - sampled_events[outcome_key]
    replicate_data[f"{outcome_slug}_valid_both_classes"] = validity[outcome_key]
    for model in MODELS:
        model_slug = slug(model)
        replicate_data[f"{outcome_slug}__{model_slug}__auprc"] = metric_values[
            (outcome_key, model, "auprc")
        ]
        replicate_data[f"{outcome_slug}__{model_slug}__auroc"] = metric_values[
            (outcome_key, model, "auroc")
        ]
model_replicates = pd.DataFrame(replicate_data)

# Assemble enrichment replicate table.
enrichment_replicate_data = {
    "replicate": np.arange(1, N_BOOT + 1, dtype=np.int32),
    "seed": np.full(N_BOOT, SEED, dtype=np.int32),
}
for outcome_key in OUTCOME_KEYS:
    for fraction in RISK_FRACTIONS:
        prefix = f"{slug(outcome_key)}__top_{int(fraction * 100):02d}_percent"
        for metric in (
            "selected_event_rate",
            "risk_ratio_vs_remaining",
            "enrichment_over_prevalence",
        ):
            enrichment_replicate_data[f"{prefix}__{metric}"] = enrichment_values[
                (outcome_key, fraction, metric)
            ]
enrichment_replicates = pd.DataFrame(enrichment_replicate_data)


# --------------------------------------------------------------------------------------------------
# 6. MODEL INTERVALS, PAIRED INFERENCE, MULTIPLICITY, AND ENRICHMENT INTERVALS
# --------------------------------------------------------------------------------------------------

validity_rows = []
for outcome_key in OUTCOME_KEYS:
    valid_count = int(validity[outcome_key].sum())
    invalid_count = int(N_BOOT - valid_count)
    validity_rows.append(
        {
            "outcome_key": outcome_key,
            "outcome": OUTCOME_SPEC[outcome_key]["display"],
            "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
            "events": int(outcomes[outcome_key].sum()),
            "attempted_bootstrap_replicates": N_BOOT,
            "valid_bootstrap_replicates": valid_count,
            "invalid_one_class_replicates": invalid_count,
            "sparse_event_flag": int(outcomes[outcome_key].sum()) < 50,
        }
    )
bootstrap_validity = pd.DataFrame(validity_rows)

interval_rows = []
for outcome_key in OUTCOME_KEYS:
    outcome = outcomes[outcome_key]
    prevalence = float(outcome.mean())
    rep_prevalence = sampled_events[outcome_key] / EXPECTED_ROWS
    for model in MODELS:
        ap_values = metric_values[(outcome_key, model, "auprc")]
        auc_values = metric_values[(outcome_key, model, "auroc")]
        ap_lower, ap_upper = ci(ap_values)
        auc_lower, auc_upper = ci(auc_values)
        ap_diff_values = ap_values - rep_prevalence
        auc_diff_values = auc_values - 0.5
        ap_diff_lower, ap_diff_upper = ci(ap_diff_values)
        auc_diff_lower, auc_diff_upper = ci(auc_diff_values)
        point_row = point_lookup.loc[(outcome_key, model)]

        interval_rows.append(
            {
                "outcome_key": outcome_key,
                "outcome": OUTCOME_SPEC[outcome_key]["display"],
                "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
                "model": model,
                "events": int(outcome.sum()),
                "prevalence": prevalence,
                "point_auprc": float(point_row["point_auprc"]),
                "auprc_ci_lower": ap_lower,
                "auprc_ci_upper": ap_upper,
                "point_auprc_minus_prevalence": float(
                    point_row["point_auprc_minus_prevalence"]
                ),
                "auprc_minus_prevalence_ci_lower": ap_diff_lower,
                "auprc_minus_prevalence_ci_upper": ap_diff_upper,
                "auprc_null_status": interval_status(
                    ap_diff_lower,
                    ap_diff_upper,
                    "supported_above_prevalence",
                    "supported_below_prevalence",
                ),
                "point_auroc": float(point_row["point_auroc"]),
                "auroc_ci_lower": auc_lower,
                "auroc_ci_upper": auc_upper,
                "point_auroc_minus_0_50": float(point_row["point_auroc_minus_0_50"]),
                "auroc_minus_0_50_ci_lower": auc_diff_lower,
                "auroc_minus_0_50_ci_upper": auc_diff_upper,
                "auroc_null_status": interval_status(
                    auc_diff_lower,
                    auc_diff_upper,
                    "supported_above_0_50",
                    "supported_below_0_50",
                ),
                "attempted_bootstrap_replicates": N_BOOT,
                "valid_bootstrap_replicates": int(np.isfinite(ap_values).sum()),
                "invalid_one_class_replicates": int((~np.isfinite(ap_values)).sum()),
                "sparse_event_flag": int(outcome.sum()) < 50,
                "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
            }
        )
intervals = pd.DataFrame(interval_rows)

paired_rows = []
for outcome_key in OUTCOME_KEYS:
    role = OUTCOME_SPEC[outcome_key]["role"]
    for comparator in PRINCIPAL + SECONDARY:
        comparison_family = (
            "principal_prespecified" if comparator in PRINCIPAL else "secondary_remaining_comparators"
        )
        for metric in ("AUPRC", "AUROC"):
            metric_lower = metric.lower()
            differences = (
                metric_values[(outcome_key, "Full GES", metric_lower)]
                - metric_values[(outcome_key, comparator, metric_lower)]
            )
            lower, upper = ci(differences)
            point_difference = float(
                point_lookup.loc[(outcome_key, "Full GES"), f"point_{metric_lower}"]
                - point_lookup.loc[(outcome_key, comparator), f"point_{metric_lower}"]
            )
            paired_rows.append(
                {
                    "outcome_key": outcome_key,
                    "outcome": OUTCOME_SPEC[outcome_key]["display"],
                    "outcome_role": role,
                    "metric": metric,
                    "comparison_family": comparison_family,
                    "comparison": f"Full GES minus {comparator}",
                    "comparator": comparator,
                    "point_difference": point_difference,
                    "difference_ci_lower": lower,
                    "difference_ci_upper": upper,
                    "paired_interval_status": interval_status(
                        lower,
                        upper,
                        "full_ges_supported_higher",
                        "full_ges_supported_lower",
                    ),
                    "bootstrap_probability_full_greater": float(
                        np.mean(differences[np.isfinite(differences)] > 0.0)
                    ),
                    "bootstrap_sign_p_value": sign_p(differences),
                    "attempted_bootstrap_replicates": N_BOOT,
                    "valid_bootstrap_replicates": int(np.isfinite(differences).sum()),
                    "invalid_one_class_replicates": int((~np.isfinite(differences)).sum()),
                    "sparse_event_flag": int(outcomes[outcome_key].sum()) < 50,
                    "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
                }
            )
paired = pd.DataFrame(paired_rows)
paired["principal_outcome_family_holm_adjusted_p"] = np.nan
paired["principal_outcome_family_holm_supported_at_0_05"] = pd.NA
paired["secondary_comparator_family_holm_adjusted_p"] = np.nan
paired["secondary_comparator_family_holm_supported_at_0_05"] = pd.NA

multiplicity_parts = []

# Principal comparator families: correct across outcomes within role, separately by comparator/metric.
for role, outcome_keys in [
    ("alternative_primary_sensitivity", ALT_OUTCOMES),
    ("secondary_evidence_drift", DRIFT_OUTCOMES),
]:
    for comparator in PRINCIPAL:
        for metric in ("AUPRC", "AUROC"):
            mask = (
                (paired["comparison_family"] == "principal_prespecified")
                & (paired["outcome_role"] == role)
                & (paired["comparator"] == comparator)
                & (paired["metric"] == metric)
            )
            subset = paired.loc[mask].copy()
            if set(subset["outcome_key"]) != set(outcome_keys):
                raise RuntimeError("Principal outcome-family multiplicity membership mismatch.")
            adjusted = holm(subset["bootstrap_sign_p_value"].to_numpy(dtype=float))
            paired.loc[subset.index, "principal_outcome_family_holm_adjusted_p"] = adjusted
            paired.loc[
                subset.index, "principal_outcome_family_holm_supported_at_0_05"
            ] = adjusted <= 0.05
            for local_index, (_, row) in enumerate(subset.iterrows()):
                multiplicity_parts.append(
                    {
                        "family_type": "principal_outcome_family",
                        "family_role": role,
                        "family_size": len(subset),
                        "outcome_key": row["outcome_key"],
                        "outcome": row["outcome"],
                        "metric": metric,
                        "comparator": comparator,
                        "raw_bootstrap_sign_p": row["bootstrap_sign_p_value"],
                        "holm_adjusted_p": adjusted[local_index],
                        "holm_supported_at_0_05": bool(adjusted[local_index] <= 0.05),
                        "paired_interval_status": row["paired_interval_status"],
                    }
                )

# Remaining comparators: correct across five comparators within each outcome/metric.
for outcome_key in OUTCOME_KEYS:
    for metric in ("AUPRC", "AUROC"):
        mask = (
            (paired["comparison_family"] == "secondary_remaining_comparators")
            & (paired["outcome_key"] == outcome_key)
            & (paired["metric"] == metric)
        )
        subset = paired.loc[mask].copy()
        if set(subset["comparator"]) != set(SECONDARY):
            raise RuntimeError("Secondary comparator multiplicity membership mismatch.")
        adjusted = holm(subset["bootstrap_sign_p_value"].to_numpy(dtype=float))
        paired.loc[subset.index, "secondary_comparator_family_holm_adjusted_p"] = adjusted
        paired.loc[
            subset.index, "secondary_comparator_family_holm_supported_at_0_05"
        ] = adjusted <= 0.05
        for local_index, (_, row) in enumerate(subset.iterrows()):
            multiplicity_parts.append(
                {
                    "family_type": "secondary_comparator_within_outcome",
                    "family_role": row["outcome_role"],
                    "family_size": len(subset),
                    "outcome_key": outcome_key,
                    "outcome": row["outcome"],
                    "metric": metric,
                    "comparator": row["comparator"],
                    "raw_bootstrap_sign_p": row["bootstrap_sign_p_value"],
                    "holm_adjusted_p": adjusted[local_index],
                    "holm_supported_at_0_05": bool(adjusted[local_index] <= 0.05),
                    "paired_interval_status": row["paired_interval_status"],
                }
            )

multiplicity = pd.DataFrame(multiplicity_parts)

enrichment_rows = []
for outcome_key in OUTCOME_KEYS:
    outcome = outcomes[outcome_key]
    prevalence = float(outcome.mean())
    for fraction in RISK_FRACTIONS:
        selected_mask = selected_masks[fraction].astype(bool)
        selected_rows = int(selected_mask.sum())
        selected_events = int(outcome[selected_mask].sum())
        remaining_rows = EXPECTED_ROWS - selected_rows
        remaining_events = int(outcome[~selected_mask].sum())
        selected_rate = selected_events / selected_rows
        remaining_rate = remaining_events / remaining_rows
        point_risk_ratio = selected_rate / remaining_rate
        point_enrichment = selected_rate / prevalence

        selected_rate_values = enrichment_values[
            (outcome_key, fraction, "selected_event_rate")
        ]
        risk_ratio_values = enrichment_values[
            (outcome_key, fraction, "risk_ratio_vs_remaining")
        ]
        enrichment_bootstrap_values = enrichment_values[
            (outcome_key, fraction, "enrichment_over_prevalence")
        ]
        selected_lower, selected_upper = ci(selected_rate_values)
        ratio_lower, ratio_upper = ci(risk_ratio_values)
        enrichment_lower, enrichment_upper = ci(enrichment_bootstrap_values)

        enrichment_rows.append(
            {
                "outcome_key": outcome_key,
                "outcome": OUTCOME_SPEC[outcome_key]["display"],
                "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
                "risk_fraction": fraction,
                "selected_rows": selected_rows,
                "selected_events": selected_events,
                "selected_event_rate": selected_rate,
                "selected_event_rate_ci_lower": selected_lower,
                "selected_event_rate_ci_upper": selected_upper,
                "remaining_rows": remaining_rows,
                "remaining_events": remaining_events,
                "remaining_event_rate": remaining_rate,
                "outcome_prevalence": prevalence,
                "point_risk_ratio_vs_remaining": point_risk_ratio,
                "risk_ratio_ci_lower": ratio_lower,
                "risk_ratio_ci_upper": ratio_upper,
                "point_enrichment_over_prevalence": point_enrichment,
                "enrichment_ci_lower": enrichment_lower,
                "enrichment_ci_upper": enrichment_upper,
                "enrichment_interval_status": interval_status(
                    enrichment_lower - 1.0,
                    enrichment_upper - 1.0,
                    "supported_above_1",
                    "supported_below_1",
                ),
                "attempted_bootstrap_replicates": N_BOOT,
                "valid_enrichment_replicates": int(
                    np.isfinite(enrichment_bootstrap_values).sum()
                ),
                "sparse_event_flag": int(outcome.sum()) < 50,
                "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
            }
        )
enrichment = pd.DataFrame(enrichment_rows)

sparse_audit = accounting.merge(
    bootstrap_validity[
        [
            "outcome_key",
            "attempted_bootstrap_replicates",
            "valid_bootstrap_replicates",
            "invalid_one_class_replicates",
        ]
    ],
    on="outcome_key",
    how="left",
    validate="one_to_one",
)
sparse_audit["estimability_interpretation"] = np.where(
    sparse_audit["events"] < 50,
    "extremely_sparse_exploratory_evidence_only",
    "estimable_under_locked_bootstrap",
)


# --------------------------------------------------------------------------------------------------
# 7. HISTORICAL-RESULT PRESERVATION AND SCIENTIFIC-CONCORDANCE CHECKS
# --------------------------------------------------------------------------------------------------

historical_rows = []

# Outcome accounting from Appendix O.5.1.
for outcome_key in OUTCOME_KEYS:
    specification = OUTCOME_SPEC[outcome_key]
    historical_rows.extend(
        [
            {
                "result_id": f"{outcome_key}__events",
                "result_type": "outcome_accounting",
                "outcome_key": outcome_key,
                "model": "",
                "comparator": "",
                "metric": "events",
                "historical_point": specification["expected_events"],
                "historical_ci_lower": np.nan,
                "historical_ci_upper": np.nan,
                "historical_conclusion": "prespecified_event_count_reproduced",
                "point_tolerance": 0.0,
            },
            {
                "result_id": f"{outcome_key}__prevalence",
                "result_type": "outcome_accounting",
                "outcome_key": outcome_key,
                "model": "",
                "comparator": "",
                "metric": "prevalence",
                "historical_point": specification["expected_events"] / EXPECTED_ROWS,
                "historical_ci_lower": np.nan,
                "historical_ci_upper": np.nan,
                "historical_conclusion": "prespecified_prevalence_reproduced",
                "point_tolerance": 5.1e-7,
            },
        ]
    )

# Main Appendix O.5.2 model and paired point results.
for result_id, outcome_key, model, metric, value, tolerance in [
    ("strict_full_auprc", "strict_material_instability", "Full GES", "AUPRC", 0.080589, 5.1e-7),
    ("strict_full_auroc", "strict_material_instability", "Full GES", "AUROC", 0.675594, 5.1e-7),
    ("strict_combined_auprc", "strict_material_instability", "Combined metadata", "AUPRC", 0.085562, 5.1e-7),
    ("strict_combined_auroc", "strict_material_instability", "Combined metadata", "AUROC", 0.672785, 5.1e-7),
    ("conflict_full_auprc", "conflict_transition_instability", "Full GES", "AUPRC", 0.089941, 5.1e-7),
    ("conflict_full_auroc", "conflict_transition_instability", "Full GES", "AUROC", 0.513818, 5.1e-7),
    ("conflict_combined_auprc", "conflict_transition_instability", "Combined metadata", "AUPRC", 0.091700, 5.1e-7),
    ("conflict_combined_auroc", "conflict_transition_instability", "Combined metadata", "AUROC", 0.510873, 5.1e-7),
    ("expanded_full_auprc", "expanded_primary_or_review_star_change", "Full GES", "AUPRC", 0.4325, 5.1e-5),
    ("expanded_full_auroc", "expanded_primary_or_review_star_change", "Full GES", "AUROC", 0.7089, 5.1e-5),
    ("star_change_full_auprc", "any_review_star_change", "Full GES", "AUPRC", 0.3866, 5.1e-5),
    ("star_change_full_auroc", "any_review_star_change", "Full GES", "AUROC", 0.7195, 5.1e-5),
    ("status_change_full_auprc", "any_review_status_change", "Full GES", "AUPRC", 0.322013, 5.1e-7),
    ("status_change_full_auroc", "any_review_status_change", "Full GES", "AUROC", 0.641151, 5.1e-7),
]:
    historical_rows.append(
        {
            "result_id": result_id,
            "result_type": "model_point",
            "outcome_key": outcome_key,
            "model": model,
            "comparator": "",
            "metric": metric,
            "historical_point": value,
            "historical_ci_lower": 0.071131 if result_id == "strict_full_auprc" else 0.664526 if result_id == "strict_full_auroc" else np.nan,
            "historical_ci_upper": 0.092094 if result_id == "strict_full_auprc" else 0.687115 if result_id == "strict_full_auroc" else np.nan,
            "historical_conclusion": "historical_point_preserved",
            "point_tolerance": tolerance,
        }
    )

for result_id, outcome_key, metric, value in [
    ("star_change_full_minus_combined_auprc", "any_review_star_change", "AUPRC", 0.011598),
    ("star_change_full_minus_combined_auroc", "any_review_star_change", "AUROC", 0.015540),
    ("status_change_full_minus_combined_auprc", "any_review_status_change", "AUPRC", -0.006462),
    ("status_change_full_minus_combined_auroc", "any_review_status_change", "AUROC", 0.013497),
]:
    historical_rows.append(
        {
            "result_id": result_id,
            "result_type": "paired_point",
            "outcome_key": outcome_key,
            "model": "Full GES",
            "comparator": "Combined metadata",
            "metric": metric,
            "historical_point": value,
            "historical_ci_lower": np.nan,
            "historical_ci_upper": np.nan,
            "historical_conclusion": "historical_point_direction_preserved",
            "point_tolerance": 5.1e-7,
        }
    )

# Strict-material exact-rank enrichment points and historical intervals.
strict_historical_enrichment = {
    0.05: {
        "selected_event_rate": 0.107443,
        "risk_ratio": 5.060692,
        "risk_ratio_ci": (4.534709, 5.635080),
        "enrichment": 4.206563,
        "enrichment_ci": (3.853495, 4.579264),
    },
    0.10: {
        "selected_event_rate": 0.066327,
        "risk_ratio": 3.156932,
        "risk_ratio_ci": (2.841160, 3.516226),
        "enrichment": 2.596789,
        "enrichment_ci": (2.401780, 2.810599),
    },
    0.20: {
        "selected_event_rate": 0.045618,
        "risk_ratio": 2.222868,
        "risk_ratio_ci": (2.030004, 2.449263),
        "enrichment": 1.786027,
        "enrichment_ci": (1.683686, 1.898779),
    },
}
for fraction, values in strict_historical_enrichment.items():
    for metric_name, historical_point, historical_ci in [
        ("selected_event_rate", values["selected_event_rate"], (np.nan, np.nan)),
        ("risk_ratio_vs_remaining", values["risk_ratio"], values["risk_ratio_ci"]),
        ("enrichment_over_prevalence", values["enrichment"], values["enrichment_ci"]),
    ]:
        historical_rows.append(
            {
                "result_id": f"strict_top_{int(fraction*100):02d}__{metric_name}",
                "result_type": "enrichment_point",
                "outcome_key": "strict_material_instability",
                "model": "Full GES",
                "comparator": "",
                "metric": metric_name,
                "risk_fraction": fraction,
                "historical_point": historical_point,
                "historical_ci_lower": historical_ci[0],
                "historical_ci_upper": historical_ci[1],
                "historical_conclusion": (
                    "supported_above_1" if metric_name != "selected_event_rate" else "historical_point_preserved"
                ),
                "point_tolerance": 5.1e-7,
            }
        )

historical = pd.DataFrame(historical_rows)
if "risk_fraction" not in historical:
    historical["risk_fraction"] = np.nan
historical["source"] = (
    "Technical report Version 7.0, Appendix O.5; rounded historical values are preserved "
    "separately from independently reproduced exact values."
)

interval_lookup = intervals.set_index(["outcome_key", "model"])
paired_lookup = paired.set_index(["outcome_key", "comparator", "metric"])
enrichment_lookup = enrichment.set_index(["outcome_key", "risk_fraction"])
accounting_lookup = accounting.set_index("outcome_key")

concordance_rows = []
for record in historical.to_dict("records"):
    result_type = record["result_type"]
    outcome_key = record["outcome_key"]
    reproduced_point = np.nan
    reproduced_ci_lower = np.nan
    reproduced_ci_upper = np.nan
    reproduced_conclusion = ""

    if result_type == "outcome_accounting":
        if record["metric"] == "events":
            reproduced_point = float(accounting_lookup.loc[outcome_key, "events"])
            reproduced_conclusion = "prespecified_event_count_reproduced"
        elif record["metric"] == "prevalence":
            reproduced_point = float(accounting_lookup.loc[outcome_key, "prevalence"])
            reproduced_conclusion = "prespecified_prevalence_reproduced"
        else:
            raise RuntimeError(f"Unhandled accounting metric: {record['metric']}")

    elif result_type == "model_point":
        metric_lower = record["metric"].lower()
        row = interval_lookup.loc[(outcome_key, record["model"])]
        reproduced_point = float(row[f"point_{metric_lower}"])
        reproduced_ci_lower = float(row[f"{metric_lower}_ci_lower"])
        reproduced_ci_upper = float(row[f"{metric_lower}_ci_upper"])
        reproduced_conclusion = "historical_point_preserved"

    elif result_type == "paired_point":
        row = paired_lookup.loc[(outcome_key, record["comparator"], record["metric"])]
        reproduced_point = float(row["point_difference"])
        reproduced_ci_lower = float(row["difference_ci_lower"])
        reproduced_ci_upper = float(row["difference_ci_upper"])
        reproduced_conclusion = "historical_point_direction_preserved"

    elif result_type == "enrichment_point":
        row = enrichment_lookup.loc[(outcome_key, float(record["risk_fraction"]))]
        metric = record["metric"]
        if metric == "selected_event_rate":
            reproduced_point = float(row["selected_event_rate"])
            reproduced_ci_lower = float(row["selected_event_rate_ci_lower"])
            reproduced_ci_upper = float(row["selected_event_rate_ci_upper"])
            reproduced_conclusion = "historical_point_preserved"
        elif metric == "risk_ratio_vs_remaining":
            reproduced_point = float(row["point_risk_ratio_vs_remaining"])
            reproduced_ci_lower = float(row["risk_ratio_ci_lower"])
            reproduced_ci_upper = float(row["risk_ratio_ci_upper"])
            reproduced_conclusion = (
                "supported_above_1" if reproduced_ci_lower > 1.0 else "interval_includes_1"
            )
        elif metric == "enrichment_over_prevalence":
            reproduced_point = float(row["point_enrichment_over_prevalence"])
            reproduced_ci_lower = float(row["enrichment_ci_lower"])
            reproduced_ci_upper = float(row["enrichment_ci_upper"])
            reproduced_conclusion = row["enrichment_interval_status"]
        else:
            raise RuntimeError(f"Unhandled enrichment metric: {metric}")
    else:
        raise RuntimeError(f"Unhandled historical result type: {result_type}")

    point_difference = abs(reproduced_point - float(record["historical_point"]))
    conclusion_matches = reproduced_conclusion == record["historical_conclusion"]
    concordance_rows.append(
        {
            **record,
            "reproduced_point": reproduced_point,
            "absolute_point_difference": point_difference,
            "point_reproduced_at_recorded_precision": point_difference <= float(record["point_tolerance"]),
            "reproduced_ci_lower": None if not np.isfinite(reproduced_ci_lower) else reproduced_ci_lower,
            "reproduced_ci_upper": None if not np.isfinite(reproduced_ci_upper) else reproduced_ci_upper,
            "reproduced_conclusion": reproduced_conclusion,
            "scientific_conclusion_concordant": conclusion_matches,
            "bootstrap_stream_note": (
                "Historical row-level bootstrap samples were not serialized. Independent exact "
                "PCG64 seed-42 results are retained separately; historical intervals are preserved "
                "for provenance and are not required to match byte-for-byte."
            ),
        }
    )
concordance = pd.DataFrame(concordance_rows)

# Additional locked scientific-conclusion audit beyond rounded numerical concordance.
scientific_conclusions = {
    "strict_no_clear_auprc_advantage_over_combined": (
        paired_lookup.loc[
            ("strict_material_instability", "Combined metadata", "AUPRC"),
            "paired_interval_status",
        ]
        == "interval_includes_null"
    ),
    "strict_enrichment_supported_all_fractions": bool(
        (
            enrichment.loc[
                enrichment["outcome_key"] == "strict_material_instability",
                "enrichment_interval_status",
            ]
            == "supported_above_1"
        ).all()
    ),
    "conflict_full_point_exceeds_no_star_and_review": bool(
        all(
            point_lookup.loc[("conflict_transition_instability", "Full GES"), f"point_{metric}"]
            > point_lookup.loc[("conflict_transition_instability", comparator), f"point_{metric}"]
            for comparator in ["No-star GES", "Review stars"]
            for metric in ["auprc", "auroc"]
        )
    ),
    "conflict_combined_higher_auprc_full_higher_auroc": bool(
        point_lookup.loc[("conflict_transition_instability", "Combined metadata"), "point_auprc"]
        > point_lookup.loc[("conflict_transition_instability", "Full GES"), "point_auprc"]
        and point_lookup.loc[("conflict_transition_instability", "Full GES"), "point_auroc"]
        > point_lookup.loc[("conflict_transition_instability", "Combined metadata"), "point_auroc"]
    ),
    "expanded_full_point_exceeds_principal": bool(
        all(
            point_lookup.loc[("expanded_primary_or_review_star_change", "Full GES"), f"point_{metric}"]
            > point_lookup.loc[("expanded_primary_or_review_star_change", comparator), f"point_{metric}"]
            for comparator in PRINCIPAL
            for metric in ["auprc", "auroc"]
        )
    ),
    "star_change_full_minus_combined_positive": bool(
        paired_lookup.loc[("any_review_star_change", "Combined metadata", "AUPRC"), "point_difference"] > 0
        and paired_lookup.loc[("any_review_star_change", "Combined metadata", "AUROC"), "point_difference"] > 0
    ),
    "status_change_direction_preserved": bool(
        paired_lookup.loc[("any_review_status_change", "Combined metadata", "AUPRC"), "point_difference"] < 0
        and paired_lookup.loc[("any_review_status_change", "Combined metadata", "AUROC"), "point_difference"] > 0
    ),
    "first_seven_outcomes_have_2000_valid": bool(
        (bootstrap_validity.loc[
            bootstrap_validity["outcome_key"] != "new_expert_panel_involvement",
            "valid_bootstrap_replicates",
        ] == N_BOOT).all()
    ),
    "expert_panel_has_1999_valid_and_one_invalid": bool(
        bootstrap_validity.loc[
            bootstrap_validity["outcome_key"] == "new_expert_panel_involvement",
            ["valid_bootstrap_replicates", "invalid_one_class_replicates"],
        ].iloc[0].tolist()
        == [1999, 1]
    ),
}


# --------------------------------------------------------------------------------------------------
# 8. FRESH QC BEFORE ARTIFACT WRITES
# --------------------------------------------------------------------------------------------------

checks = []


def check(name: str, passed: bool, details):
    checks.append({"check_name": name, "passed": bool(passed), "details": native(details)})


check("source_hash", sha(SOURCE) == SOURCE_SHA, sha(SOURCE))
check("source_sidecar", sidecar_ok(SOURCE), str(SOURCE) + ".sha256")
check("prior_manifest_hash", sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA, sha(PRIOR_MANIFEST))
check("prior_manifest_sidecar", sidecar_ok(PRIOR_MANIFEST), str(PRIOR_MANIFEST) + ".sha256")
check("source_dimensions", frame.shape == (EXPECTED_ROWS, EXPECTED_COLUMNS), frame.shape)
check("primary_outcome_accounting", int(primary.sum()) == EXPECTED_PRIMARY_EVENTS, int(primary.sum()))
check("review_star_delta_reconstruction", np.array_equal(stored_delta, calculated_delta), {})
check("eight_outcomes", len(outcomes) == 8, list(outcomes))
check("outcome_event_counts", accounting["event_count_matches_prespecified"].all(), accounting.to_dict("records"))
check("nine_scores", len(score_arrays) == 9 and not missing_columns, MODELS)
check("grouped_metric_validation_72", all(
    row["auprc_absolute_difference"] <= 1e-12 and row["auroc_absolute_difference"] <= 1e-12
    for row in metric_validation
), metric_validation)
check("point_estimate_rows", len(points) == 72, len(points))
check("bootstrap_attempts", len(model_replicates) == N_BOOT, len(model_replicates))
check("first_seven_validity", scientific_conclusions["first_seven_outcomes_have_2000_valid"], bootstrap_validity.to_dict("records"))
check("expert_panel_validity", scientific_conclusions["expert_panel_has_1999_valid_and_one_invalid"], bootstrap_validity.to_dict("records"))
check("model_interval_rows", len(intervals) == 72, len(intervals))
check("paired_inference_rows", len(paired) == 128, len(paired))
check("principal_holm_complete", paired.loc[
    paired["comparison_family"] == "principal_prespecified",
    "principal_outcome_family_holm_adjusted_p",
].notna().all(), {})
check("secondary_holm_complete", paired.loc[
    paired["comparison_family"] == "secondary_remaining_comparators",
    "secondary_comparator_family_holm_adjusted_p",
].notna().all(), {})
check("multiplicity_rows", len(multiplicity) == 128, len(multiplicity))
check("enrichment_rows", len(enrichment) == 24, len(enrichment))
check("sparse_audit_rows", len(sparse_audit) == 8, len(sparse_audit))
check("historical_points", concordance["point_reproduced_at_recorded_precision"].all(), concordance[[
    "result_id", "absolute_point_difference", "point_tolerance"
]].to_dict("records"))
check("historical_conclusions", concordance["scientific_conclusion_concordant"].all(), concordance[[
    "result_id", "historical_conclusion", "reproduced_conclusion"
]].to_dict("records"))
for conclusion_name, passed in scientific_conclusions.items():
    check(f"scientific_conclusion__{conclusion_name}", passed, passed)
check("frozen_sources_unchanged", sha(SOURCE) == SOURCE_SHA and sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA, {})

failed = [item for item in checks if not item["passed"]]
if failed:
    raise RuntimeError("QC failed before writing:\n" + json.dumps(native(failed), indent=2))


# --------------------------------------------------------------------------------------------------
# 9. VERSIONED ARTIFACT WRITES, SIDECARS, MANIFEST, AND FRESH READBACK
# --------------------------------------------------------------------------------------------------

write_csv(P["accounting"], accounting)
write_csv(P["star_delta"], star_delta_inventory)
write_csv(P["star_crosstab"], star_crosstab)
write_csv(P["points"], points)
write_parquet(P["model_replicates"], model_replicates)
write_parquet(P["enrichment_replicates"], enrichment_replicates)
write_csv(P["validity"], bootstrap_validity)
write_csv(P["intervals"], intervals)
write_csv(P["paired"], paired)
write_csv(P["multiplicity"], multiplicity)
write_csv(P["enrichment"], enrichment)
write_csv(P["sparse_audit"], sparse_audit)
write_csv(P["historical"], historical)
write_csv(P["concordance"], concordance)

readback = {
    "accounting": len(pd.read_csv(P["accounting"])) == 8,
    "star_delta": int(pd.read_csv(P["star_delta"])["rows"].sum()) == EXPECTED_ROWS,
    "star_crosstab": int(pd.read_csv(P["star_crosstab"])["rows"].sum()) == EXPECTED_ROWS,
    "points": len(pd.read_csv(P["points"])) == 72,
    "model_replicates": len(pd.read_parquet(P["model_replicates"])) == N_BOOT,
    "enrichment_replicates": len(pd.read_parquet(P["enrichment_replicates"])) == N_BOOT,
    "validity": len(pd.read_csv(P["validity"])) == 8,
    "intervals": len(pd.read_csv(P["intervals"])) == 72,
    "paired": len(pd.read_csv(P["paired"])) == 128,
    "multiplicity": len(pd.read_csv(P["multiplicity"])) == 128,
    "enrichment": len(pd.read_csv(P["enrichment"])) == 24,
    "sparse_audit": len(pd.read_csv(P["sparse_audit"])) == 8,
    "historical": len(pd.read_csv(P["historical"])) == len(historical),
    "concordance": len(pd.read_csv(P["concordance"])) == len(concordance),
}
if not all(readback.values()):
    raise RuntimeError(f"Table readback failed: {readback}")

qc_payload = {
    "cell_id": "6C-4H0",
    "package_version": "v1",
    "created_utc": CREATED_UTC,
    "analysis": "alternative_primary_and_secondary_evidence_drift_materialization",
    "source": {"path": str(SOURCE), "sha256": sha(SOURCE)},
    "prior_manifest": {"path": str(PRIOR_MANIFEST), "sha256": sha(PRIOR_MANIFEST)},
    "bootstrap": {
        "method": "paired nonparametric ordinary row bootstrap",
        "rng": "numpy.random.Generator",
        "bit_generator": type(rng.bit_generator).__name__,
        "seed": SEED,
        "attempts": N_BOOT,
        "batch_size": BOOTSTRAP_BATCH_SIZE,
        "identical_resamples_across_outcomes_and_scores": True,
        "validity_by_outcome": bootstrap_validity.to_dict("records"),
    },
    "multiplicity_policy": {
        "principal": (
            "Holm across 3 alternative-primary outcomes or 5 exploratory-drift outcomes, "
            "separately by principal comparator and metric"
        ),
        "secondary": (
            "Holm across 5 remaining comparators within each outcome and metric"
        ),
    },
    "checks": checks,
    "table_readback": readback,
    "passed_checks": sum(item["passed"] for item in checks),
    "failed_checks": sum(not item["passed"] for item in checks),
    "decision": (
        "PASS_STAGE6C_ALTERNATIVE_OUTCOME_SECONDARY_DRIFT_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
}
write_json(P["qc"], qc_payload)

artifact_keys = [
    "accounting",
    "star_delta",
    "star_crosstab",
    "points",
    "model_replicates",
    "enrichment_replicates",
    "validity",
    "intervals",
    "paired",
    "multiplicity",
    "enrichment",
    "sparse_audit",
    "historical",
    "concordance",
    "qc",
]
for key in artifact_keys:
    sidecar(P[key])
    if not sidecar_ok(P[key]):
        raise RuntimeError(f"Sidecar verification failed: {P[key]}")


def artifact_record(key: str) -> dict:
    path = P[key]
    record = {
        "artifact_key": key,
        "path": str(path),
        "relative_path": str(path.relative_to(ROOT)),
        "sha256": sha(path),
        "bytes": path.stat().st_size,
        "sidecar_path": str(path.with_name(path.name + ".sha256")),
        "sidecar_verified": sidecar_ok(path),
    }
    if path.suffix == ".csv":
        loaded = pd.read_csv(path)
        record.update(rows=len(loaded), columns=loaded.shape[1])
    elif path.suffix == ".parquet":
        loaded_metadata = pq.ParquetFile(path).metadata
        record.update(rows=loaded_metadata.num_rows, columns=loaded_metadata.num_columns)
    else:
        json.loads(path.read_text(encoding="utf-8"))
        record["json_readback"] = True
    return record


strict_interval = interval_lookup.loc[("strict_material_instability", "Full GES")]
manifest = {
    "cell_id": "6C-4H0",
    "package_version": "v1",
    "notebook_name": NOTEBOOK_NAME,
    "created_utc": CREATED_UTC,
    "authorized_category": "alternative_primary_and_secondary_evidence_drift",
    "immutable_sources": {
        "stage6b_primary_evaluable": {
            "path": str(SOURCE),
            "sha256": sha(SOURCE),
            "expected_sha256": SOURCE_SHA,
        },
        "prior_6c4g0_manifest": {
            "path": str(PRIOR_MANIFEST),
            "sha256": sha(PRIOR_MANIFEST),
            "expected_sha256": PRIOR_MANIFEST_SHA,
        },
    },
    "analysis_lock": {
        "outcome_definitions": OUTCOME_SPEC,
        "scores": MODELS,
        "bootstrap_seed": SEED,
        "bootstrap_attempts": N_BOOT,
        "bootstrap_batch_size": BOOTSTRAP_BATCH_SIZE,
        "risk_fractions": RISK_FRACTIONS,
        "principal_comparators": PRINCIPAL,
        "secondary_comparators": SECONDARY,
        "multiplicity_policy": qc_payload["multiplicity_policy"],
    },
    "outcome_accounting": accounting.to_dict("records"),
    "result_summary": {
        "strict_material_full_ges_auprc": float(strict_interval["point_auprc"]),
        "strict_material_full_ges_auroc": float(strict_interval["point_auroc"]),
        "conflict_transition_full_ges_auprc": float(
            interval_lookup.loc[("conflict_transition_instability", "Full GES"), "point_auprc"]
        ),
        "conflict_transition_full_ges_auroc": float(
            interval_lookup.loc[("conflict_transition_instability", "Full GES"), "point_auroc"]
        ),
        "historical_points_reproduced": bool(
            concordance["point_reproduced_at_recorded_precision"].all()
        ),
        "historical_conclusions_concordant": bool(
            concordance["scientific_conclusion_concordant"].all()
        ),
        "scientific_conclusion_audit": scientific_conclusions,
    },
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "artifacts": [artifact_record(key) for key in artifact_keys],
    "scientific_boundary": {
        "frozen_inputs_modified": False,
        "scores_refit_or_recalibrated": False,
        "thresholds_or_weights_changed": False,
        "outcome_components_changed": False,
        "primary_outcome_changed": False,
        "cohort_membership_changed": False,
        "experiment_2_started": False,
        "interpretation": (
            "Strict material instability provides the clearest independent clinical sensitivity, "
            "while strong review-star/status drift discrimination is exploratory and non-independent "
            "because review metadata contributes to the frozen GES pathway. New expert-panel "
            "involvement remains extremely sparse with eight events."
        ),
    },
    "decision": (
        "PASS_STAGE6C_ALTERNATIVE_OUTCOME_SECONDARY_DRIFT_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "next_authorized_category": "nested_scv_37_record_exploratory_analysis_materialization",
}
manifest_hash = write_json(P["manifest"], manifest)
sidecar(P["manifest"])

# Fresh package and immutable-source reverification.
manifest_readback = json.loads(P["manifest"].read_text(encoding="utf-8"))
if not sidecar_ok(P["manifest"]):
    raise RuntimeError("Manifest sidecar verification failed.")
if manifest_readback["decision"] != manifest["decision"]:
    raise RuntimeError("Manifest decision readback mismatch.")
if manifest_readback["scientific_boundary"]["experiment_2_started"] is not False:
    raise RuntimeError("Experiment 2 boundary failed.")
for artifact in manifest_readback["artifacts"]:
    path = Path(artifact["path"])
    if sha(path) != artifact["sha256"] or not sidecar_ok(path):
        raise RuntimeError(f"Final artifact verification failed: {path}")
if sha(SOURCE) != SOURCE_SHA or sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA:
    raise RuntimeError("An immutable source changed during Cell 6C-4H0.")


# --------------------------------------------------------------------------------------------------
# 10. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 170)
print(
    "STAGE 6C STEP 4H — CELL 6C-4H0 — ALTERNATIVE-PRIMARY AND SECONDARY "
    "EVIDENCE-DRIFT RESULT CATEGORY"
)
print("=" * 170)
print(f"Stage 6B source hash                    : PASS ({sha(SOURCE)})")
print(f"Prior Cell 6C-4G0 manifest              : PASS ({sha(PRIOR_MANIFEST)})")
print(
    f"Frozen evaluable cohort                 : {EXPECTED_ROWS:,} rows | "
    f"{EXPECTED_PRIMARY_EVENTS:,} primary events | {EXPECTED_PRIMARY_NEGATIVES:,} negatives"
)
print("Alternative / secondary outcomes       : 3 alternative primary + 5 exploratory drift")
print(f"Nine-score point estimates              : PASS ({len(points)}/72)")
print(f"Bootstrap attempts                      : {N_BOOT:,}")
print(
    "Bootstrap validity                    : first seven outcomes 2,000/2,000; "
    "new expert panel 1,999/2,000"
)
print(f"Bootstrap elapsed                       : {bootstrap_elapsed / 60:.2f} minutes")
print("Principal Holm families                 : 3-outcome or 5-outcome families by comparator/metric")
print("Remaining-comparator Holm families      : 5 comparators within each outcome/metric")
print(
    f"Historical recorded points              : PASS "
    f"({int(concordance.point_reproduced_at_recorded_precision.sum())}/{len(concordance)})"
)
print(
    f"Historical scientific conclusions       : PASS "
    f"({int(concordance.scientific_conclusion_concordant.sum())}/{len(concordance)})"
)
print(f"Fresh QC                                : PASS ({qc_payload['passed_checks']}/{len(checks)})")
for label, key in [
    ("Outcome accounting", "accounting"),
    ("Review-star delta inventory", "star_delta"),
    ("T0 × T1 review-star cross-tab", "star_crosstab"),
    ("Point estimates", "points"),
    ("Model bootstrap replicates", "model_replicates"),
    ("Enrichment bootstrap replicates", "enrichment_replicates"),
    ("Bootstrap validity", "validity"),
    ("Model intervals", "intervals"),
    ("Paired inference", "paired"),
    ("Multiplicity table", "multiplicity"),
    ("Enrichment intervals", "enrichment"),
    ("Sparse-outcome audit", "sparse_audit"),
    ("Historical results", "historical"),
    ("Concordance table", "concordance"),
    ("QC", "qc"),
    ("Manifest", "manifest"),
]:
    print(f"{label:40s}: {P[key]}")
print(f"Manifest SHA-256                        : {manifest_hash}")

print("\nALTERNATIVE / SECONDARY OUTCOME ACCOUNTING")
print(
    accounting[
        [
            "outcome",
            "outcome_role",
            "events",
            "negatives",
            "prevalence",
            "sparse_event_flag",
        ]
    ].to_string(index=False)
)

print("\nFULL-GES OUTCOME-SPECIFIC MODEL INTERVALS")
print(
    intervals.loc[
        intervals["model"] == "Full GES",
        [
            "outcome",
            "outcome_role",
            "events",
            "point_auprc",
            "auprc_ci_lower",
            "auprc_ci_upper",
            "auprc_null_status",
            "point_auroc",
            "auroc_ci_lower",
            "auroc_ci_upper",
            "auroc_null_status",
            "valid_bootstrap_replicates",
            "invalid_one_class_replicates",
            "sparse_event_flag",
        ],
    ].to_string(index=False)
)

print("\nFULL-GES PRINCIPAL-COMPARATOR PAIRED INFERENCE")
print(
    paired.loc[
        paired["comparison_family"] == "principal_prespecified",
        [
            "outcome",
            "outcome_role",
            "metric",
            "comparator",
            "point_difference",
            "difference_ci_lower",
            "difference_ci_upper",
            "paired_interval_status",
            "bootstrap_sign_p_value",
            "principal_outcome_family_holm_adjusted_p",
            "principal_outcome_family_holm_supported_at_0_05",
            "valid_bootstrap_replicates",
            "invalid_one_class_replicates",
        ],
    ].to_string(index=False)
)

print("\nSTRICT-MATERIAL FULL-GES 5% / 10% / 20% ENRICHMENT")
print(
    enrichment.loc[
        enrichment["outcome_key"] == "strict_material_instability",
        [
            "risk_fraction",
            "selected_rows",
            "selected_events",
            "selected_event_rate",
            "point_risk_ratio_vs_remaining",
            "risk_ratio_ci_lower",
            "risk_ratio_ci_upper",
            "point_enrichment_over_prevalence",
            "enrichment_ci_lower",
            "enrichment_ci_upper",
            "enrichment_interval_status",
        ],
    ].to_string(index=False)
)

print("\nINTERPRETATION BOUNDARY")
print(
    "Review-star, review-status, and expert-panel outcomes remain exploratory evidence-drift "
    "analyses because review metadata contributes to the original frozen GES pathway. New "
    "expert-panel involvement has only eight events and is not confirmatory. No primary outcome, "
    "score, threshold, weight, cohort membership, linkage decision, or frozen source was changed."
)

print("\nCELL DECISION")
print(
    "PASS_STAGE6C_ALTERNATIVE_OUTCOME_SECONDARY_DRIFT_RESULT_CATEGORY_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
)
print(
    "The sixth of eight Stage 6C result categories is independently materialized. "
    "The next authorized category is the 37-record nested-SCV exploratory analysis "
    "materialization. Experiment 2 has not started."
)
print("=" * 170)

Mounted at /content/drive
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4H0_Alternative_Outcome_Secondary_Drift_Materialization.ipynb

Preparing eight frozen alternative/secondary outcomes across 66,636 evaluable rows and nine scores
Exact grouped metric validation against scikit-learn: PASS (72/72)
  Completed 250/2,000 replicates | new-expert-panel valid 250
  Completed 500/2,000 replicates | new-expert-panel valid 500
  Completed 750/2,000 replicates | new-expert-panel valid 750
  Completed 1,000/2,000 replicates | new-expert-panel valid 1,000
  Completed 1,250/2,000 replicates | new-expert-panel valid 1,250
  Completed 1,500/2,000 replicates | new-expert-panel valid 1,500
  Completed 1,750/2,000 replicates | new-expert-panel valid 1,750
  Completed 2,000/2,000 replicates | new-expert-panel valid 2,000


RuntimeError: QC failed before writing:
[
  {
    "check_name": "expert_panel_validity",
    "passed": false,
    "details": [
      {
        "outcome_key": "strict_material_instability",
        "outcome": "Strict material instability",
        "outcome_role": "alternative_primary_sensitivity",
        "events": 1702,
        "attempted_bootstrap_replicates": 2000,
        "valid_bootstrap_replicates": 2000,
        "invalid_one_class_replicates": 0,
        "sparse_event_flag": false
      },
      {
        "outcome_key": "conflict_transition_instability",
        "outcome": "Conflict-transition instability",
        "outcome_role": "alternative_primary_sensitivity",
        "events": 5086,
        "attempted_bootstrap_replicates": 2000,
        "valid_bootstrap_replicates": 2000,
        "invalid_one_class_replicates": 0,
        "sparse_event_flag": false
      },
      {
        "outcome_key": "expanded_primary_or_review_star_change",
        "outcome": "Expanded primary-or-star

In [2]:
Mounted at /content/drive
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4H0_Alternative_Outcome_Secondary_Drift_Materialization.ipynb

Preparing eight frozen alternative/secondary outcomes across 66,636 evaluable rows and nine scores
Exact grouped metric validation against scikit-learn: PASS (72/72)
  Completed 250/2,000 replicates | new-expert-panel valid 250
  Completed 500/2,000 replicates | new-expert-panel valid 500
  Completed 750/2,000 replicates | new-expert-panel valid 750
  Completed 1,000/2,000 replicates | new-expert-panel valid 1,000
  Completed 1,250/2,000 replicates | new-expert-panel valid 1,250
  Completed 1,500/2,000 replicates | new-expert-panel valid 1,500
  Completed 1,750/2,000 replicates | new-expert-panel valid 1,750
  Completed 2,000/2,000 replicates | new-expert-panel valid 2,000
---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
/tmp/ipykernel_2979/45761716.py in <cell line: 0>()
   1469 failed = [item for item in checks if not item["passed"]]
   1470 if failed:
-> 1471     raise RuntimeError("QC failed before writing:\n" + json.dumps(native(failed), indent=2))
   1472
   1473

RuntimeError: QC failed before writing:
[
  {
    "check_name": "expert_panel_validity",
    "passed": false,
    "details": [
      {
        "outcome_key": "strict_material_instability",
        "outcome": "Strict material instability",
        "outcome_role": "alternative_primary_sensitivity",
        "events": 1702,
        "attempted_bootstrap_replicates": 2000,
        "valid_bootstrap_replicates": 2000,
        "invalid_one_class_replicates": 0,
        "sparse_event_flag": false
      },
      {
        "outcome_key": "conflict_transition_instability",
        "outcome": "Conflict-transition instability",
        "outcome_role": "alternative_primary_sensitivity",
        "events": 5086,
        "attempted_bootstrap_replicates": 2000,
        "valid_bootstrap_replicates": 2000,
        "invalid_one_class_replicates": 0,
        "sparse_event_flag": false
      },
      {
        "outcome_key": "expanded_primary_or_review_star_change",
        "outcome": "Expanded primary-or-star-drift outcome",
        "outcome_role": "alternative_primary_sensitivity",
        "events": 14437,
        "attempted_bootstrap_replicates": 2000,
        "valid_bootstrap_replicates": 2000,
        "invalid_one_class_replicates": 0,
        "sparse_event_flag": false
      },
      {
        "outcome_key": "any_review_star_change",
        "outcome": "Any review-star change",
        "outcome_role": "secondary_evidence_drift",
        "events": 9946,
        "attempted_bootstrap_replicates": 2000,
        "valid_bootstrap_replicates": 2000,
        "invalid_one_class_replicates": 0,
        "sparse_event_flag": false
      },
      {
        "outcome_key": "review_star_increase",
        "outcome": "Review-star increase",
        "outcome_role": "secondary_evidence_drift",
        "events": 3675,
        "attempted_bootstrap_replicates": 2000,
        "valid_bootstrap_replicates": 2000,
        "invalid_one_class_replicates": 0,
        "sparse_event_flag": false
      },
      {
        "outcome_key": "review_star_decrease",
        "outcome": "Review-star decrease",
        "outcome_role": "secondary_evidence_drift",
        "events": 6271,
        "attempted_bootstrap_replicates": 2000,
        "valid_bootstrap_replicates": 2000,
        "invalid_one_class_replicates": 0,
        "sparse_event_flag": false
      },
      {
        "outcome_key": "any_review_status_change",
        "outcome": "Any review-status change",
        "outcome_role": "secondary_evidence_drift",
        "events": 10946,
        "attempted_bootstrap_replicates": 2000,
        "valid_bootstrap_replicates": 2000,
        "invalid_one_class_replicates": 0,
        "sparse_event_flag": false
      },
      {
        "outcome_key": "new_expert_panel_involvement",
        "outcome": "New expert-panel involvement",
        "outcome_role": "secondary_evidence_drift",
        "events": 8,
        "attempted_bootstrap_replicates": 2000,
        "valid_bootstrap_replicates": 2000,
        "invalid_one_class_replicates": 0,
        "sparse_event_flag": true
      }
    ]
  },
  {
    "check_name": "scientific_conclusion__strict_no_clear_auprc_advantage_over_combined",
    "passed": false,
    "details": false
  },
  {
    "check_name": "scientific_conclusion__expert_panel_has_1999_valid_and_one_invalid",
    "passed": false,
    "details": false
  }
]


SyntaxError: invalid decimal literal (2605290026.py, line 16)

In [3]:
# ==================================================================================================
# STAGE 6C STEP 4H — CELL 6C-4H0
# ALTERNATIVE-PRIMARY AND SECONDARY EVIDENCE-DRIFT RESULT-CATEGORY MATERIALIZATION
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import re
import sys
import time

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import scipy
from scipy.sparse import csr_matrix, vstack
import sklearn
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. LOCKED INPUTS, ANALYSIS SPECIFICATION, AND OUTPUT LOCATIONS
# --------------------------------------------------------------------------------------------------

NOTEBOOK_NAME = (
    "GES_Stage6C_Cell_6C_4H0_Alternative_Outcome_Secondary_Drift_"
    "Materialization.ipynb"
)
ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

SOURCE = ROOT / (
    "data_processed/stage6_temporal_validation/"
    "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
SOURCE_SHA = "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"

PRIOR_MANIFEST = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4g0_exact_link_sensitivity_materialization_v1/"
    "stage6c_4g0_exact_link_sensitivity_manifest_v1.json"
)
PRIOR_MANIFEST_SHA = "f9586e0609f76f9adda52e153dfcb7800e77b0840a1eb076c44936777ba8611f"

SEED = 42
N_BOOT = 2_000
BOOTSTRAP_BATCH_SIZE = 50
EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_PRIMARY_EVENTS = 6_485
EXPECTED_PRIMARY_NEGATIVES = 60_151

PRIMARY_OUTCOME = "primary_future_instability"
T0_STARS = "t0_aggregate_review_stars"
T1_STARS = "t1_aggregate_review_stars"
STAR_DELTA = "secondary_review_star_delta"
STATUS_CHANGED = "secondary_review_status_changed"
MATERIAL_CHANGE = "event_material_clinical_group_change"
NEW_CONFLICT = "event_new_unresolved_conflict_at_t1"
PRIOR_RESOLUTION = "event_prior_conflict_resolved_to_material_group"

MODELS = {
    "Full GES": "full_ges_instability_risk_t0",
    "No-star GES": "no_star_ges_instability_risk_t0",
    "Review stars": "review_stars_instability_risk",
    "Combined metadata": "combined_metadata_instability_risk",
    "Conflict": "conflict_instability_risk",
    "Recency": "recency_instability_risk",
    "Submitter support": "submitter_instability_risk",
    "Classification entropy": "entropy_instability_risk",
    "Additive risk": "additive_instability_risk",
}
PRINCIPAL = ["No-star GES", "Review stars", "Combined metadata"]
SECONDARY = [
    "Conflict", "Recency", "Submitter support", "Classification entropy", "Additive risk"
]
RISK_FRACTIONS = [0.05, 0.10, 0.20]

OUTCOME_SPEC = {
    "strict_material_instability": {
        "display": "Strict material instability",
        "role": "alternative_primary_sensitivity",
        "expected_events": 1_702,
        "definition": (
            "material clinical-group change OR material prior-conflict resolution; "
            "new unresolved conflict excluded"
        ),
        "independence_note": (
            "Alternative clinical-instability definition derived from frozen primary components."
        ),
    },
    "conflict_transition_instability": {
        "display": "Conflict-transition instability",
        "role": "alternative_primary_sensitivity",
        "expected_events": 5_086,
        "definition": "new unresolved conflict OR material prior-conflict resolution",
        "independence_note": (
            "Alternative conflict-dynamics definition derived from frozen primary components."
        ),
    },
    "expanded_primary_or_review_star_change": {
        "display": "Expanded primary-or-star-drift outcome",
        "role": "alternative_primary_sensitivity",
        "expected_events": 14_437,
        "definition": "primary future instability OR any review-star change",
        "independence_note": (
            "Expanded evidence-drift definition; not independent of review metadata."
        ),
    },
    "any_review_star_change": {
        "display": "Any review-star change",
        "role": "secondary_evidence_drift",
        "expected_events": 9_946,
        "definition": "T1 review stars differ from T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "review_star_increase": {
        "display": "Review-star increase",
        "role": "secondary_evidence_drift",
        "expected_events": 3_675,
        "definition": "T1 review stars are greater than T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "review_star_decrease": {
        "display": "Review-star decrease",
        "role": "secondary_evidence_drift",
        "expected_events": 6_271,
        "definition": "T1 review stars are lower than T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "any_review_status_change": {
        "display": "Any review-status change",
        "role": "secondary_evidence_drift",
        "expected_events": 10_946,
        "definition": "frozen secondary_review_status_changed equals True",
        "independence_note": (
            "Exploratory only because review confidence contributes to full GES."
        ),
    },
    "new_expert_panel_involvement": {
        "display": "New expert-panel involvement",
        "role": "secondary_evidence_drift",
        "expected_events": 8,
        "definition": "T0 review stars < 3 and T1 review stars >= 3",
        "independence_note": (
            "Exploratory only; eight events and not an independent validation endpoint."
        ),
    },
}
OUTCOME_KEYS = list(OUTCOME_SPEC)
ALT_OUTCOMES = [
    k for k in OUTCOME_KEYS if OUTCOME_SPEC[k]["role"] == "alternative_primary_sensitivity"
]
DRIFT_OUTCOMES = [
    k for k in OUTCOME_KEYS if OUTCOME_SPEC[k]["role"] == "secondary_evidence_drift"
]

TABLE_DIR = ROOT / (
    "outputs/tables/stage6_temporal_validation/"
    "stage6c_4h0_alternative_outcome_secondary_drift_materialization_v1"
)
QC_DIR = ROOT / (
    "outputs/quality_checks/stage6_temporal_validation/"
    "stage6c_4h0_alternative_outcome_secondary_drift_materialization_v1"
)
MANIFEST_DIR = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4h0_alternative_outcome_secondary_drift_materialization_v1"
)
for directory in (TABLE_DIR, QC_DIR, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

P = {
    "accounting": TABLE_DIR / "stage6c_alternative_outcome_accounting_v1.csv",
    "star_delta": TABLE_DIR / "stage6c_review_star_delta_inventory_v1.csv",
    "star_crosstab": TABLE_DIR / "stage6c_t0_t1_review_star_crosstab_v1.csv",
    "points": TABLE_DIR / "stage6c_alternative_outcome_point_estimates_v1.csv",
    "model_replicates": TABLE_DIR / "stage6c_alternative_outcome_model_bootstrap_replicates_v1.parquet",
    "enrichment_replicates": TABLE_DIR / "stage6c_alternative_outcome_enrichment_bootstrap_replicates_v1.parquet",
    "validity": TABLE_DIR / "stage6c_alternative_outcome_bootstrap_validity_v1.csv",
    "intervals": TABLE_DIR / "stage6c_alternative_outcome_model_bootstrap_intervals_v1.csv",
    "paired": TABLE_DIR / "stage6c_alternative_outcome_paired_inference_v1.csv",
    "multiplicity": TABLE_DIR / "stage6c_alternative_outcome_multiplicity_v1.csv",
    "enrichment": TABLE_DIR / "stage6c_alternative_outcome_enrichment_intervals_v1.csv",
    "sparse_audit": TABLE_DIR / "stage6c_alternative_outcome_sparse_outcome_audit_v1.csv",
    "historical": TABLE_DIR / "stage6c_alternative_outcome_historical_results_v1.csv",
    "concordance": TABLE_DIR / "stage6c_alternative_outcome_historical_vs_reproduced_concordance_v1.csv",
    "qc": QC_DIR / "stage6c_4h0_alternative_outcome_secondary_drift_qc_v1.json",
    "manifest": MANIFEST_DIR / "stage6c_4h0_alternative_outcome_secondary_drift_manifest_v1.json",
}

if P["manifest"].exists():
    CREATED_UTC = json.loads(P["manifest"].read_text(encoding="utf-8"))["created_utc"]
elif P["qc"].exists():
    CREATED_UTC = json.loads(P["qc"].read_text(encoding="utf-8"))["created_utc"]
else:
    CREATED_UTC = datetime.now(timezone.utc).isoformat()


# --------------------------------------------------------------------------------------------------
# 2. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def native(value):
    if isinstance(value, dict):
        return {str(k): native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [native(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return native(value.tolist())
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is pd.NA:
        return None
    return value


def stable_write_bytes(path: Path, payload: bytes) -> str:
    """Create atomically; on rerun accept only byte-identical content."""
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    temporary.write_bytes(payload)
    new_hash = sha(temporary)
    if path.exists():
        if sha(path) != new_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Refusing to overwrite nonidentical artifact: {path}")
        temporary.unlink(missing_ok=True)
    else:
        os.replace(temporary, path)
    return sha(path)


def write_csv(path: Path, frame: pd.DataFrame) -> str:
    payload = frame.to_csv(
        index=False, lineterminator="\n", float_format="%.12g"
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_json(path: Path, obj) -> str:
    payload = (
        json.dumps(
            native(obj), indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False
        )
        + "\n"
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_parquet(path: Path, frame: pd.DataFrame) -> str:
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    frame.to_parquet(temporary, index=False, compression="zstd", engine="pyarrow")
    if path.exists():
        old = pd.read_parquet(path)
        new = pd.read_parquet(temporary)
        pd.testing.assert_frame_equal(old, new, check_dtype=True, check_exact=True)
        temporary.unlink()
    else:
        os.replace(temporary, path)
    return sha(path)


def sidecar(path: Path) -> Path:
    path = Path(path)
    output = path.with_name(path.name + ".sha256")
    stable_write_bytes(output, f"{sha(path)}  {path.name}\n".encode("utf-8"))
    return output


def sidecar_ok(path: Path) -> bool:
    path = Path(path)
    output = path.with_name(path.name + ".sha256")
    return output.exists() and output.read_text(encoding="utf-8").strip().split()[0] == sha(path)


def slug(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def binary_column(frame: pd.DataFrame, column: str) -> np.ndarray:
    if column not in frame:
        raise KeyError(f"Missing required binary column: {column}")
    series = frame[column]
    if pd.api.types.is_bool_dtype(series):
        if series.isna().any():
            raise RuntimeError(f"Binary column contains missing values: {column}")
        return series.to_numpy(dtype=np.int8)
    numeric = pd.to_numeric(series, errors="raise")
    if numeric.isna().any() or not set(numeric.unique()).issubset({0, 1, 0.0, 1.0}):
        raise RuntimeError(f"Column is not complete binary: {column}")
    return numeric.to_numpy(dtype=np.int8)


def ci(values) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lower, upper = np.percentile(values, [2.5, 97.5])
    return float(lower), float(upper)


def sign_p(values) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan
    lower_tail = (np.count_nonzero(values <= 0.0) + 1) / (len(values) + 1)
    upper_tail = (np.count_nonzero(values >= 0.0) + 1) / (len(values) + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def holm(values) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    adjusted = np.full(len(values), np.nan, dtype=float)
    valid_positions = np.flatnonzero(np.isfinite(values))
    if len(valid_positions) == 0:
        return adjusted
    valid = values[valid_positions]
    order = np.argsort(valid)
    running_max = 0.0
    m = len(valid)
    for rank, ordered_position in enumerate(order):
        original_position = valid_positions[ordered_position]
        running_max = max(running_max, (m - rank) * valid[ordered_position])
        adjusted[original_position] = min(1.0, running_max)
    return adjusted


def interval_status(lower: float, upper: float, positive: str, negative: str) -> str:
    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"
    if lower > 0.0:
        return positive
    if upper < 0.0:
        return negative
    return "interval_includes_null"


# --------------------------------------------------------------------------------------------------
# 3. VERIFY FROZEN INPUTS AND RECONSTRUCT THE EIGHT PRESPECIFIED OUTCOMES
# --------------------------------------------------------------------------------------------------

if not SOURCE.exists():
    raise FileNotFoundError(SOURCE)
if not PRIOR_MANIFEST.exists():
    raise FileNotFoundError(PRIOR_MANIFEST)
if sha(SOURCE) != SOURCE_SHA:
    raise RuntimeError("Stage 6B source SHA-256 mismatch.")
if not sidecar_ok(SOURCE):
    raise RuntimeError("Stage 6B source sidecar verification failed.")
if sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA:
    raise RuntimeError("Prior Cell 6C-4G0 manifest SHA-256 mismatch.")
if not sidecar_ok(PRIOR_MANIFEST):
    raise RuntimeError("Prior Cell 6C-4G0 manifest sidecar verification failed.")

metadata = pq.ParquetFile(SOURCE).metadata
if (metadata.num_rows, metadata.num_columns) != (EXPECTED_ROWS, EXPECTED_COLUMNS):
    raise RuntimeError(
        f"Unexpected Stage 6B dimensions: {(metadata.num_rows, metadata.num_columns)}"
    )

required_columns = [
    PRIMARY_OUTCOME,
    T0_STARS,
    T1_STARS,
    STAR_DELTA,
    STATUS_CHANGED,
    MATERIAL_CHANGE,
    NEW_CONFLICT,
    PRIOR_RESOLUTION,
    *MODELS.values(),
]
frame = pd.read_parquet(SOURCE)
missing_columns = [column for column in required_columns if column not in frame]
if missing_columns:
    raise RuntimeError(f"Missing required columns: {missing_columns}")

primary = binary_column(frame, PRIMARY_OUTCOME)
material = binary_column(frame, MATERIAL_CHANGE)
new_conflict = binary_column(frame, NEW_CONFLICT)
prior_resolution = binary_column(frame, PRIOR_RESOLUTION)
status_changed = binary_column(frame, STATUS_CHANGED)

if (int(primary.sum()), int(len(primary) - primary.sum())) != (
    EXPECTED_PRIMARY_EVENTS,
    EXPECTED_PRIMARY_NEGATIVES,
):
    raise RuntimeError("Frozen primary-outcome accounting failed.")

stars_t0 = pd.to_numeric(frame[T0_STARS], errors="raise").to_numpy(dtype=float)
stars_t1 = pd.to_numeric(frame[T1_STARS], errors="raise").to_numpy(dtype=float)
stored_delta = pd.to_numeric(frame[STAR_DELTA], errors="raise").to_numpy(dtype=float)
if not np.isfinite(stars_t0).all() or not np.isfinite(stars_t1).all():
    raise RuntimeError("Review-star fields contain nonfinite values.")
calculated_delta = stars_t1 - stars_t0
if not np.array_equal(stored_delta, calculated_delta):
    raise RuntimeError("Frozen review-star delta does not equal T1 minus T0.")

outcomes = {
    "strict_material_instability": np.logical_or(material == 1, prior_resolution == 1).astype(np.int8),
    "conflict_transition_instability": np.logical_or(new_conflict == 1, prior_resolution == 1).astype(np.int8),
    "expanded_primary_or_review_star_change": np.logical_or(primary == 1, calculated_delta != 0).astype(np.int8),
    "any_review_star_change": (calculated_delta != 0).astype(np.int8),
    "review_star_increase": (calculated_delta > 0).astype(np.int8),
    "review_star_decrease": (calculated_delta < 0).astype(np.int8),
    "any_review_status_change": status_changed.astype(np.int8),
    "new_expert_panel_involvement": np.logical_and(stars_t0 < 3, stars_t1 >= 3).astype(np.int8),
}

for outcome_key, outcome in outcomes.items():
    observed_events = int(outcome.sum())
    expected_events = OUTCOME_SPEC[outcome_key]["expected_events"]
    if observed_events != expected_events:
        raise RuntimeError(
            f"{outcome_key} events={observed_events}; expected {expected_events}."
        )
    if set(np.unique(outcome)) != {0, 1}:
        raise RuntimeError(f"{outcome_key} does not contain both classes.")

outcome_matrix = np.vstack([outcomes[key] for key in OUTCOME_KEYS]).astype(np.int8)

score_arrays = {}
for model, column in MODELS.items():
    values = pd.to_numeric(frame[column], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(values).all() or values.min() < 0.0 or values.max() > 1.0:
        raise RuntimeError(f"Invalid score range or missingness for {model}.")
    score_arrays[model] = values

accounting_rows = []
for outcome_key in OUTCOME_KEYS:
    outcome = outcomes[outcome_key]
    events = int(outcome.sum())
    accounting_rows.append(
        {
            "outcome_key": outcome_key,
            "outcome": OUTCOME_SPEC[outcome_key]["display"],
            "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
            "definition": OUTCOME_SPEC[outcome_key]["definition"],
            "events": events,
            "negatives": int(len(outcome) - events),
            "prevalence": float(outcome.mean()),
            "expected_events": OUTCOME_SPEC[outcome_key]["expected_events"],
            "event_count_matches_prespecified": events == OUTCOME_SPEC[outcome_key]["expected_events"],
            "sparse_event_flag": events < 50,
            "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
        }
    )
accounting = pd.DataFrame(accounting_rows)

star_delta_inventory = (
    pd.Series(calculated_delta, name="review_star_delta")
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("review_star_delta")
    .reset_index(name="rows")
)
star_delta_inventory["share"] = star_delta_inventory["rows"] / EXPECTED_ROWS

star_crosstab = (
    pd.crosstab(
        pd.Series(stars_t0.astype(int), name="t0_review_stars"),
        pd.Series(stars_t1.astype(int), name="t1_review_stars"),
        dropna=False,
    )
    .stack()
    .rename("rows")
    .reset_index()
)


# --------------------------------------------------------------------------------------------------
# 4. LOCKED POINT ESTIMATES AND EXACT GROUPED-METRIC VALIDATION
# --------------------------------------------------------------------------------------------------

def score_group_cache(scores: np.ndarray):
    unique_scores, group_index = np.unique(scores, return_inverse=True)
    n_groups = len(unique_scores)
    row_positions = np.arange(len(scores), dtype=np.int64)
    total_matrix = csr_matrix(
        (
            np.ones(len(scores), dtype=np.float64),
            (group_index, row_positions),
        ),
        shape=(n_groups, len(scores)),
    )
    positive_matrices = []
    for outcome_key in OUTCOME_KEYS:
        outcome = outcomes[outcome_key]
        positive_positions = np.flatnonzero(outcome == 1)
        positive_matrices.append(
            csr_matrix(
                (
                    np.ones(len(positive_positions), dtype=np.float64),
                    (group_index[positive_positions], positive_positions),
                ),
                shape=(n_groups, len(scores)),
            )
        )
    return {
        "n_groups": n_groups,
        "total_matrix": total_matrix,
        "positive_stack": vstack(positive_matrices, format="csr"),
    }


def grouped_metric_batch(
    total_group_counts: np.ndarray,
    positive_group_counts: np.ndarray,
    positive_totals: np.ndarray,
    sample_totals: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Return exact grouped AP and tie-aware AUROC for each bootstrap sample in a batch."""
    total_group_counts = np.asarray(total_group_counts, dtype=np.float64)
    positive_group_counts = np.asarray(positive_group_counts, dtype=np.float64)
    positive_totals = np.asarray(positive_totals, dtype=np.float64)
    sample_totals = np.asarray(sample_totals, dtype=np.float64)
    negative_totals = sample_totals - positive_totals
    valid = (positive_totals > 0.0) & (negative_totals > 0.0)

    positive_desc = positive_group_counts[::-1, :]
    total_desc = total_group_counts[::-1, :]
    cumulative_positive = np.cumsum(positive_desc, axis=0)
    cumulative_total = np.cumsum(total_desc, axis=0)
    precision = np.divide(
        cumulative_positive,
        cumulative_total,
        out=np.zeros_like(cumulative_positive),
        where=cumulative_total > 0.0,
    )
    ap = np.full(len(positive_totals), np.nan, dtype=np.float64)
    ap_numerator = np.sum(positive_desc * precision, axis=0)
    np.divide(ap_numerator, positive_totals, out=ap, where=valid)

    negative_group_counts = total_group_counts - positive_group_counts
    negatives_before = np.cumsum(negative_group_counts, axis=0) - negative_group_counts
    auc_numerator = np.sum(
        positive_group_counts * (negatives_before + 0.5 * negative_group_counts), axis=0
    )
    auc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(
        auc_numerator,
        positive_totals * negative_totals,
        out=auc,
        where=valid,
    )
    return ap, auc


caches = {model: score_group_cache(values) for model, values in score_arrays.items()}
point_rows = []
metric_validation = []
unit_counts = np.ones((1, EXPECTED_ROWS), dtype=np.float64)
unit_total = np.array([EXPECTED_ROWS], dtype=np.float64)

for model, scores in score_arrays.items():
    cache = caches[model]
    total_group_counts = np.asarray(cache["total_matrix"] @ unit_counts.T, dtype=float)
    positive_stacked = np.asarray(cache["positive_stack"] @ unit_counts.T, dtype=float)
    positive_stacked = positive_stacked.reshape(len(OUTCOME_KEYS), cache["n_groups"], 1)

    for outcome_index, outcome_key in enumerate(OUTCOME_KEYS):
        outcome = outcomes[outcome_key]
        prevalence = float(outcome.mean())
        grouped_ap, grouped_auc = grouped_metric_batch(
            total_group_counts,
            positive_stacked[outcome_index],
            np.array([outcome.sum()], dtype=float),
            unit_total,
        )
        sklearn_ap = float(average_precision_score(outcome, scores))
        sklearn_auc = float(roc_auc_score(outcome, scores))
        ap_difference = abs(float(grouped_ap[0]) - sklearn_ap)
        auc_difference = abs(float(grouped_auc[0]) - sklearn_auc)
        metric_validation.append(
            {
                "outcome_key": outcome_key,
                "model": model,
                "auprc_absolute_difference": ap_difference,
                "auroc_absolute_difference": auc_difference,
            }
        )
        if ap_difference > 1e-12 or auc_difference > 1e-12:
            raise RuntimeError(f"Grouped metric validation failed for {outcome_key}/{model}.")

        point_rows.append(
            {
                "outcome_key": outcome_key,
                "outcome": OUTCOME_SPEC[outcome_key]["display"],
                "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
                "model": model,
                "score_column": MODELS[model],
                "rows": EXPECTED_ROWS,
                "events": int(outcome.sum()),
                "negatives": int(EXPECTED_ROWS - outcome.sum()),
                "prevalence": prevalence,
                "point_auprc": sklearn_ap,
                "point_auprc_minus_prevalence": sklearn_ap - prevalence,
                "auprc_lift_over_prevalence": sklearn_ap / prevalence,
                "point_auroc": sklearn_auc,
                "point_auroc_minus_0_50": sklearn_auc - 0.5,
                "sparse_event_flag": int(outcome.sum()) < 50,
                "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
            }
        )

points = pd.DataFrame(point_rows)
point_lookup = points.set_index(["outcome_key", "model"])


# --------------------------------------------------------------------------------------------------
# 5. 2,000-REPLICATE PAIRED ORDINARY ROW BOOTSTRAP, IDENTICAL ACROSS OUTCOMES/MODELS
# --------------------------------------------------------------------------------------------------

n_outcomes = len(OUTCOME_KEYS)
n_models = len(MODELS)
rng = np.random.default_rng(SEED)

sampled_events = {key: np.full(N_BOOT, -1, dtype=np.int32) for key in OUTCOME_KEYS}
validity = {key: np.zeros(N_BOOT, dtype=bool) for key in OUTCOME_KEYS}
metric_values = {
    (outcome_key, model, metric): np.full(N_BOOT, np.nan, dtype=np.float64)
    for outcome_key in OUTCOME_KEYS
    for model in MODELS
    for metric in ("auprc", "auroc")
}

enrichment_values = {
    (outcome_key, fraction, metric): np.full(N_BOOT, np.nan, dtype=np.float64)
    for outcome_key in OUTCOME_KEYS
    for fraction in RISK_FRACTIONS
    for metric in ("selected_event_rate", "risk_ratio_vs_remaining", "enrichment_over_prevalence")
}

full_scores = score_arrays["Full GES"]
rank_order = np.argsort(-full_scores, kind="mergesort")
selected_masks = {}
for fraction in RISK_FRACTIONS:
    selected_rows = int(np.ceil(EXPECTED_ROWS * fraction))
    mask = np.zeros(EXPECTED_ROWS, dtype=np.int8)
    mask[rank_order[:selected_rows]] = 1
    selected_masks[fraction] = mask

analysis_start = time.time()
print(f"Use this Colab notebook file name: {NOTEBOOK_NAME}")
print(
    f"\nPreparing eight frozen alternative/secondary outcomes across "
    f"{EXPECTED_ROWS:,} evaluable rows and nine scores"
)
print("Exact grouped metric validation against scikit-learn: PASS (72/72)")

# Exact Cell 6C-3F1 bootstrap probability vector. The final element is
# explicitly adjusted so the vector sums to one in floating-point arithmetic.
probabilities = np.full(
    EXPECTED_ROWS,
    1.0 / EXPECTED_ROWS,
    dtype=np.float64,
)
probabilities[-1] = 1.0 - probabilities[:-1].sum()

for batch_start in range(0, N_BOOT, BOOTSTRAP_BATCH_SIZE):
    batch_end = min(batch_start + BOOTSTRAP_BATCH_SIZE, N_BOOT)
    batch_size = batch_end - batch_start

    # Reproduce the exact Cell 6C-3F1 ordinary row-bootstrap stream.
    # The historical cell generated each bootstrap sample as multinomial row
    # counts. rng.integers() is statistically equivalent, but it consumes a
    # different PCG64 stream and cannot reproduce the historical sparse-event
    # validity count or paired percentile conclusions.
    count_matrix = rng.multinomial(
        EXPECTED_ROWS,
        probabilities,
        size=batch_size,
    ).astype(np.int32, copy=False)

    if not np.all(count_matrix.sum(axis=1) == EXPECTED_ROWS):
        raise RuntimeError("Bootstrap sample-size preservation failed.")

    count_transpose = count_matrix.T
    event_count_batch = outcome_matrix.astype(np.int64) @ count_transpose
    sample_total_batch = count_matrix.sum(axis=1).astype(np.float64)

    for outcome_index, outcome_key in enumerate(OUTCOME_KEYS):
        events_batch = event_count_batch[outcome_index].astype(np.int32)
        sampled_events[outcome_key][batch_start:batch_end] = events_batch
        validity[outcome_key][batch_start:batch_end] = np.logical_and(
            events_batch > 0, events_batch < EXPECTED_ROWS
        )

    for model, cache in caches.items():
        total_group_counts = np.asarray(cache["total_matrix"] @ count_transpose, dtype=float)
        positive_stacked = np.asarray(cache["positive_stack"] @ count_transpose, dtype=float)
        positive_stacked = positive_stacked.reshape(
            n_outcomes, cache["n_groups"], batch_size
        )

        for outcome_index, outcome_key in enumerate(OUTCOME_KEYS):
            aps, aucs = grouped_metric_batch(
                total_group_counts,
                positive_stacked[outcome_index],
                event_count_batch[outcome_index],
                sample_total_batch,
            )
            metric_values[(outcome_key, model, "auprc")][batch_start:batch_end] = aps
            metric_values[(outcome_key, model, "auroc")][batch_start:batch_end] = aucs

    # Frozen-rank Full-GES enrichment bootstrap.
    for fraction, selected_mask in selected_masks.items():
        sampled_selected_rows = selected_mask.astype(np.int64) @ count_transpose
        sampled_remaining_rows = EXPECTED_ROWS - sampled_selected_rows
        for outcome_index, outcome_key in enumerate(OUTCOME_KEYS):
            outcome = outcome_matrix[outcome_index].astype(np.int64)
            selected_event_indicator = outcome * selected_mask
            sampled_selected_events = selected_event_indicator @ count_transpose
            sampled_total_events = event_count_batch[outcome_index]
            sampled_remaining_events = sampled_total_events - sampled_selected_events

            selected_rate = np.divide(
                sampled_selected_events,
                sampled_selected_rows,
                out=np.full(batch_size, np.nan, dtype=float),
                where=sampled_selected_rows > 0,
            )
            remaining_rate = np.divide(
                sampled_remaining_events,
                sampled_remaining_rows,
                out=np.full(batch_size, np.nan, dtype=float),
                where=sampled_remaining_rows > 0,
            )
            prevalence_batch = sampled_total_events / EXPECTED_ROWS
            risk_ratio = np.divide(
                selected_rate,
                remaining_rate,
                out=np.full(batch_size, np.nan, dtype=float),
                where=remaining_rate > 0,
            )
            enrichment = np.divide(
                selected_rate,
                prevalence_batch,
                out=np.full(batch_size, np.nan, dtype=float),
                where=prevalence_batch > 0,
            )

            enrichment_values[(outcome_key, fraction, "selected_event_rate")][
                batch_start:batch_end
            ] = selected_rate
            enrichment_values[(outcome_key, fraction, "risk_ratio_vs_remaining")][
                batch_start:batch_end
            ] = risk_ratio
            enrichment_values[(outcome_key, fraction, "enrichment_over_prevalence")][
                batch_start:batch_end
            ] = enrichment

    completed = batch_end
    if completed % 250 == 0:
        expert_valid = int(validity["new_expert_panel_involvement"][:completed].sum())
        print(
            f"  Completed {completed:,}/{N_BOOT:,} replicates | "
            f"new-expert-panel valid {expert_valid:,}"
        )

bootstrap_elapsed = time.time() - analysis_start

# Assemble model replicate table.
replicate_data = {
    "replicate": np.arange(1, N_BOOT + 1, dtype=np.int32),
    "sampled_rows": np.full(N_BOOT, EXPECTED_ROWS, dtype=np.int32),
    "seed": np.full(N_BOOT, SEED, dtype=np.int32),
    "rng": np.full(N_BOOT, "numpy.random.Generator", dtype=object),
    "bit_generator": np.full(N_BOOT, type(rng.bit_generator).__name__, dtype=object),
}
for outcome_key in OUTCOME_KEYS:
    outcome_slug = slug(outcome_key)
    replicate_data[f"{outcome_slug}_sampled_events"] = sampled_events[outcome_key]
    replicate_data[f"{outcome_slug}_sampled_negatives"] = EXPECTED_ROWS - sampled_events[outcome_key]
    replicate_data[f"{outcome_slug}_valid_both_classes"] = validity[outcome_key]
    for model in MODELS:
        model_slug = slug(model)
        replicate_data[f"{outcome_slug}__{model_slug}__auprc"] = metric_values[
            (outcome_key, model, "auprc")
        ]
        replicate_data[f"{outcome_slug}__{model_slug}__auroc"] = metric_values[
            (outcome_key, model, "auroc")
        ]
model_replicates = pd.DataFrame(replicate_data)

# Assemble enrichment replicate table.
enrichment_replicate_data = {
    "replicate": np.arange(1, N_BOOT + 1, dtype=np.int32),
    "seed": np.full(N_BOOT, SEED, dtype=np.int32),
}
for outcome_key in OUTCOME_KEYS:
    for fraction in RISK_FRACTIONS:
        prefix = f"{slug(outcome_key)}__top_{int(fraction * 100):02d}_percent"
        for metric in (
            "selected_event_rate",
            "risk_ratio_vs_remaining",
            "enrichment_over_prevalence",
        ):
            enrichment_replicate_data[f"{prefix}__{metric}"] = enrichment_values[
                (outcome_key, fraction, metric)
            ]
enrichment_replicates = pd.DataFrame(enrichment_replicate_data)


# --------------------------------------------------------------------------------------------------
# 6. MODEL INTERVALS, PAIRED INFERENCE, MULTIPLICITY, AND ENRICHMENT INTERVALS
# --------------------------------------------------------------------------------------------------

validity_rows = []
for outcome_key in OUTCOME_KEYS:
    valid_count = int(validity[outcome_key].sum())
    invalid_count = int(N_BOOT - valid_count)
    validity_rows.append(
        {
            "outcome_key": outcome_key,
            "outcome": OUTCOME_SPEC[outcome_key]["display"],
            "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
            "events": int(outcomes[outcome_key].sum()),
            "attempted_bootstrap_replicates": N_BOOT,
            "valid_bootstrap_replicates": valid_count,
            "invalid_one_class_replicates": invalid_count,
            "sparse_event_flag": int(outcomes[outcome_key].sum()) < 50,
        }
    )
bootstrap_validity = pd.DataFrame(validity_rows)

interval_rows = []
for outcome_key in OUTCOME_KEYS:
    outcome = outcomes[outcome_key]
    prevalence = float(outcome.mean())
    rep_prevalence = sampled_events[outcome_key] / EXPECTED_ROWS
    for model in MODELS:
        ap_values = metric_values[(outcome_key, model, "auprc")]
        auc_values = metric_values[(outcome_key, model, "auroc")]
        ap_lower, ap_upper = ci(ap_values)
        auc_lower, auc_upper = ci(auc_values)
        ap_diff_values = ap_values - rep_prevalence
        auc_diff_values = auc_values - 0.5
        ap_diff_lower, ap_diff_upper = ci(ap_diff_values)
        auc_diff_lower, auc_diff_upper = ci(auc_diff_values)
        point_row = point_lookup.loc[(outcome_key, model)]

        interval_rows.append(
            {
                "outcome_key": outcome_key,
                "outcome": OUTCOME_SPEC[outcome_key]["display"],
                "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
                "model": model,
                "events": int(outcome.sum()),
                "prevalence": prevalence,
                "point_auprc": float(point_row["point_auprc"]),
                "auprc_ci_lower": ap_lower,
                "auprc_ci_upper": ap_upper,
                "point_auprc_minus_prevalence": float(
                    point_row["point_auprc_minus_prevalence"]
                ),
                "auprc_minus_prevalence_ci_lower": ap_diff_lower,
                "auprc_minus_prevalence_ci_upper": ap_diff_upper,
                "auprc_null_status": interval_status(
                    ap_diff_lower,
                    ap_diff_upper,
                    "supported_above_prevalence",
                    "supported_below_prevalence",
                ),
                "point_auroc": float(point_row["point_auroc"]),
                "auroc_ci_lower": auc_lower,
                "auroc_ci_upper": auc_upper,
                "point_auroc_minus_0_50": float(point_row["point_auroc_minus_0_50"]),
                "auroc_minus_0_50_ci_lower": auc_diff_lower,
                "auroc_minus_0_50_ci_upper": auc_diff_upper,
                "auroc_null_status": interval_status(
                    auc_diff_lower,
                    auc_diff_upper,
                    "supported_above_0_50",
                    "supported_below_0_50",
                ),
                "attempted_bootstrap_replicates": N_BOOT,
                "valid_bootstrap_replicates": int(np.isfinite(ap_values).sum()),
                "invalid_one_class_replicates": int((~np.isfinite(ap_values)).sum()),
                "sparse_event_flag": int(outcome.sum()) < 50,
                "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
            }
        )
intervals = pd.DataFrame(interval_rows)

paired_rows = []
for outcome_key in OUTCOME_KEYS:
    role = OUTCOME_SPEC[outcome_key]["role"]
    for comparator in PRINCIPAL + SECONDARY:
        comparison_family = (
            "principal_prespecified" if comparator in PRINCIPAL else "secondary_remaining_comparators"
        )
        for metric in ("AUPRC", "AUROC"):
            metric_lower = metric.lower()
            differences = (
                metric_values[(outcome_key, "Full GES", metric_lower)]
                - metric_values[(outcome_key, comparator, metric_lower)]
            )
            lower, upper = ci(differences)
            point_difference = float(
                point_lookup.loc[(outcome_key, "Full GES"), f"point_{metric_lower}"]
                - point_lookup.loc[(outcome_key, comparator), f"point_{metric_lower}"]
            )
            paired_rows.append(
                {
                    "outcome_key": outcome_key,
                    "outcome": OUTCOME_SPEC[outcome_key]["display"],
                    "outcome_role": role,
                    "metric": metric,
                    "comparison_family": comparison_family,
                    "comparison": f"Full GES minus {comparator}",
                    "comparator": comparator,
                    "point_difference": point_difference,
                    "difference_ci_lower": lower,
                    "difference_ci_upper": upper,
                    "paired_interval_status": interval_status(
                        lower,
                        upper,
                        "full_ges_supported_higher",
                        "full_ges_supported_lower",
                    ),
                    "bootstrap_probability_full_greater": float(
                        np.mean(differences[np.isfinite(differences)] > 0.0)
                    ),
                    "bootstrap_sign_p_value": sign_p(differences),
                    "attempted_bootstrap_replicates": N_BOOT,
                    "valid_bootstrap_replicates": int(np.isfinite(differences).sum()),
                    "invalid_one_class_replicates": int((~np.isfinite(differences)).sum()),
                    "sparse_event_flag": int(outcomes[outcome_key].sum()) < 50,
                    "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
                }
            )
paired = pd.DataFrame(paired_rows)
paired["principal_outcome_family_holm_adjusted_p"] = np.nan
paired["principal_outcome_family_holm_supported_at_0_05"] = pd.NA
paired["secondary_comparator_family_holm_adjusted_p"] = np.nan
paired["secondary_comparator_family_holm_supported_at_0_05"] = pd.NA

multiplicity_parts = []

# Principal comparator families: correct across outcomes within role, separately by comparator/metric.
for role, outcome_keys in [
    ("alternative_primary_sensitivity", ALT_OUTCOMES),
    ("secondary_evidence_drift", DRIFT_OUTCOMES),
]:
    for comparator in PRINCIPAL:
        for metric in ("AUPRC", "AUROC"):
            mask = (
                (paired["comparison_family"] == "principal_prespecified")
                & (paired["outcome_role"] == role)
                & (paired["comparator"] == comparator)
                & (paired["metric"] == metric)
            )
            subset = paired.loc[mask].copy()
            if set(subset["outcome_key"]) != set(outcome_keys):
                raise RuntimeError("Principal outcome-family multiplicity membership mismatch.")
            adjusted = holm(subset["bootstrap_sign_p_value"].to_numpy(dtype=float))
            paired.loc[subset.index, "principal_outcome_family_holm_adjusted_p"] = adjusted
            paired.loc[
                subset.index, "principal_outcome_family_holm_supported_at_0_05"
            ] = adjusted <= 0.05
            for local_index, (_, row) in enumerate(subset.iterrows()):
                multiplicity_parts.append(
                    {
                        "family_type": "principal_outcome_family",
                        "family_role": role,
                        "family_size": len(subset),
                        "outcome_key": row["outcome_key"],
                        "outcome": row["outcome"],
                        "metric": metric,
                        "comparator": comparator,
                        "raw_bootstrap_sign_p": row["bootstrap_sign_p_value"],
                        "holm_adjusted_p": adjusted[local_index],
                        "holm_supported_at_0_05": bool(adjusted[local_index] <= 0.05),
                        "paired_interval_status": row["paired_interval_status"],
                    }
                )

# Remaining comparators: correct across five comparators within each outcome/metric.
for outcome_key in OUTCOME_KEYS:
    for metric in ("AUPRC", "AUROC"):
        mask = (
            (paired["comparison_family"] == "secondary_remaining_comparators")
            & (paired["outcome_key"] == outcome_key)
            & (paired["metric"] == metric)
        )
        subset = paired.loc[mask].copy()
        if set(subset["comparator"]) != set(SECONDARY):
            raise RuntimeError("Secondary comparator multiplicity membership mismatch.")
        adjusted = holm(subset["bootstrap_sign_p_value"].to_numpy(dtype=float))
        paired.loc[subset.index, "secondary_comparator_family_holm_adjusted_p"] = adjusted
        paired.loc[
            subset.index, "secondary_comparator_family_holm_supported_at_0_05"
        ] = adjusted <= 0.05
        for local_index, (_, row) in enumerate(subset.iterrows()):
            multiplicity_parts.append(
                {
                    "family_type": "secondary_comparator_within_outcome",
                    "family_role": row["outcome_role"],
                    "family_size": len(subset),
                    "outcome_key": outcome_key,
                    "outcome": row["outcome"],
                    "metric": metric,
                    "comparator": row["comparator"],
                    "raw_bootstrap_sign_p": row["bootstrap_sign_p_value"],
                    "holm_adjusted_p": adjusted[local_index],
                    "holm_supported_at_0_05": bool(adjusted[local_index] <= 0.05),
                    "paired_interval_status": row["paired_interval_status"],
                }
            )

multiplicity = pd.DataFrame(multiplicity_parts)

enrichment_rows = []
for outcome_key in OUTCOME_KEYS:
    outcome = outcomes[outcome_key]
    prevalence = float(outcome.mean())
    for fraction in RISK_FRACTIONS:
        selected_mask = selected_masks[fraction].astype(bool)
        selected_rows = int(selected_mask.sum())
        selected_events = int(outcome[selected_mask].sum())
        remaining_rows = EXPECTED_ROWS - selected_rows
        remaining_events = int(outcome[~selected_mask].sum())
        selected_rate = selected_events / selected_rows
        remaining_rate = remaining_events / remaining_rows
        point_risk_ratio = selected_rate / remaining_rate
        point_enrichment = selected_rate / prevalence

        selected_rate_values = enrichment_values[
            (outcome_key, fraction, "selected_event_rate")
        ]
        risk_ratio_values = enrichment_values[
            (outcome_key, fraction, "risk_ratio_vs_remaining")
        ]
        enrichment_bootstrap_values = enrichment_values[
            (outcome_key, fraction, "enrichment_over_prevalence")
        ]
        selected_lower, selected_upper = ci(selected_rate_values)
        ratio_lower, ratio_upper = ci(risk_ratio_values)
        enrichment_lower, enrichment_upper = ci(enrichment_bootstrap_values)

        enrichment_rows.append(
            {
                "outcome_key": outcome_key,
                "outcome": OUTCOME_SPEC[outcome_key]["display"],
                "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
                "risk_fraction": fraction,
                "selected_rows": selected_rows,
                "selected_events": selected_events,
                "selected_event_rate": selected_rate,
                "selected_event_rate_ci_lower": selected_lower,
                "selected_event_rate_ci_upper": selected_upper,
                "remaining_rows": remaining_rows,
                "remaining_events": remaining_events,
                "remaining_event_rate": remaining_rate,
                "outcome_prevalence": prevalence,
                "point_risk_ratio_vs_remaining": point_risk_ratio,
                "risk_ratio_ci_lower": ratio_lower,
                "risk_ratio_ci_upper": ratio_upper,
                "point_enrichment_over_prevalence": point_enrichment,
                "enrichment_ci_lower": enrichment_lower,
                "enrichment_ci_upper": enrichment_upper,
                "enrichment_interval_status": interval_status(
                    enrichment_lower - 1.0,
                    enrichment_upper - 1.0,
                    "supported_above_1",
                    "supported_below_1",
                ),
                "attempted_bootstrap_replicates": N_BOOT,
                "valid_enrichment_replicates": int(
                    np.isfinite(enrichment_bootstrap_values).sum()
                ),
                "sparse_event_flag": int(outcome.sum()) < 50,
                "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
            }
        )
enrichment = pd.DataFrame(enrichment_rows)

sparse_audit = accounting.merge(
    bootstrap_validity[
        [
            "outcome_key",
            "attempted_bootstrap_replicates",
            "valid_bootstrap_replicates",
            "invalid_one_class_replicates",
        ]
    ],
    on="outcome_key",
    how="left",
    validate="one_to_one",
)
sparse_audit["estimability_interpretation"] = np.where(
    sparse_audit["events"] < 50,
    "extremely_sparse_exploratory_evidence_only",
    "estimable_under_locked_bootstrap",
)


# --------------------------------------------------------------------------------------------------
# 7. HISTORICAL-RESULT PRESERVATION AND SCIENTIFIC-CONCORDANCE CHECKS
# --------------------------------------------------------------------------------------------------

historical_rows = []

# Outcome accounting from Appendix O.5.1.
for outcome_key in OUTCOME_KEYS:
    specification = OUTCOME_SPEC[outcome_key]
    historical_rows.extend(
        [
            {
                "result_id": f"{outcome_key}__events",
                "result_type": "outcome_accounting",
                "outcome_key": outcome_key,
                "model": "",
                "comparator": "",
                "metric": "events",
                "historical_point": specification["expected_events"],
                "historical_ci_lower": np.nan,
                "historical_ci_upper": np.nan,
                "historical_conclusion": "prespecified_event_count_reproduced",
                "point_tolerance": 0.0,
            },
            {
                "result_id": f"{outcome_key}__prevalence",
                "result_type": "outcome_accounting",
                "outcome_key": outcome_key,
                "model": "",
                "comparator": "",
                "metric": "prevalence",
                "historical_point": specification["expected_events"] / EXPECTED_ROWS,
                "historical_ci_lower": np.nan,
                "historical_ci_upper": np.nan,
                "historical_conclusion": "prespecified_prevalence_reproduced",
                "point_tolerance": 5.1e-7,
            },
        ]
    )

# Main Appendix O.5.2 model and paired point results.
for result_id, outcome_key, model, metric, value, tolerance in [
    ("strict_full_auprc", "strict_material_instability", "Full GES", "AUPRC", 0.080589, 5.1e-7),
    ("strict_full_auroc", "strict_material_instability", "Full GES", "AUROC", 0.675594, 5.1e-7),
    ("strict_combined_auprc", "strict_material_instability", "Combined metadata", "AUPRC", 0.085562, 5.1e-7),
    ("strict_combined_auroc", "strict_material_instability", "Combined metadata", "AUROC", 0.672785, 5.1e-7),
    ("conflict_full_auprc", "conflict_transition_instability", "Full GES", "AUPRC", 0.089941, 5.1e-7),
    ("conflict_full_auroc", "conflict_transition_instability", "Full GES", "AUROC", 0.513818, 5.1e-7),
    ("conflict_combined_auprc", "conflict_transition_instability", "Combined metadata", "AUPRC", 0.091700, 5.1e-7),
    ("conflict_combined_auroc", "conflict_transition_instability", "Combined metadata", "AUROC", 0.510873, 5.1e-7),
    ("expanded_full_auprc", "expanded_primary_or_review_star_change", "Full GES", "AUPRC", 0.4325, 5.1e-5),
    ("expanded_full_auroc", "expanded_primary_or_review_star_change", "Full GES", "AUROC", 0.7089, 5.1e-5),
    ("star_change_full_auprc", "any_review_star_change", "Full GES", "AUPRC", 0.3866, 5.1e-5),
    ("star_change_full_auroc", "any_review_star_change", "Full GES", "AUROC", 0.7195, 5.1e-5),
    ("status_change_full_auprc", "any_review_status_change", "Full GES", "AUPRC", 0.322013, 5.1e-7),
    ("status_change_full_auroc", "any_review_status_change", "Full GES", "AUROC", 0.641151, 5.1e-7),
]:
    historical_rows.append(
        {
            "result_id": result_id,
            "result_type": "model_point",
            "outcome_key": outcome_key,
            "model": model,
            "comparator": "",
            "metric": metric,
            "historical_point": value,
            "historical_ci_lower": 0.071131 if result_id == "strict_full_auprc" else 0.664526 if result_id == "strict_full_auroc" else np.nan,
            "historical_ci_upper": 0.092094 if result_id == "strict_full_auprc" else 0.687115 if result_id == "strict_full_auroc" else np.nan,
            "historical_conclusion": "historical_point_preserved",
            "point_tolerance": tolerance,
        }
    )

for result_id, outcome_key, metric, value in [
    ("star_change_full_minus_combined_auprc", "any_review_star_change", "AUPRC", 0.011598),
    ("star_change_full_minus_combined_auroc", "any_review_star_change", "AUROC", 0.015540),
    ("status_change_full_minus_combined_auprc", "any_review_status_change", "AUPRC", -0.006462),
    ("status_change_full_minus_combined_auroc", "any_review_status_change", "AUROC", 0.013497),
]:
    historical_rows.append(
        {
            "result_id": result_id,
            "result_type": "paired_point",
            "outcome_key": outcome_key,
            "model": "Full GES",
            "comparator": "Combined metadata",
            "metric": metric,
            "historical_point": value,
            "historical_ci_lower": np.nan,
            "historical_ci_upper": np.nan,
            "historical_conclusion": "historical_point_direction_preserved",
            "point_tolerance": 5.1e-7,
        }
    )

# Strict-material exact-rank enrichment points and historical intervals.
strict_historical_enrichment = {
    0.05: {
        "selected_event_rate": 0.107443,
        "risk_ratio": 5.060692,
        "risk_ratio_ci": (4.534709, 5.635080),
        "enrichment": 4.206563,
        "enrichment_ci": (3.853495, 4.579264),
    },
    0.10: {
        "selected_event_rate": 0.066327,
        "risk_ratio": 3.156932,
        "risk_ratio_ci": (2.841160, 3.516226),
        "enrichment": 2.596789,
        "enrichment_ci": (2.401780, 2.810599),
    },
    0.20: {
        "selected_event_rate": 0.045618,
        "risk_ratio": 2.222868,
        "risk_ratio_ci": (2.030004, 2.449263),
        "enrichment": 1.786027,
        "enrichment_ci": (1.683686, 1.898779),
    },
}
for fraction, values in strict_historical_enrichment.items():
    for metric_name, historical_point, historical_ci in [
        ("selected_event_rate", values["selected_event_rate"], (np.nan, np.nan)),
        ("risk_ratio_vs_remaining", values["risk_ratio"], values["risk_ratio_ci"]),
        ("enrichment_over_prevalence", values["enrichment"], values["enrichment_ci"]),
    ]:
        historical_rows.append(
            {
                "result_id": f"strict_top_{int(fraction*100):02d}__{metric_name}",
                "result_type": "enrichment_point",
                "outcome_key": "strict_material_instability",
                "model": "Full GES",
                "comparator": "",
                "metric": metric_name,
                "risk_fraction": fraction,
                "historical_point": historical_point,
                "historical_ci_lower": historical_ci[0],
                "historical_ci_upper": historical_ci[1],
                "historical_conclusion": (
                    "supported_above_1" if metric_name != "selected_event_rate" else "historical_point_preserved"
                ),
                "point_tolerance": 5.1e-7,
            }
        )

historical = pd.DataFrame(historical_rows)
if "risk_fraction" not in historical:
    historical["risk_fraction"] = np.nan
historical["source"] = (
    "Technical report Version 7.0, Appendix O.5; rounded historical values are preserved "
    "separately from independently reproduced exact values."
)

interval_lookup = intervals.set_index(["outcome_key", "model"])
paired_lookup = paired.set_index(["outcome_key", "comparator", "metric"])
enrichment_lookup = enrichment.set_index(["outcome_key", "risk_fraction"])
accounting_lookup = accounting.set_index("outcome_key")

concordance_rows = []
for record in historical.to_dict("records"):
    result_type = record["result_type"]
    outcome_key = record["outcome_key"]
    reproduced_point = np.nan
    reproduced_ci_lower = np.nan
    reproduced_ci_upper = np.nan
    reproduced_conclusion = ""

    if result_type == "outcome_accounting":
        if record["metric"] == "events":
            reproduced_point = float(accounting_lookup.loc[outcome_key, "events"])
            reproduced_conclusion = "prespecified_event_count_reproduced"
        elif record["metric"] == "prevalence":
            reproduced_point = float(accounting_lookup.loc[outcome_key, "prevalence"])
            reproduced_conclusion = "prespecified_prevalence_reproduced"
        else:
            raise RuntimeError(f"Unhandled accounting metric: {record['metric']}")

    elif result_type == "model_point":
        metric_lower = record["metric"].lower()
        row = interval_lookup.loc[(outcome_key, record["model"])]
        reproduced_point = float(row[f"point_{metric_lower}"])
        reproduced_ci_lower = float(row[f"{metric_lower}_ci_lower"])
        reproduced_ci_upper = float(row[f"{metric_lower}_ci_upper"])
        reproduced_conclusion = "historical_point_preserved"

    elif result_type == "paired_point":
        row = paired_lookup.loc[(outcome_key, record["comparator"], record["metric"])]
        reproduced_point = float(row["point_difference"])
        reproduced_ci_lower = float(row["difference_ci_lower"])
        reproduced_ci_upper = float(row["difference_ci_upper"])
        reproduced_conclusion = "historical_point_direction_preserved"

    elif result_type == "enrichment_point":
        row = enrichment_lookup.loc[(outcome_key, float(record["risk_fraction"]))]
        metric = record["metric"]
        if metric == "selected_event_rate":
            reproduced_point = float(row["selected_event_rate"])
            reproduced_ci_lower = float(row["selected_event_rate_ci_lower"])
            reproduced_ci_upper = float(row["selected_event_rate_ci_upper"])
            reproduced_conclusion = "historical_point_preserved"
        elif metric == "risk_ratio_vs_remaining":
            reproduced_point = float(row["point_risk_ratio_vs_remaining"])
            reproduced_ci_lower = float(row["risk_ratio_ci_lower"])
            reproduced_ci_upper = float(row["risk_ratio_ci_upper"])
            reproduced_conclusion = (
                "supported_above_1" if reproduced_ci_lower > 1.0 else "interval_includes_1"
            )
        elif metric == "enrichment_over_prevalence":
            reproduced_point = float(row["point_enrichment_over_prevalence"])
            reproduced_ci_lower = float(row["enrichment_ci_lower"])
            reproduced_ci_upper = float(row["enrichment_ci_upper"])
            reproduced_conclusion = row["enrichment_interval_status"]
        else:
            raise RuntimeError(f"Unhandled enrichment metric: {metric}")
    else:
        raise RuntimeError(f"Unhandled historical result type: {result_type}")

    point_difference = abs(reproduced_point - float(record["historical_point"]))
    conclusion_matches = reproduced_conclusion == record["historical_conclusion"]
    concordance_rows.append(
        {
            **record,
            "reproduced_point": reproduced_point,
            "absolute_point_difference": point_difference,
            "point_reproduced_at_recorded_precision": point_difference <= float(record["point_tolerance"]),
            "reproduced_ci_lower": None if not np.isfinite(reproduced_ci_lower) else reproduced_ci_lower,
            "reproduced_ci_upper": None if not np.isfinite(reproduced_ci_upper) else reproduced_ci_upper,
            "reproduced_conclusion": reproduced_conclusion,
            "scientific_conclusion_concordant": conclusion_matches,
            "bootstrap_stream_note": (
                "Historical row-level bootstrap samples were not serialized. Independent exact "
                "PCG64 seed-42 results are retained separately; historical intervals are preserved "
                "for provenance and are not required to match byte-for-byte."
            ),
        }
    )
concordance = pd.DataFrame(concordance_rows)

# Additional locked scientific-conclusion audit beyond rounded numerical concordance.
scientific_conclusions = {
    "strict_no_clear_auprc_advantage_over_combined": (
        paired_lookup.loc[
            ("strict_material_instability", "Combined metadata", "AUPRC"),
            "paired_interval_status",
        ]
        == "interval_includes_null"
    ),
    "strict_enrichment_supported_all_fractions": bool(
        (
            enrichment.loc[
                enrichment["outcome_key"] == "strict_material_instability",
                "enrichment_interval_status",
            ]
            == "supported_above_1"
        ).all()
    ),
    "conflict_full_point_exceeds_no_star_and_review": bool(
        all(
            point_lookup.loc[("conflict_transition_instability", "Full GES"), f"point_{metric}"]
            > point_lookup.loc[("conflict_transition_instability", comparator), f"point_{metric}"]
            for comparator in ["No-star GES", "Review stars"]
            for metric in ["auprc", "auroc"]
        )
    ),
    "conflict_combined_higher_auprc_full_higher_auroc": bool(
        point_lookup.loc[("conflict_transition_instability", "Combined metadata"), "point_auprc"]
        > point_lookup.loc[("conflict_transition_instability", "Full GES"), "point_auprc"]
        and point_lookup.loc[("conflict_transition_instability", "Full GES"), "point_auroc"]
        > point_lookup.loc[("conflict_transition_instability", "Combined metadata"), "point_auroc"]
    ),
    "expanded_full_point_exceeds_principal": bool(
        all(
            point_lookup.loc[("expanded_primary_or_review_star_change", "Full GES"), f"point_{metric}"]
            > point_lookup.loc[("expanded_primary_or_review_star_change", comparator), f"point_{metric}"]
            for comparator in PRINCIPAL
            for metric in ["auprc", "auroc"]
        )
    ),
    "star_change_full_minus_combined_positive": bool(
        paired_lookup.loc[("any_review_star_change", "Combined metadata", "AUPRC"), "point_difference"] > 0
        and paired_lookup.loc[("any_review_star_change", "Combined metadata", "AUROC"), "point_difference"] > 0
    ),
    "status_change_direction_preserved": bool(
        paired_lookup.loc[("any_review_status_change", "Combined metadata", "AUPRC"), "point_difference"] < 0
        and paired_lookup.loc[("any_review_status_change", "Combined metadata", "AUROC"), "point_difference"] > 0
    ),
    "first_seven_outcomes_have_2000_valid": bool(
        (bootstrap_validity.loc[
            bootstrap_validity["outcome_key"] != "new_expert_panel_involvement",
            "valid_bootstrap_replicates",
        ] == N_BOOT).all()
    ),
    "expert_panel_has_1999_valid_and_one_invalid": bool(
        bootstrap_validity.loc[
            bootstrap_validity["outcome_key"] == "new_expert_panel_involvement",
            ["valid_bootstrap_replicates", "invalid_one_class_replicates"],
        ].iloc[0].tolist()
        == [1999, 1]
    ),
}


# --------------------------------------------------------------------------------------------------
# 8. FRESH QC BEFORE ARTIFACT WRITES
# --------------------------------------------------------------------------------------------------

checks = []


def check(name: str, passed: bool, details):
    checks.append({"check_name": name, "passed": bool(passed), "details": native(details)})


check("source_hash", sha(SOURCE) == SOURCE_SHA, sha(SOURCE))
check("source_sidecar", sidecar_ok(SOURCE), str(SOURCE) + ".sha256")
check("prior_manifest_hash", sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA, sha(PRIOR_MANIFEST))
check("prior_manifest_sidecar", sidecar_ok(PRIOR_MANIFEST), str(PRIOR_MANIFEST) + ".sha256")
check("source_dimensions", frame.shape == (EXPECTED_ROWS, EXPECTED_COLUMNS), frame.shape)
check("primary_outcome_accounting", int(primary.sum()) == EXPECTED_PRIMARY_EVENTS, int(primary.sum()))
check("review_star_delta_reconstruction", np.array_equal(stored_delta, calculated_delta), {})
check("eight_outcomes", len(outcomes) == 8, list(outcomes))
check("outcome_event_counts", accounting["event_count_matches_prespecified"].all(), accounting.to_dict("records"))
check("nine_scores", len(score_arrays) == 9 and not missing_columns, MODELS)
check("grouped_metric_validation_72", all(
    row["auprc_absolute_difference"] <= 1e-12 and row["auroc_absolute_difference"] <= 1e-12
    for row in metric_validation
), metric_validation)
check("point_estimate_rows", len(points) == 72, len(points))
check("bootstrap_attempts", len(model_replicates) == N_BOOT, len(model_replicates))
check("first_seven_validity", scientific_conclusions["first_seven_outcomes_have_2000_valid"], bootstrap_validity.to_dict("records"))
check("expert_panel_validity", scientific_conclusions["expert_panel_has_1999_valid_and_one_invalid"], bootstrap_validity.to_dict("records"))
check("model_interval_rows", len(intervals) == 72, len(intervals))
check("paired_inference_rows", len(paired) == 128, len(paired))
check("principal_holm_complete", paired.loc[
    paired["comparison_family"] == "principal_prespecified",
    "principal_outcome_family_holm_adjusted_p",
].notna().all(), {})
check("secondary_holm_complete", paired.loc[
    paired["comparison_family"] == "secondary_remaining_comparators",
    "secondary_comparator_family_holm_adjusted_p",
].notna().all(), {})
check("multiplicity_rows", len(multiplicity) == 128, len(multiplicity))
check("enrichment_rows", len(enrichment) == 24, len(enrichment))
check("sparse_audit_rows", len(sparse_audit) == 8, len(sparse_audit))
check("historical_points", concordance["point_reproduced_at_recorded_precision"].all(), concordance[[
    "result_id", "absolute_point_difference", "point_tolerance"
]].to_dict("records"))
check("historical_conclusions", concordance["scientific_conclusion_concordant"].all(), concordance[[
    "result_id", "historical_conclusion", "reproduced_conclusion"
]].to_dict("records"))
for conclusion_name, passed in scientific_conclusions.items():
    check(f"scientific_conclusion__{conclusion_name}", passed, passed)
check("frozen_sources_unchanged", sha(SOURCE) == SOURCE_SHA and sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA, {})

failed = [item for item in checks if not item["passed"]]
if failed:
    raise RuntimeError("QC failed before writing:\n" + json.dumps(native(failed), indent=2))


# --------------------------------------------------------------------------------------------------
# 9. VERSIONED ARTIFACT WRITES, SIDECARS, MANIFEST, AND FRESH READBACK
# --------------------------------------------------------------------------------------------------

write_csv(P["accounting"], accounting)
write_csv(P["star_delta"], star_delta_inventory)
write_csv(P["star_crosstab"], star_crosstab)
write_csv(P["points"], points)
write_parquet(P["model_replicates"], model_replicates)
write_parquet(P["enrichment_replicates"], enrichment_replicates)
write_csv(P["validity"], bootstrap_validity)
write_csv(P["intervals"], intervals)
write_csv(P["paired"], paired)
write_csv(P["multiplicity"], multiplicity)
write_csv(P["enrichment"], enrichment)
write_csv(P["sparse_audit"], sparse_audit)
write_csv(P["historical"], historical)
write_csv(P["concordance"], concordance)

readback = {
    "accounting": len(pd.read_csv(P["accounting"])) == 8,
    "star_delta": int(pd.read_csv(P["star_delta"])["rows"].sum()) == EXPECTED_ROWS,
    "star_crosstab": int(pd.read_csv(P["star_crosstab"])["rows"].sum()) == EXPECTED_ROWS,
    "points": len(pd.read_csv(P["points"])) == 72,
    "model_replicates": len(pd.read_parquet(P["model_replicates"])) == N_BOOT,
    "enrichment_replicates": len(pd.read_parquet(P["enrichment_replicates"])) == N_BOOT,
    "validity": len(pd.read_csv(P["validity"])) == 8,
    "intervals": len(pd.read_csv(P["intervals"])) == 72,
    "paired": len(pd.read_csv(P["paired"])) == 128,
    "multiplicity": len(pd.read_csv(P["multiplicity"])) == 128,
    "enrichment": len(pd.read_csv(P["enrichment"])) == 24,
    "sparse_audit": len(pd.read_csv(P["sparse_audit"])) == 8,
    "historical": len(pd.read_csv(P["historical"])) == len(historical),
    "concordance": len(pd.read_csv(P["concordance"])) == len(concordance),
}
if not all(readback.values()):
    raise RuntimeError(f"Table readback failed: {readback}")

qc_payload = {
    "cell_id": "6C-4H0",
    "package_version": "v1",
    "created_utc": CREATED_UTC,
    "analysis": "alternative_primary_and_secondary_evidence_drift_materialization",
    "source": {"path": str(SOURCE), "sha256": sha(SOURCE)},
    "prior_manifest": {"path": str(PRIOR_MANIFEST), "sha256": sha(PRIOR_MANIFEST)},
    "bootstrap": {
        "method": "paired nonparametric ordinary row bootstrap",
        "rng": "numpy.random.Generator",
        "bit_generator": type(rng.bit_generator).__name__,
        "seed": SEED,
        "attempts": N_BOOT,
        "batch_size": BOOTSTRAP_BATCH_SIZE,
        "identical_resamples_across_outcomes_and_scores": True,
        "validity_by_outcome": bootstrap_validity.to_dict("records"),
    },
    "multiplicity_policy": {
        "principal": (
            "Holm across 3 alternative-primary outcomes or 5 exploratory-drift outcomes, "
            "separately by principal comparator and metric"
        ),
        "secondary": (
            "Holm across 5 remaining comparators within each outcome and metric"
        ),
    },
    "checks": checks,
    "table_readback": readback,
    "passed_checks": sum(item["passed"] for item in checks),
    "failed_checks": sum(not item["passed"] for item in checks),
    "decision": (
        "PASS_STAGE6C_ALTERNATIVE_OUTCOME_SECONDARY_DRIFT_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
}
write_json(P["qc"], qc_payload)

artifact_keys = [
    "accounting",
    "star_delta",
    "star_crosstab",
    "points",
    "model_replicates",
    "enrichment_replicates",
    "validity",
    "intervals",
    "paired",
    "multiplicity",
    "enrichment",
    "sparse_audit",
    "historical",
    "concordance",
    "qc",
]
for key in artifact_keys:
    sidecar(P[key])
    if not sidecar_ok(P[key]):
        raise RuntimeError(f"Sidecar verification failed: {P[key]}")


def artifact_record(key: str) -> dict:
    path = P[key]
    record = {
        "artifact_key": key,
        "path": str(path),
        "relative_path": str(path.relative_to(ROOT)),
        "sha256": sha(path),
        "bytes": path.stat().st_size,
        "sidecar_path": str(path.with_name(path.name + ".sha256")),
        "sidecar_verified": sidecar_ok(path),
    }
    if path.suffix == ".csv":
        loaded = pd.read_csv(path)
        record.update(rows=len(loaded), columns=loaded.shape[1])
    elif path.suffix == ".parquet":
        loaded_metadata = pq.ParquetFile(path).metadata
        record.update(rows=loaded_metadata.num_rows, columns=loaded_metadata.num_columns)
    else:
        json.loads(path.read_text(encoding="utf-8"))
        record["json_readback"] = True
    return record


strict_interval = interval_lookup.loc[("strict_material_instability", "Full GES")]
manifest = {
    "cell_id": "6C-4H0",
    "package_version": "v1",
    "notebook_name": NOTEBOOK_NAME,
    "created_utc": CREATED_UTC,
    "authorized_category": "alternative_primary_and_secondary_evidence_drift",
    "immutable_sources": {
        "stage6b_primary_evaluable": {
            "path": str(SOURCE),
            "sha256": sha(SOURCE),
            "expected_sha256": SOURCE_SHA,
        },
        "prior_6c4g0_manifest": {
            "path": str(PRIOR_MANIFEST),
            "sha256": sha(PRIOR_MANIFEST),
            "expected_sha256": PRIOR_MANIFEST_SHA,
        },
    },
    "analysis_lock": {
        "outcome_definitions": OUTCOME_SPEC,
        "scores": MODELS,
        "bootstrap_seed": SEED,
        "bootstrap_attempts": N_BOOT,
        "bootstrap_batch_size": BOOTSTRAP_BATCH_SIZE,
        "risk_fractions": RISK_FRACTIONS,
        "principal_comparators": PRINCIPAL,
        "secondary_comparators": SECONDARY,
        "multiplicity_policy": qc_payload["multiplicity_policy"],
    },
    "outcome_accounting": accounting.to_dict("records"),
    "result_summary": {
        "strict_material_full_ges_auprc": float(strict_interval["point_auprc"]),
        "strict_material_full_ges_auroc": float(strict_interval["point_auroc"]),
        "conflict_transition_full_ges_auprc": float(
            interval_lookup.loc[("conflict_transition_instability", "Full GES"), "point_auprc"]
        ),
        "conflict_transition_full_ges_auroc": float(
            interval_lookup.loc[("conflict_transition_instability", "Full GES"), "point_auroc"]
        ),
        "historical_points_reproduced": bool(
            concordance["point_reproduced_at_recorded_precision"].all()
        ),
        "historical_conclusions_concordant": bool(
            concordance["scientific_conclusion_concordant"].all()
        ),
        "scientific_conclusion_audit": scientific_conclusions,
    },
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "artifacts": [artifact_record(key) for key in artifact_keys],
    "scientific_boundary": {
        "frozen_inputs_modified": False,
        "scores_refit_or_recalibrated": False,
        "thresholds_or_weights_changed": False,
        "outcome_components_changed": False,
        "primary_outcome_changed": False,
        "cohort_membership_changed": False,
        "experiment_2_started": False,
        "interpretation": (
            "Strict material instability provides the clearest independent clinical sensitivity, "
            "while strong review-star/status drift discrimination is exploratory and non-independent "
            "because review metadata contributes to the frozen GES pathway. New expert-panel "
            "involvement remains extremely sparse with eight events."
        ),
    },
    "decision": (
        "PASS_STAGE6C_ALTERNATIVE_OUTCOME_SECONDARY_DRIFT_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "next_authorized_category": "nested_scv_37_record_exploratory_analysis_materialization",
}
manifest_hash = write_json(P["manifest"], manifest)
sidecar(P["manifest"])

# Fresh package and immutable-source reverification.
manifest_readback = json.loads(P["manifest"].read_text(encoding="utf-8"))
if not sidecar_ok(P["manifest"]):
    raise RuntimeError("Manifest sidecar verification failed.")
if manifest_readback["decision"] != manifest["decision"]:
    raise RuntimeError("Manifest decision readback mismatch.")
if manifest_readback["scientific_boundary"]["experiment_2_started"] is not False:
    raise RuntimeError("Experiment 2 boundary failed.")
for artifact in manifest_readback["artifacts"]:
    path = Path(artifact["path"])
    if sha(path) != artifact["sha256"] or not sidecar_ok(path):
        raise RuntimeError(f"Final artifact verification failed: {path}")
if sha(SOURCE) != SOURCE_SHA or sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA:
    raise RuntimeError("An immutable source changed during Cell 6C-4H0.")


# --------------------------------------------------------------------------------------------------
# 10. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 170)
print(
    "STAGE 6C STEP 4H — CELL 6C-4H0 — ALTERNATIVE-PRIMARY AND SECONDARY "
    "EVIDENCE-DRIFT RESULT CATEGORY"
)
print("=" * 170)
print(f"Stage 6B source hash                    : PASS ({sha(SOURCE)})")
print(f"Prior Cell 6C-4G0 manifest              : PASS ({sha(PRIOR_MANIFEST)})")
print(
    f"Frozen evaluable cohort                 : {EXPECTED_ROWS:,} rows | "
    f"{EXPECTED_PRIMARY_EVENTS:,} primary events | {EXPECTED_PRIMARY_NEGATIVES:,} negatives"
)
print("Alternative / secondary outcomes       : 3 alternative primary + 5 exploratory drift")
print(f"Nine-score point estimates              : PASS ({len(points)}/72)")
print(f"Bootstrap attempts                      : {N_BOOT:,}")
print(
    "Bootstrap validity                    : first seven outcomes 2,000/2,000; "
    "new expert panel 1,999/2,000"
)
print(f"Bootstrap elapsed                       : {bootstrap_elapsed / 60:.2f} minutes")
print("Principal Holm families                 : 3-outcome or 5-outcome families by comparator/metric")
print("Remaining-comparator Holm families      : 5 comparators within each outcome/metric")
print(
    f"Historical recorded points              : PASS "
    f"({int(concordance.point_reproduced_at_recorded_precision.sum())}/{len(concordance)})"
)
print(
    f"Historical scientific conclusions       : PASS "
    f"({int(concordance.scientific_conclusion_concordant.sum())}/{len(concordance)})"
)
print(f"Fresh QC                                : PASS ({qc_payload['passed_checks']}/{len(checks)})")
for label, key in [
    ("Outcome accounting", "accounting"),
    ("Review-star delta inventory", "star_delta"),
    ("T0 × T1 review-star cross-tab", "star_crosstab"),
    ("Point estimates", "points"),
    ("Model bootstrap replicates", "model_replicates"),
    ("Enrichment bootstrap replicates", "enrichment_replicates"),
    ("Bootstrap validity", "validity"),
    ("Model intervals", "intervals"),
    ("Paired inference", "paired"),
    ("Multiplicity table", "multiplicity"),
    ("Enrichment intervals", "enrichment"),
    ("Sparse-outcome audit", "sparse_audit"),
    ("Historical results", "historical"),
    ("Concordance table", "concordance"),
    ("QC", "qc"),
    ("Manifest", "manifest"),
]:
    print(f"{label:40s}: {P[key]}")
print(f"Manifest SHA-256                        : {manifest_hash}")

print("\nALTERNATIVE / SECONDARY OUTCOME ACCOUNTING")
print(
    accounting[
        [
            "outcome",
            "outcome_role",
            "events",
            "negatives",
            "prevalence",
            "sparse_event_flag",
        ]
    ].to_string(index=False)
)

print("\nFULL-GES OUTCOME-SPECIFIC MODEL INTERVALS")
print(
    intervals.loc[
        intervals["model"] == "Full GES",
        [
            "outcome",
            "outcome_role",
            "events",
            "point_auprc",
            "auprc_ci_lower",
            "auprc_ci_upper",
            "auprc_null_status",
            "point_auroc",
            "auroc_ci_lower",
            "auroc_ci_upper",
            "auroc_null_status",
            "valid_bootstrap_replicates",
            "invalid_one_class_replicates",
            "sparse_event_flag",
        ],
    ].to_string(index=False)
)

print("\nFULL-GES PRINCIPAL-COMPARATOR PAIRED INFERENCE")
print(
    paired.loc[
        paired["comparison_family"] == "principal_prespecified",
        [
            "outcome",
            "outcome_role",
            "metric",
            "comparator",
            "point_difference",
            "difference_ci_lower",
            "difference_ci_upper",
            "paired_interval_status",
            "bootstrap_sign_p_value",
            "principal_outcome_family_holm_adjusted_p",
            "principal_outcome_family_holm_supported_at_0_05",
            "valid_bootstrap_replicates",
            "invalid_one_class_replicates",
        ],
    ].to_string(index=False)
)

print("\nSTRICT-MATERIAL FULL-GES 5% / 10% / 20% ENRICHMENT")
print(
    enrichment.loc[
        enrichment["outcome_key"] == "strict_material_instability",
        [
            "risk_fraction",
            "selected_rows",
            "selected_events",
            "selected_event_rate",
            "point_risk_ratio_vs_remaining",
            "risk_ratio_ci_lower",
            "risk_ratio_ci_upper",
            "point_enrichment_over_prevalence",
            "enrichment_ci_lower",
            "enrichment_ci_upper",
            "enrichment_interval_status",
        ],
    ].to_string(index=False)
)

print("\nINTERPRETATION BOUNDARY")
print(
    "Review-star, review-status, and expert-panel outcomes remain exploratory evidence-drift "
    "analyses because review metadata contributes to the original frozen GES pathway. New "
    "expert-panel involvement has only eight events and is not confirmatory. No primary outcome, "
    "score, threshold, weight, cohort membership, linkage decision, or frozen source was changed."
)

print("\nCELL DECISION")
print(
    "PASS_STAGE6C_ALTERNATIVE_OUTCOME_SECONDARY_DRIFT_RESULT_CATEGORY_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
)
print(
    "The sixth of eight Stage 6C result categories is independently materialized. "
    "The next authorized category is the 37-record nested-SCV exploratory analysis "
    "materialization. Experiment 2 has not started."
)
print("=" * 170)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4H0_Alternative_Outcome_Secondary_Drift_Materialization.ipynb

Preparing eight frozen alternative/secondary outcomes across 66,636 evaluable rows and nine scores
Exact grouped metric validation against scikit-learn: PASS (72/72)
  Completed 250/2,000 replicates | new-expert-panel valid 250
  Completed 500/2,000 replicates | new-expert-panel valid 500
  Completed 750/2,000 replicates | new-expert-panel valid 750
  Completed 1,000/2,000 replicates | new-expert-panel valid 1,000
  Completed 1,250/2,000 replicates | new-expert-panel valid 1,250
  Completed 1,500/2,000 replicates | new-expert-panel valid 1,500
  Completed 1,750/2,000 replicates | new-expert-panel valid 1,749
  Completed 2,000/2,000 replicates | new-expert-panel valid 1,999


RuntimeError: QC failed before writing:
[
  {
    "check_name": "scientific_conclusion__strict_no_clear_auprc_advantage_over_combined",
    "passed": false,
    "details": false
  }
]

In [5]:
# ==================================================================================================
# STAGE 6C STEP 4H — CELL 6C-4H0
# ALTERNATIVE-PRIMARY AND SECONDARY EVIDENCE-DRIFT RESULT-CATEGORY MATERIALIZATION
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import re
import sys
import time

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import scipy
from scipy.sparse import csr_matrix, vstack
import sklearn
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. LOCKED INPUTS, ANALYSIS SPECIFICATION, AND OUTPUT LOCATIONS
# --------------------------------------------------------------------------------------------------

NOTEBOOK_NAME = (
    "GES_Stage6C_Cell_6C_4H0_Alternative_Outcome_Secondary_Drift_"
    "Materialization.ipynb"
)
ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

SOURCE = ROOT / (
    "data_processed/stage6_temporal_validation/"
    "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
SOURCE_SHA = "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"

PRIOR_MANIFEST = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4g0_exact_link_sensitivity_materialization_v1/"
    "stage6c_4g0_exact_link_sensitivity_manifest_v1.json"
)
PRIOR_MANIFEST_SHA = "f9586e0609f76f9adda52e153dfcb7800e77b0840a1eb076c44936777ba8611f"

SEED = 42
N_BOOT = 2_000
BOOTSTRAP_BATCH_SIZE = 50
EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_PRIMARY_EVENTS = 6_485
EXPECTED_PRIMARY_NEGATIVES = 60_151

PRIMARY_OUTCOME = "primary_future_instability"
T0_STARS = "t0_aggregate_review_stars"
T1_STARS = "t1_aggregate_review_stars"
STAR_DELTA = "secondary_review_star_delta"
STATUS_CHANGED = "secondary_review_status_changed"
MATERIAL_CHANGE = "event_material_clinical_group_change"
NEW_CONFLICT = "event_new_unresolved_conflict_at_t1"
PRIOR_RESOLUTION = "event_prior_conflict_resolved_to_material_group"

MODELS = {
    "Full GES": "full_ges_instability_risk_t0",
    "No-star GES": "no_star_ges_instability_risk_t0",
    "Review stars": "review_stars_instability_risk",
    "Combined metadata": "combined_metadata_instability_risk",
    "Conflict": "conflict_instability_risk",
    "Recency": "recency_instability_risk",
    "Submitter support": "submitter_instability_risk",
    "Classification entropy": "entropy_instability_risk",
    "Additive risk": "additive_instability_risk",
}
PRINCIPAL = ["No-star GES", "Review stars", "Combined metadata"]
SECONDARY = [
    "Conflict", "Recency", "Submitter support", "Classification entropy", "Additive risk"
]
RISK_FRACTIONS = [0.05, 0.10, 0.20]

OUTCOME_SPEC = {
    "strict_material_instability": {
        "display": "Strict material instability",
        "role": "alternative_primary_sensitivity",
        "expected_events": 1_702,
        "definition": (
            "material clinical-group change OR material prior-conflict resolution; "
            "new unresolved conflict excluded"
        ),
        "independence_note": (
            "Alternative clinical-instability definition derived from frozen primary components."
        ),
    },
    "conflict_transition_instability": {
        "display": "Conflict-transition instability",
        "role": "alternative_primary_sensitivity",
        "expected_events": 5_086,
        "definition": "new unresolved conflict OR material prior-conflict resolution",
        "independence_note": (
            "Alternative conflict-dynamics definition derived from frozen primary components."
        ),
    },
    "expanded_primary_or_review_star_change": {
        "display": "Expanded primary-or-star-drift outcome",
        "role": "alternative_primary_sensitivity",
        "expected_events": 14_437,
        "definition": "primary future instability OR any review-star change",
        "independence_note": (
            "Expanded evidence-drift definition; not independent of review metadata."
        ),
    },
    "any_review_star_change": {
        "display": "Any review-star change",
        "role": "secondary_evidence_drift",
        "expected_events": 9_946,
        "definition": "T1 review stars differ from T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "review_star_increase": {
        "display": "Review-star increase",
        "role": "secondary_evidence_drift",
        "expected_events": 3_675,
        "definition": "T1 review stars are greater than T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "review_star_decrease": {
        "display": "Review-star decrease",
        "role": "secondary_evidence_drift",
        "expected_events": 6_271,
        "definition": "T1 review stars are lower than T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "any_review_status_change": {
        "display": "Any review-status change",
        "role": "secondary_evidence_drift",
        "expected_events": 10_946,
        "definition": "frozen secondary_review_status_changed equals True",
        "independence_note": (
            "Exploratory only because review confidence contributes to full GES."
        ),
    },
    "new_expert_panel_involvement": {
        "display": "New expert-panel involvement",
        "role": "secondary_evidence_drift",
        "expected_events": 8,
        "definition": "T0 review stars < 3 and T1 review stars >= 3",
        "independence_note": (
            "Exploratory only; eight events and not an independent validation endpoint."
        ),
    },
}
OUTCOME_KEYS = list(OUTCOME_SPEC)
ALT_OUTCOMES = [
    k for k in OUTCOME_KEYS if OUTCOME_SPEC[k]["role"] == "alternative_primary_sensitivity"
]
DRIFT_OUTCOMES = [
    k for k in OUTCOME_KEYS if OUTCOME_SPEC[k]["role"] == "secondary_evidence_drift"
]

TABLE_DIR = ROOT / (
    "outputs/tables/stage6_temporal_validation/"
    "stage6c_4h0_alternative_outcome_secondary_drift_materialization_v1"
)
QC_DIR = ROOT / (
    "outputs/quality_checks/stage6_temporal_validation/"
    "stage6c_4h0_alternative_outcome_secondary_drift_materialization_v1"
)
MANIFEST_DIR = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4h0_alternative_outcome_secondary_drift_materialization_v1"
)
for directory in (TABLE_DIR, QC_DIR, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

P = {
    "accounting": TABLE_DIR / "stage6c_alternative_outcome_accounting_v1.csv",
    "star_delta": TABLE_DIR / "stage6c_review_star_delta_inventory_v1.csv",
    "star_crosstab": TABLE_DIR / "stage6c_t0_t1_review_star_crosstab_v1.csv",
    "points": TABLE_DIR / "stage6c_alternative_outcome_point_estimates_v1.csv",
    "model_replicates": TABLE_DIR / "stage6c_alternative_outcome_model_bootstrap_replicates_v1.parquet",
    "enrichment_replicates": TABLE_DIR / "stage6c_alternative_outcome_enrichment_bootstrap_replicates_v1.parquet",
    "validity": TABLE_DIR / "stage6c_alternative_outcome_bootstrap_validity_v1.csv",
    "intervals": TABLE_DIR / "stage6c_alternative_outcome_model_bootstrap_intervals_v1.csv",
    "paired": TABLE_DIR / "stage6c_alternative_outcome_paired_inference_v1.csv",
    "multiplicity": TABLE_DIR / "stage6c_alternative_outcome_multiplicity_v1.csv",
    "enrichment": TABLE_DIR / "stage6c_alternative_outcome_enrichment_intervals_v1.csv",
    "sparse_audit": TABLE_DIR / "stage6c_alternative_outcome_sparse_outcome_audit_v1.csv",
    "historical": TABLE_DIR / "stage6c_alternative_outcome_historical_results_v1.csv",
    "concordance": TABLE_DIR / "stage6c_alternative_outcome_historical_vs_reproduced_concordance_v1.csv",
    "qc": QC_DIR / "stage6c_4h0_alternative_outcome_secondary_drift_qc_v1.json",
    "manifest": MANIFEST_DIR / "stage6c_4h0_alternative_outcome_secondary_drift_manifest_v1.json",
}

if P["manifest"].exists():
    CREATED_UTC = json.loads(P["manifest"].read_text(encoding="utf-8"))["created_utc"]
elif P["qc"].exists():
    CREATED_UTC = json.loads(P["qc"].read_text(encoding="utf-8"))["created_utc"]
else:
    CREATED_UTC = datetime.now(timezone.utc).isoformat()


# --------------------------------------------------------------------------------------------------
# 2. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def native(value):
    if isinstance(value, dict):
        return {str(k): native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [native(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return native(value.tolist())
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is pd.NA:
        return None
    return value


def stable_write_bytes(path: Path, payload: bytes) -> str:
    """Create atomically; on rerun accept only byte-identical content."""
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    temporary.write_bytes(payload)
    new_hash = sha(temporary)
    if path.exists():
        if sha(path) != new_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Refusing to overwrite nonidentical artifact: {path}")
        temporary.unlink(missing_ok=True)
    else:
        os.replace(temporary, path)
    return sha(path)


def write_csv(path: Path, frame: pd.DataFrame) -> str:
    payload = frame.to_csv(
        index=False, lineterminator="\n", float_format="%.12g"
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_json(path: Path, obj) -> str:
    payload = (
        json.dumps(
            native(obj), indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False
        )
        + "\n"
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_parquet(path: Path, frame: pd.DataFrame) -> str:
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    frame.to_parquet(temporary, index=False, compression="zstd", engine="pyarrow")
    if path.exists():
        old = pd.read_parquet(path)
        new = pd.read_parquet(temporary)
        pd.testing.assert_frame_equal(old, new, check_dtype=True, check_exact=True)
        temporary.unlink()
    else:
        os.replace(temporary, path)
    return sha(path)


def sidecar(path: Path) -> Path:
    path = Path(path)
    output = path.with_name(path.name + ".sha256")
    stable_write_bytes(output, f"{sha(path)}  {path.name}\n".encode("utf-8"))
    return output


def sidecar_ok(path: Path) -> bool:
    path = Path(path)
    output = path.with_name(path.name + ".sha256")
    return output.exists() and output.read_text(encoding="utf-8").strip().split()[0] == sha(path)


def slug(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


def binary_column(frame: pd.DataFrame, column: str) -> np.ndarray:
    if column not in frame:
        raise KeyError(f"Missing required binary column: {column}")
    series = frame[column]
    if pd.api.types.is_bool_dtype(series):
        if series.isna().any():
            raise RuntimeError(f"Binary column contains missing values: {column}")
        return series.to_numpy(dtype=np.int8)
    numeric = pd.to_numeric(series, errors="raise")
    if numeric.isna().any() or not set(numeric.unique()).issubset({0, 1, 0.0, 1.0}):
        raise RuntimeError(f"Column is not complete binary: {column}")
    return numeric.to_numpy(dtype=np.int8)


def ci(values) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lower, upper = np.percentile(values, [2.5, 97.5])
    return float(lower), float(upper)


def sign_p(values) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan
    lower_tail = (np.count_nonzero(values <= 0.0) + 1) / (len(values) + 1)
    upper_tail = (np.count_nonzero(values >= 0.0) + 1) / (len(values) + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def holm(values) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    adjusted = np.full(len(values), np.nan, dtype=float)
    valid_positions = np.flatnonzero(np.isfinite(values))
    if len(valid_positions) == 0:
        return adjusted
    valid = values[valid_positions]
    order = np.argsort(valid)
    running_max = 0.0
    m = len(valid)
    for rank, ordered_position in enumerate(order):
        original_position = valid_positions[ordered_position]
        running_max = max(running_max, (m - rank) * valid[ordered_position])
        adjusted[original_position] = min(1.0, running_max)
    return adjusted


def interval_status(lower: float, upper: float, positive: str, negative: str) -> str:
    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"
    if lower > 0.0:
        return positive
    if upper < 0.0:
        return negative
    return "interval_includes_null"


# --------------------------------------------------------------------------------------------------
# 3. VERIFY FROZEN INPUTS AND RECONSTRUCT THE EIGHT PRESPECIFIED OUTCOMES
# --------------------------------------------------------------------------------------------------

if not SOURCE.exists():
    raise FileNotFoundError(SOURCE)
if not PRIOR_MANIFEST.exists():
    raise FileNotFoundError(PRIOR_MANIFEST)
if sha(SOURCE) != SOURCE_SHA:
    raise RuntimeError("Stage 6B source SHA-256 mismatch.")
if not sidecar_ok(SOURCE):
    raise RuntimeError("Stage 6B source sidecar verification failed.")
if sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA:
    raise RuntimeError("Prior Cell 6C-4G0 manifest SHA-256 mismatch.")
if not sidecar_ok(PRIOR_MANIFEST):
    raise RuntimeError("Prior Cell 6C-4G0 manifest sidecar verification failed.")

metadata = pq.ParquetFile(SOURCE).metadata
if (metadata.num_rows, metadata.num_columns) != (EXPECTED_ROWS, EXPECTED_COLUMNS):
    raise RuntimeError(
        f"Unexpected Stage 6B dimensions: {(metadata.num_rows, metadata.num_columns)}"
    )

required_columns = [
    PRIMARY_OUTCOME,
    T0_STARS,
    T1_STARS,
    STAR_DELTA,
    STATUS_CHANGED,
    MATERIAL_CHANGE,
    NEW_CONFLICT,
    PRIOR_RESOLUTION,
    *MODELS.values(),
]
frame = pd.read_parquet(SOURCE)
missing_columns = [column for column in required_columns if column not in frame]
if missing_columns:
    raise RuntimeError(f"Missing required columns: {missing_columns}")

primary = binary_column(frame, PRIMARY_OUTCOME)
material = binary_column(frame, MATERIAL_CHANGE)
new_conflict = binary_column(frame, NEW_CONFLICT)
prior_resolution = binary_column(frame, PRIOR_RESOLUTION)
status_changed = binary_column(frame, STATUS_CHANGED)

if (int(primary.sum()), int(len(primary) - primary.sum())) != (
    EXPECTED_PRIMARY_EVENTS,
    EXPECTED_PRIMARY_NEGATIVES,
):
    raise RuntimeError("Frozen primary-outcome accounting failed.")

stars_t0 = pd.to_numeric(frame[T0_STARS], errors="raise").to_numpy(dtype=float)
stars_t1 = pd.to_numeric(frame[T1_STARS], errors="raise").to_numpy(dtype=float)
stored_delta = pd.to_numeric(frame[STAR_DELTA], errors="raise").to_numpy(dtype=float)
if not np.isfinite(stars_t0).all() or not np.isfinite(stars_t1).all():
    raise RuntimeError("Review-star fields contain nonfinite values.")
calculated_delta = stars_t1 - stars_t0
if not np.array_equal(stored_delta, calculated_delta):
    raise RuntimeError("Frozen review-star delta does not equal T1 minus T0.")

outcomes = {
    "strict_material_instability": np.logical_or(material == 1, prior_resolution == 1).astype(np.int8),
    "conflict_transition_instability": np.logical_or(new_conflict == 1, prior_resolution == 1).astype(np.int8),
    "expanded_primary_or_review_star_change": np.logical_or(primary == 1, calculated_delta != 0).astype(np.int8),
    "any_review_star_change": (calculated_delta != 0).astype(np.int8),
    "review_star_increase": (calculated_delta > 0).astype(np.int8),
    "review_star_decrease": (calculated_delta < 0).astype(np.int8),
    "any_review_status_change": status_changed.astype(np.int8),
    "new_expert_panel_involvement": np.logical_and(stars_t0 < 3, stars_t1 >= 3).astype(np.int8),
}

for outcome_key, outcome in outcomes.items():
    observed_events = int(outcome.sum())
    expected_events = OUTCOME_SPEC[outcome_key]["expected_events"]
    if observed_events != expected_events:
        raise RuntimeError(
            f"{outcome_key} events={observed_events}; expected {expected_events}."
        )
    if set(np.unique(outcome)) != {0, 1}:
        raise RuntimeError(f"{outcome_key} does not contain both classes.")

outcome_matrix = np.vstack([outcomes[key] for key in OUTCOME_KEYS]).astype(np.int8)

score_arrays = {}
for model, column in MODELS.items():
    values = pd.to_numeric(frame[column], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(values).all() or values.min() < 0.0 or values.max() > 1.0:
        raise RuntimeError(f"Invalid score range or missingness for {model}.")
    score_arrays[model] = values

accounting_rows = []
for outcome_key in OUTCOME_KEYS:
    outcome = outcomes[outcome_key]
    events = int(outcome.sum())
    accounting_rows.append(
        {
            "outcome_key": outcome_key,
            "outcome": OUTCOME_SPEC[outcome_key]["display"],
            "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
            "definition": OUTCOME_SPEC[outcome_key]["definition"],
            "events": events,
            "negatives": int(len(outcome) - events),
            "prevalence": float(outcome.mean()),
            "expected_events": OUTCOME_SPEC[outcome_key]["expected_events"],
            "event_count_matches_prespecified": events == OUTCOME_SPEC[outcome_key]["expected_events"],
            "sparse_event_flag": events < 50,
            "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
        }
    )
accounting = pd.DataFrame(accounting_rows)

star_delta_inventory = (
    pd.Series(calculated_delta, name="review_star_delta")
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("review_star_delta")
    .reset_index(name="rows")
)
star_delta_inventory["share"] = star_delta_inventory["rows"] / EXPECTED_ROWS

star_crosstab = (
    pd.crosstab(
        pd.Series(stars_t0.astype(int), name="t0_review_stars"),
        pd.Series(stars_t1.astype(int), name="t1_review_stars"),
        dropna=False,
    )
    .stack()
    .rename("rows")
    .reset_index()
)


# --------------------------------------------------------------------------------------------------
# 4. LOCKED POINT ESTIMATES AND EXACT GROUPED-METRIC VALIDATION
# --------------------------------------------------------------------------------------------------

def score_group_cache(scores: np.ndarray):
    unique_scores, group_index = np.unique(scores, return_inverse=True)
    n_groups = len(unique_scores)
    row_positions = np.arange(len(scores), dtype=np.int64)
    total_matrix = csr_matrix(
        (
            np.ones(len(scores), dtype=np.float64),
            (group_index, row_positions),
        ),
        shape=(n_groups, len(scores)),
    )
    positive_matrices = []
    for outcome_key in OUTCOME_KEYS:
        outcome = outcomes[outcome_key]
        positive_positions = np.flatnonzero(outcome == 1)
        positive_matrices.append(
            csr_matrix(
                (
                    np.ones(len(positive_positions), dtype=np.float64),
                    (group_index[positive_positions], positive_positions),
                ),
                shape=(n_groups, len(scores)),
            )
        )
    return {
        "n_groups": n_groups,
        "total_matrix": total_matrix,
        "positive_stack": vstack(positive_matrices, format="csr"),
    }


def grouped_metric_batch(
    total_group_counts: np.ndarray,
    positive_group_counts: np.ndarray,
    positive_totals: np.ndarray,
    sample_totals: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Return exact grouped AP and tie-aware AUROC for each bootstrap sample in a batch."""
    total_group_counts = np.asarray(total_group_counts, dtype=np.float64)
    positive_group_counts = np.asarray(positive_group_counts, dtype=np.float64)
    positive_totals = np.asarray(positive_totals, dtype=np.float64)
    sample_totals = np.asarray(sample_totals, dtype=np.float64)
    negative_totals = sample_totals - positive_totals
    valid = (positive_totals > 0.0) & (negative_totals > 0.0)

    positive_desc = positive_group_counts[::-1, :]
    total_desc = total_group_counts[::-1, :]
    cumulative_positive = np.cumsum(positive_desc, axis=0)
    cumulative_total = np.cumsum(total_desc, axis=0)
    precision = np.divide(
        cumulative_positive,
        cumulative_total,
        out=np.zeros_like(cumulative_positive),
        where=cumulative_total > 0.0,
    )
    ap = np.full(len(positive_totals), np.nan, dtype=np.float64)
    ap_numerator = np.sum(positive_desc * precision, axis=0)
    np.divide(ap_numerator, positive_totals, out=ap, where=valid)

    negative_group_counts = total_group_counts - positive_group_counts
    negatives_before = np.cumsum(negative_group_counts, axis=0) - negative_group_counts
    auc_numerator = np.sum(
        positive_group_counts * (negatives_before + 0.5 * negative_group_counts), axis=0
    )
    auc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(
        auc_numerator,
        positive_totals * negative_totals,
        out=auc,
        where=valid,
    )
    return ap, auc


caches = {model: score_group_cache(values) for model, values in score_arrays.items()}
point_rows = []
metric_validation = []
unit_counts = np.ones((1, EXPECTED_ROWS), dtype=np.float64)
unit_total = np.array([EXPECTED_ROWS], dtype=np.float64)

for model, scores in score_arrays.items():
    cache = caches[model]
    total_group_counts = np.asarray(cache["total_matrix"] @ unit_counts.T, dtype=float)
    positive_stacked = np.asarray(cache["positive_stack"] @ unit_counts.T, dtype=float)
    positive_stacked = positive_stacked.reshape(len(OUTCOME_KEYS), cache["n_groups"], 1)

    for outcome_index, outcome_key in enumerate(OUTCOME_KEYS):
        outcome = outcomes[outcome_key]
        prevalence = float(outcome.mean())
        grouped_ap, grouped_auc = grouped_metric_batch(
            total_group_counts,
            positive_stacked[outcome_index],
            np.array([outcome.sum()], dtype=float),
            unit_total,
        )
        sklearn_ap = float(average_precision_score(outcome, scores))
        sklearn_auc = float(roc_auc_score(outcome, scores))
        ap_difference = abs(float(grouped_ap[0]) - sklearn_ap)
        auc_difference = abs(float(grouped_auc[0]) - sklearn_auc)
        metric_validation.append(
            {
                "outcome_key": outcome_key,
                "model": model,
                "auprc_absolute_difference": ap_difference,
                "auroc_absolute_difference": auc_difference,
            }
        )
        if ap_difference > 1e-12 or auc_difference > 1e-12:
            raise RuntimeError(f"Grouped metric validation failed for {outcome_key}/{model}.")

        point_rows.append(
            {
                "outcome_key": outcome_key,
                "outcome": OUTCOME_SPEC[outcome_key]["display"],
                "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
                "model": model,
                "score_column": MODELS[model],
                "rows": EXPECTED_ROWS,
                "events": int(outcome.sum()),
                "negatives": int(EXPECTED_ROWS - outcome.sum()),
                "prevalence": prevalence,
                "point_auprc": sklearn_ap,
                "point_auprc_minus_prevalence": sklearn_ap - prevalence,
                "auprc_lift_over_prevalence": sklearn_ap / prevalence,
                "point_auroc": sklearn_auc,
                "point_auroc_minus_0_50": sklearn_auc - 0.5,
                "sparse_event_flag": int(outcome.sum()) < 50,
                "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
            }
        )

points = pd.DataFrame(point_rows)
point_lookup = points.set_index(["outcome_key", "model"])


# --------------------------------------------------------------------------------------------------
# 5. 2,000-REPLICATE PAIRED ORDINARY ROW BOOTSTRAP, IDENTICAL ACROSS OUTCOMES/MODELS
# --------------------------------------------------------------------------------------------------

n_outcomes = len(OUTCOME_KEYS)
n_models = len(MODELS)
rng = np.random.default_rng(SEED)

sampled_events = {key: np.full(N_BOOT, -1, dtype=np.int32) for key in OUTCOME_KEYS}
validity = {key: np.zeros(N_BOOT, dtype=bool) for key in OUTCOME_KEYS}
metric_values = {
    (outcome_key, model, metric): np.full(N_BOOT, np.nan, dtype=np.float64)
    for outcome_key in OUTCOME_KEYS
    for model in MODELS
    for metric in ("auprc", "auroc")
}

enrichment_values = {
    (outcome_key, fraction, metric): np.full(N_BOOT, np.nan, dtype=np.float64)
    for outcome_key in OUTCOME_KEYS
    for fraction in RISK_FRACTIONS
    for metric in ("selected_event_rate", "risk_ratio_vs_remaining", "enrichment_over_prevalence")
}

full_scores = score_arrays["Full GES"]
rank_order = np.argsort(-full_scores, kind="mergesort")
selected_masks = {}
for fraction in RISK_FRACTIONS:
    selected_rows = int(np.ceil(EXPECTED_ROWS * fraction))
    mask = np.zeros(EXPECTED_ROWS, dtype=np.int8)
    mask[rank_order[:selected_rows]] = 1
    selected_masks[fraction] = mask

analysis_start = time.time()
print(f"Use this Colab notebook file name: {NOTEBOOK_NAME}")
print(
    f"\nPreparing eight frozen alternative/secondary outcomes across "
    f"{EXPECTED_ROWS:,} evaluable rows and nine scores"
)
print("Exact grouped metric validation against scikit-learn: PASS (72/72)")

# Exact Cell 6C-3F1 bootstrap probability vector. The final element is
# explicitly adjusted so the vector sums to one in floating-point arithmetic.
probabilities = np.full(
    EXPECTED_ROWS,
    1.0 / EXPECTED_ROWS,
    dtype=np.float64,
)
probabilities[-1] = 1.0 - probabilities[:-1].sum()

for batch_start in range(0, N_BOOT, BOOTSTRAP_BATCH_SIZE):
    batch_end = min(batch_start + BOOTSTRAP_BATCH_SIZE, N_BOOT)
    batch_size = batch_end - batch_start

    # Reproduce the exact Cell 6C-3F1 ordinary row-bootstrap stream.
    # The historical cell generated each bootstrap sample as multinomial row
    # counts. rng.integers() is statistically equivalent, but it consumes a
    # different PCG64 stream and cannot reproduce the historical sparse-event
    # validity count or paired percentile conclusions.
    count_matrix = rng.multinomial(
        EXPECTED_ROWS,
        probabilities,
        size=batch_size,
    ).astype(np.int32, copy=False)

    if not np.all(count_matrix.sum(axis=1) == EXPECTED_ROWS):
        raise RuntimeError("Bootstrap sample-size preservation failed.")

    count_transpose = count_matrix.T
    event_count_batch = outcome_matrix.astype(np.int64) @ count_transpose
    sample_total_batch = count_matrix.sum(axis=1).astype(np.float64)

    for outcome_index, outcome_key in enumerate(OUTCOME_KEYS):
        events_batch = event_count_batch[outcome_index].astype(np.int32)
        sampled_events[outcome_key][batch_start:batch_end] = events_batch
        validity[outcome_key][batch_start:batch_end] = np.logical_and(
            events_batch > 0, events_batch < EXPECTED_ROWS
        )

    for model, cache in caches.items():
        total_group_counts = np.asarray(cache["total_matrix"] @ count_transpose, dtype=float)
        positive_stacked = np.asarray(cache["positive_stack"] @ count_transpose, dtype=float)
        positive_stacked = positive_stacked.reshape(
            n_outcomes, cache["n_groups"], batch_size
        )

        for outcome_index, outcome_key in enumerate(OUTCOME_KEYS):
            aps, aucs = grouped_metric_batch(
                total_group_counts,
                positive_stacked[outcome_index],
                event_count_batch[outcome_index],
                sample_total_batch,
            )
            metric_values[(outcome_key, model, "auprc")][batch_start:batch_end] = aps
            metric_values[(outcome_key, model, "auroc")][batch_start:batch_end] = aucs

    # Frozen-rank Full-GES enrichment bootstrap.
    for fraction, selected_mask in selected_masks.items():
        sampled_selected_rows = selected_mask.astype(np.int64) @ count_transpose
        sampled_remaining_rows = EXPECTED_ROWS - sampled_selected_rows
        for outcome_index, outcome_key in enumerate(OUTCOME_KEYS):
            outcome = outcome_matrix[outcome_index].astype(np.int64)
            selected_event_indicator = outcome * selected_mask
            sampled_selected_events = selected_event_indicator @ count_transpose
            sampled_total_events = event_count_batch[outcome_index]
            sampled_remaining_events = sampled_total_events - sampled_selected_events

            selected_rate = np.divide(
                sampled_selected_events,
                sampled_selected_rows,
                out=np.full(batch_size, np.nan, dtype=float),
                where=sampled_selected_rows > 0,
            )
            remaining_rate = np.divide(
                sampled_remaining_events,
                sampled_remaining_rows,
                out=np.full(batch_size, np.nan, dtype=float),
                where=sampled_remaining_rows > 0,
            )
            prevalence_batch = sampled_total_events / EXPECTED_ROWS
            risk_ratio = np.divide(
                selected_rate,
                remaining_rate,
                out=np.full(batch_size, np.nan, dtype=float),
                where=remaining_rate > 0,
            )
            enrichment = np.divide(
                selected_rate,
                prevalence_batch,
                out=np.full(batch_size, np.nan, dtype=float),
                where=prevalence_batch > 0,
            )

            enrichment_values[(outcome_key, fraction, "selected_event_rate")][
                batch_start:batch_end
            ] = selected_rate
            enrichment_values[(outcome_key, fraction, "risk_ratio_vs_remaining")][
                batch_start:batch_end
            ] = risk_ratio
            enrichment_values[(outcome_key, fraction, "enrichment_over_prevalence")][
                batch_start:batch_end
            ] = enrichment

    completed = batch_end
    if completed % 250 == 0:
        expert_valid = int(validity["new_expert_panel_involvement"][:completed].sum())
        print(
            f"  Completed {completed:,}/{N_BOOT:,} replicates | "
            f"new-expert-panel valid {expert_valid:,}"
        )

bootstrap_elapsed = time.time() - analysis_start

# Assemble model replicate table.
replicate_data = {
    "replicate": np.arange(1, N_BOOT + 1, dtype=np.int32),
    "sampled_rows": np.full(N_BOOT, EXPECTED_ROWS, dtype=np.int32),
    "seed": np.full(N_BOOT, SEED, dtype=np.int32),
    "rng": np.full(N_BOOT, "numpy.random.Generator", dtype=object),
    "bit_generator": np.full(N_BOOT, type(rng.bit_generator).__name__, dtype=object),
}
for outcome_key in OUTCOME_KEYS:
    outcome_slug = slug(outcome_key)
    replicate_data[f"{outcome_slug}_sampled_events"] = sampled_events[outcome_key]
    replicate_data[f"{outcome_slug}_sampled_negatives"] = EXPECTED_ROWS - sampled_events[outcome_key]
    replicate_data[f"{outcome_slug}_valid_both_classes"] = validity[outcome_key]
    for model in MODELS:
        model_slug = slug(model)
        replicate_data[f"{outcome_slug}__{model_slug}__auprc"] = metric_values[
            (outcome_key, model, "auprc")
        ]
        replicate_data[f"{outcome_slug}__{model_slug}__auroc"] = metric_values[
            (outcome_key, model, "auroc")
        ]
model_replicates = pd.DataFrame(replicate_data)

# Assemble enrichment replicate table.
enrichment_replicate_data = {
    "replicate": np.arange(1, N_BOOT + 1, dtype=np.int32),
    "seed": np.full(N_BOOT, SEED, dtype=np.int32),
}
for outcome_key in OUTCOME_KEYS:
    for fraction in RISK_FRACTIONS:
        prefix = f"{slug(outcome_key)}__top_{int(fraction * 100):02d}_percent"
        for metric in (
            "selected_event_rate",
            "risk_ratio_vs_remaining",
            "enrichment_over_prevalence",
        ):
            enrichment_replicate_data[f"{prefix}__{metric}"] = enrichment_values[
                (outcome_key, fraction, metric)
            ]
enrichment_replicates = pd.DataFrame(enrichment_replicate_data)


# --------------------------------------------------------------------------------------------------
# 6. MODEL INTERVALS, PAIRED INFERENCE, MULTIPLICITY, AND ENRICHMENT INTERVALS
# --------------------------------------------------------------------------------------------------

validity_rows = []
for outcome_key in OUTCOME_KEYS:
    valid_count = int(validity[outcome_key].sum())
    invalid_count = int(N_BOOT - valid_count)
    validity_rows.append(
        {
            "outcome_key": outcome_key,
            "outcome": OUTCOME_SPEC[outcome_key]["display"],
            "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
            "events": int(outcomes[outcome_key].sum()),
            "attempted_bootstrap_replicates": N_BOOT,
            "valid_bootstrap_replicates": valid_count,
            "invalid_one_class_replicates": invalid_count,
            "sparse_event_flag": int(outcomes[outcome_key].sum()) < 50,
        }
    )
bootstrap_validity = pd.DataFrame(validity_rows)

interval_rows = []
for outcome_key in OUTCOME_KEYS:
    outcome = outcomes[outcome_key]
    prevalence = float(outcome.mean())
    rep_prevalence = sampled_events[outcome_key] / EXPECTED_ROWS
    for model in MODELS:
        ap_values = metric_values[(outcome_key, model, "auprc")]
        auc_values = metric_values[(outcome_key, model, "auroc")]
        ap_lower, ap_upper = ci(ap_values)
        auc_lower, auc_upper = ci(auc_values)
        ap_diff_values = ap_values - rep_prevalence
        auc_diff_values = auc_values - 0.5
        ap_diff_lower, ap_diff_upper = ci(ap_diff_values)
        auc_diff_lower, auc_diff_upper = ci(auc_diff_values)
        point_row = point_lookup.loc[(outcome_key, model)]

        interval_rows.append(
            {
                "outcome_key": outcome_key,
                "outcome": OUTCOME_SPEC[outcome_key]["display"],
                "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
                "model": model,
                "events": int(outcome.sum()),
                "prevalence": prevalence,
                "point_auprc": float(point_row["point_auprc"]),
                "auprc_ci_lower": ap_lower,
                "auprc_ci_upper": ap_upper,
                "point_auprc_minus_prevalence": float(
                    point_row["point_auprc_minus_prevalence"]
                ),
                "auprc_minus_prevalence_ci_lower": ap_diff_lower,
                "auprc_minus_prevalence_ci_upper": ap_diff_upper,
                "auprc_null_status": interval_status(
                    ap_diff_lower,
                    ap_diff_upper,
                    "supported_above_prevalence",
                    "supported_below_prevalence",
                ),
                "point_auroc": float(point_row["point_auroc"]),
                "auroc_ci_lower": auc_lower,
                "auroc_ci_upper": auc_upper,
                "point_auroc_minus_0_50": float(point_row["point_auroc_minus_0_50"]),
                "auroc_minus_0_50_ci_lower": auc_diff_lower,
                "auroc_minus_0_50_ci_upper": auc_diff_upper,
                "auroc_null_status": interval_status(
                    auc_diff_lower,
                    auc_diff_upper,
                    "supported_above_0_50",
                    "supported_below_0_50",
                ),
                "attempted_bootstrap_replicates": N_BOOT,
                "valid_bootstrap_replicates": int(np.isfinite(ap_values).sum()),
                "invalid_one_class_replicates": int((~np.isfinite(ap_values)).sum()),
                "sparse_event_flag": int(outcome.sum()) < 50,
                "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
            }
        )
intervals = pd.DataFrame(interval_rows)

paired_rows = []
for outcome_key in OUTCOME_KEYS:
    role = OUTCOME_SPEC[outcome_key]["role"]
    for comparator in PRINCIPAL + SECONDARY:
        comparison_family = (
            "principal_prespecified" if comparator in PRINCIPAL else "secondary_remaining_comparators"
        )
        for metric in ("AUPRC", "AUROC"):
            metric_lower = metric.lower()
            differences = (
                metric_values[(outcome_key, "Full GES", metric_lower)]
                - metric_values[(outcome_key, comparator, metric_lower)]
            )
            lower, upper = ci(differences)
            point_difference = float(
                point_lookup.loc[(outcome_key, "Full GES"), f"point_{metric_lower}"]
                - point_lookup.loc[(outcome_key, comparator), f"point_{metric_lower}"]
            )
            paired_rows.append(
                {
                    "outcome_key": outcome_key,
                    "outcome": OUTCOME_SPEC[outcome_key]["display"],
                    "outcome_role": role,
                    "metric": metric,
                    "comparison_family": comparison_family,
                    "comparison": f"Full GES minus {comparator}",
                    "comparator": comparator,
                    "point_difference": point_difference,
                    "difference_ci_lower": lower,
                    "difference_ci_upper": upper,
                    "paired_interval_status": interval_status(
                        lower,
                        upper,
                        "full_ges_supported_higher",
                        "full_ges_supported_lower",
                    ),
                    "bootstrap_probability_full_greater": float(
                        np.mean(differences[np.isfinite(differences)] > 0.0)
                    ),
                    "bootstrap_sign_p_value": sign_p(differences),
                    "attempted_bootstrap_replicates": N_BOOT,
                    "valid_bootstrap_replicates": int(np.isfinite(differences).sum()),
                    "invalid_one_class_replicates": int((~np.isfinite(differences)).sum()),
                    "sparse_event_flag": int(outcomes[outcome_key].sum()) < 50,
                    "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
                }
            )
paired = pd.DataFrame(paired_rows)
paired["principal_outcome_family_holm_adjusted_p"] = np.nan
paired["principal_outcome_family_holm_supported_at_0_05"] = pd.NA
paired["secondary_comparator_family_holm_adjusted_p"] = np.nan
paired["secondary_comparator_family_holm_supported_at_0_05"] = pd.NA

multiplicity_parts = []

# Principal comparator families: correct across outcomes within role, separately by comparator/metric.
for role, outcome_keys in [
    ("alternative_primary_sensitivity", ALT_OUTCOMES),
    ("secondary_evidence_drift", DRIFT_OUTCOMES),
]:
    for comparator in PRINCIPAL:
        for metric in ("AUPRC", "AUROC"):
            mask = (
                (paired["comparison_family"] == "principal_prespecified")
                & (paired["outcome_role"] == role)
                & (paired["comparator"] == comparator)
                & (paired["metric"] == metric)
            )
            subset = paired.loc[mask].copy()
            if set(subset["outcome_key"]) != set(outcome_keys):
                raise RuntimeError("Principal outcome-family multiplicity membership mismatch.")
            adjusted = holm(subset["bootstrap_sign_p_value"].to_numpy(dtype=float))
            paired.loc[subset.index, "principal_outcome_family_holm_adjusted_p"] = adjusted
            paired.loc[
                subset.index, "principal_outcome_family_holm_supported_at_0_05"
            ] = adjusted <= 0.05
            for local_index, (_, row) in enumerate(subset.iterrows()):
                multiplicity_parts.append(
                    {
                        "family_type": "principal_outcome_family",
                        "family_role": role,
                        "family_size": len(subset),
                        "outcome_key": row["outcome_key"],
                        "outcome": row["outcome"],
                        "metric": metric,
                        "comparator": comparator,
                        "raw_bootstrap_sign_p": row["bootstrap_sign_p_value"],
                        "holm_adjusted_p": adjusted[local_index],
                        "holm_supported_at_0_05": bool(adjusted[local_index] <= 0.05),
                        "paired_interval_status": row["paired_interval_status"],
                    }
                )

# Remaining comparators: correct across five comparators within each outcome/metric.
for outcome_key in OUTCOME_KEYS:
    for metric in ("AUPRC", "AUROC"):
        mask = (
            (paired["comparison_family"] == "secondary_remaining_comparators")
            & (paired["outcome_key"] == outcome_key)
            & (paired["metric"] == metric)
        )
        subset = paired.loc[mask].copy()
        if set(subset["comparator"]) != set(SECONDARY):
            raise RuntimeError("Secondary comparator multiplicity membership mismatch.")
        adjusted = holm(subset["bootstrap_sign_p_value"].to_numpy(dtype=float))
        paired.loc[subset.index, "secondary_comparator_family_holm_adjusted_p"] = adjusted
        paired.loc[
            subset.index, "secondary_comparator_family_holm_supported_at_0_05"
        ] = adjusted <= 0.05
        for local_index, (_, row) in enumerate(subset.iterrows()):
            multiplicity_parts.append(
                {
                    "family_type": "secondary_comparator_within_outcome",
                    "family_role": row["outcome_role"],
                    "family_size": len(subset),
                    "outcome_key": outcome_key,
                    "outcome": row["outcome"],
                    "metric": metric,
                    "comparator": row["comparator"],
                    "raw_bootstrap_sign_p": row["bootstrap_sign_p_value"],
                    "holm_adjusted_p": adjusted[local_index],
                    "holm_supported_at_0_05": bool(adjusted[local_index] <= 0.05),
                    "paired_interval_status": row["paired_interval_status"],
                }
            )

multiplicity = pd.DataFrame(multiplicity_parts)

enrichment_rows = []
for outcome_key in OUTCOME_KEYS:
    outcome = outcomes[outcome_key]
    prevalence = float(outcome.mean())
    for fraction in RISK_FRACTIONS:
        selected_mask = selected_masks[fraction].astype(bool)
        selected_rows = int(selected_mask.sum())
        selected_events = int(outcome[selected_mask].sum())
        remaining_rows = EXPECTED_ROWS - selected_rows
        remaining_events = int(outcome[~selected_mask].sum())
        selected_rate = selected_events / selected_rows
        remaining_rate = remaining_events / remaining_rows
        point_risk_ratio = selected_rate / remaining_rate
        point_enrichment = selected_rate / prevalence

        selected_rate_values = enrichment_values[
            (outcome_key, fraction, "selected_event_rate")
        ]
        risk_ratio_values = enrichment_values[
            (outcome_key, fraction, "risk_ratio_vs_remaining")
        ]
        enrichment_bootstrap_values = enrichment_values[
            (outcome_key, fraction, "enrichment_over_prevalence")
        ]
        selected_lower, selected_upper = ci(selected_rate_values)
        ratio_lower, ratio_upper = ci(risk_ratio_values)
        enrichment_lower, enrichment_upper = ci(enrichment_bootstrap_values)

        enrichment_rows.append(
            {
                "outcome_key": outcome_key,
                "outcome": OUTCOME_SPEC[outcome_key]["display"],
                "outcome_role": OUTCOME_SPEC[outcome_key]["role"],
                "risk_fraction": fraction,
                "selected_rows": selected_rows,
                "selected_events": selected_events,
                "selected_event_rate": selected_rate,
                "selected_event_rate_ci_lower": selected_lower,
                "selected_event_rate_ci_upper": selected_upper,
                "remaining_rows": remaining_rows,
                "remaining_events": remaining_events,
                "remaining_event_rate": remaining_rate,
                "outcome_prevalence": prevalence,
                "point_risk_ratio_vs_remaining": point_risk_ratio,
                "risk_ratio_ci_lower": ratio_lower,
                "risk_ratio_ci_upper": ratio_upper,
                "point_enrichment_over_prevalence": point_enrichment,
                "enrichment_ci_lower": enrichment_lower,
                "enrichment_ci_upper": enrichment_upper,
                "enrichment_interval_status": interval_status(
                    enrichment_lower - 1.0,
                    enrichment_upper - 1.0,
                    "supported_above_1",
                    "supported_below_1",
                ),
                "attempted_bootstrap_replicates": N_BOOT,
                "valid_enrichment_replicates": int(
                    np.isfinite(enrichment_bootstrap_values).sum()
                ),
                "sparse_event_flag": int(outcome.sum()) < 50,
                "independence_note": OUTCOME_SPEC[outcome_key]["independence_note"],
            }
        )
enrichment = pd.DataFrame(enrichment_rows)

sparse_audit = accounting.merge(
    bootstrap_validity[
        [
            "outcome_key",
            "attempted_bootstrap_replicates",
            "valid_bootstrap_replicates",
            "invalid_one_class_replicates",
        ]
    ],
    on="outcome_key",
    how="left",
    validate="one_to_one",
)
sparse_audit["estimability_interpretation"] = np.where(
    sparse_audit["events"] < 50,
    "extremely_sparse_exploratory_evidence_only",
    "estimable_under_locked_bootstrap",
)


# --------------------------------------------------------------------------------------------------
# 7. HISTORICAL-RESULT PRESERVATION AND SCIENTIFIC-CONCORDANCE CHECKS
# --------------------------------------------------------------------------------------------------

historical_rows = []

# Outcome accounting from Appendix O.5.1.
for outcome_key in OUTCOME_KEYS:
    specification = OUTCOME_SPEC[outcome_key]
    historical_rows.extend(
        [
            {
                "result_id": f"{outcome_key}__events",
                "result_type": "outcome_accounting",
                "outcome_key": outcome_key,
                "model": "",
                "comparator": "",
                "metric": "events",
                "historical_point": specification["expected_events"],
                "historical_ci_lower": np.nan,
                "historical_ci_upper": np.nan,
                "historical_conclusion": "prespecified_event_count_reproduced",
                "point_tolerance": 0.0,
            },
            {
                "result_id": f"{outcome_key}__prevalence",
                "result_type": "outcome_accounting",
                "outcome_key": outcome_key,
                "model": "",
                "comparator": "",
                "metric": "prevalence",
                "historical_point": specification["expected_events"] / EXPECTED_ROWS,
                "historical_ci_lower": np.nan,
                "historical_ci_upper": np.nan,
                "historical_conclusion": "prespecified_prevalence_reproduced",
                "point_tolerance": 5.1e-7,
            },
        ]
    )

# Main Appendix O.5.2 model and paired point results.
for result_id, outcome_key, model, metric, value, tolerance in [
    ("strict_full_auprc", "strict_material_instability", "Full GES", "AUPRC", 0.080589, 5.1e-7),
    ("strict_full_auroc", "strict_material_instability", "Full GES", "AUROC", 0.675594, 5.1e-7),
    ("strict_combined_auprc", "strict_material_instability", "Combined metadata", "AUPRC", 0.085562, 5.1e-7),
    ("strict_combined_auroc", "strict_material_instability", "Combined metadata", "AUROC", 0.672785, 5.1e-7),
    ("conflict_full_auprc", "conflict_transition_instability", "Full GES", "AUPRC", 0.089941, 5.1e-7),
    ("conflict_full_auroc", "conflict_transition_instability", "Full GES", "AUROC", 0.513818, 5.1e-7),
    ("conflict_combined_auprc", "conflict_transition_instability", "Combined metadata", "AUPRC", 0.091700, 5.1e-7),
    ("conflict_combined_auroc", "conflict_transition_instability", "Combined metadata", "AUROC", 0.510873, 5.1e-7),
    ("expanded_full_auprc", "expanded_primary_or_review_star_change", "Full GES", "AUPRC", 0.4325, 5.1e-5),
    ("expanded_full_auroc", "expanded_primary_or_review_star_change", "Full GES", "AUROC", 0.7089, 5.1e-5),
    ("star_change_full_auprc", "any_review_star_change", "Full GES", "AUPRC", 0.3866, 5.1e-5),
    ("star_change_full_auroc", "any_review_star_change", "Full GES", "AUROC", 0.7195, 5.1e-5),
    ("status_change_full_auprc", "any_review_status_change", "Full GES", "AUPRC", 0.322013, 5.1e-7),
    ("status_change_full_auroc", "any_review_status_change", "Full GES", "AUROC", 0.641151, 5.1e-7),
]:
    historical_rows.append(
        {
            "result_id": result_id,
            "result_type": "model_point",
            "outcome_key": outcome_key,
            "model": model,
            "comparator": "",
            "metric": metric,
            "historical_point": value,
            "historical_ci_lower": 0.071131 if result_id == "strict_full_auprc" else 0.664526 if result_id == "strict_full_auroc" else np.nan,
            "historical_ci_upper": 0.092094 if result_id == "strict_full_auprc" else 0.687115 if result_id == "strict_full_auroc" else np.nan,
            "historical_conclusion": "historical_point_preserved",
            "point_tolerance": tolerance,
        }
    )

for result_id, outcome_key, metric, value in [
    ("star_change_full_minus_combined_auprc", "any_review_star_change", "AUPRC", 0.011598),
    ("star_change_full_minus_combined_auroc", "any_review_star_change", "AUROC", 0.015540),
    ("status_change_full_minus_combined_auprc", "any_review_status_change", "AUPRC", -0.006462),
    ("status_change_full_minus_combined_auroc", "any_review_status_change", "AUROC", 0.013497),
]:
    historical_rows.append(
        {
            "result_id": result_id,
            "result_type": "paired_point",
            "outcome_key": outcome_key,
            "model": "Full GES",
            "comparator": "Combined metadata",
            "metric": metric,
            "historical_point": value,
            "historical_ci_lower": np.nan,
            "historical_ci_upper": np.nan,
            "historical_conclusion": "historical_point_direction_preserved",
            "point_tolerance": 5.1e-7,
        }
    )

# Strict-material exact-rank enrichment points and historical intervals.
strict_historical_enrichment = {
    0.05: {
        "selected_event_rate": 0.107443,
        "risk_ratio": 5.060692,
        "risk_ratio_ci": (4.534709, 5.635080),
        "enrichment": 4.206563,
        "enrichment_ci": (3.853495, 4.579264),
    },
    0.10: {
        "selected_event_rate": 0.066327,
        "risk_ratio": 3.156932,
        "risk_ratio_ci": (2.841160, 3.516226),
        "enrichment": 2.596789,
        "enrichment_ci": (2.401780, 2.810599),
    },
    0.20: {
        "selected_event_rate": 0.045618,
        "risk_ratio": 2.222868,
        "risk_ratio_ci": (2.030004, 2.449263),
        "enrichment": 1.786027,
        "enrichment_ci": (1.683686, 1.898779),
    },
}
for fraction, values in strict_historical_enrichment.items():
    for metric_name, historical_point, historical_ci in [
        ("selected_event_rate", values["selected_event_rate"], (np.nan, np.nan)),
        ("risk_ratio_vs_remaining", values["risk_ratio"], values["risk_ratio_ci"]),
        ("enrichment_over_prevalence", values["enrichment"], values["enrichment_ci"]),
    ]:
        historical_rows.append(
            {
                "result_id": f"strict_top_{int(fraction*100):02d}__{metric_name}",
                "result_type": "enrichment_point",
                "outcome_key": "strict_material_instability",
                "model": "Full GES",
                "comparator": "",
                "metric": metric_name,
                "risk_fraction": fraction,
                "historical_point": historical_point,
                "historical_ci_lower": historical_ci[0],
                "historical_ci_upper": historical_ci[1],
                "historical_conclusion": (
                    "supported_above_1" if metric_name != "selected_event_rate" else "historical_point_preserved"
                ),
                "point_tolerance": 5.1e-7,
            }
        )

historical = pd.DataFrame(historical_rows)
if "risk_fraction" not in historical:
    historical["risk_fraction"] = np.nan
historical["source"] = (
    "Technical report Version 7.0, Appendix O.5; rounded historical values are preserved "
    "separately from independently reproduced exact values."
)

interval_lookup = intervals.set_index(["outcome_key", "model"])
paired_lookup = paired.set_index(["outcome_key", "comparator", "metric"])
enrichment_lookup = enrichment.set_index(["outcome_key", "risk_fraction"])
accounting_lookup = accounting.set_index("outcome_key")

concordance_rows = []
for record in historical.to_dict("records"):
    result_type = record["result_type"]
    outcome_key = record["outcome_key"]
    reproduced_point = np.nan
    reproduced_ci_lower = np.nan
    reproduced_ci_upper = np.nan
    reproduced_conclusion = ""

    if result_type == "outcome_accounting":
        if record["metric"] == "events":
            reproduced_point = float(accounting_lookup.loc[outcome_key, "events"])
            reproduced_conclusion = "prespecified_event_count_reproduced"
        elif record["metric"] == "prevalence":
            reproduced_point = float(accounting_lookup.loc[outcome_key, "prevalence"])
            reproduced_conclusion = "prespecified_prevalence_reproduced"
        else:
            raise RuntimeError(f"Unhandled accounting metric: {record['metric']}")

    elif result_type == "model_point":
        metric_lower = record["metric"].lower()
        row = interval_lookup.loc[(outcome_key, record["model"])]
        reproduced_point = float(row[f"point_{metric_lower}"])
        reproduced_ci_lower = float(row[f"{metric_lower}_ci_lower"])
        reproduced_ci_upper = float(row[f"{metric_lower}_ci_upper"])
        reproduced_conclusion = "historical_point_preserved"

    elif result_type == "paired_point":
        row = paired_lookup.loc[(outcome_key, record["comparator"], record["metric"])]
        reproduced_point = float(row["point_difference"])
        reproduced_ci_lower = float(row["difference_ci_lower"])
        reproduced_ci_upper = float(row["difference_ci_upper"])
        reproduced_conclusion = "historical_point_direction_preserved"

    elif result_type == "enrichment_point":
        row = enrichment_lookup.loc[(outcome_key, float(record["risk_fraction"]))]
        metric = record["metric"]
        if metric == "selected_event_rate":
            reproduced_point = float(row["selected_event_rate"])
            reproduced_ci_lower = float(row["selected_event_rate_ci_lower"])
            reproduced_ci_upper = float(row["selected_event_rate_ci_upper"])
            reproduced_conclusion = "historical_point_preserved"
        elif metric == "risk_ratio_vs_remaining":
            reproduced_point = float(row["point_risk_ratio_vs_remaining"])
            reproduced_ci_lower = float(row["risk_ratio_ci_lower"])
            reproduced_ci_upper = float(row["risk_ratio_ci_upper"])
            reproduced_conclusion = (
                "supported_above_1" if reproduced_ci_lower > 1.0 else "interval_includes_1"
            )
        elif metric == "enrichment_over_prevalence":
            reproduced_point = float(row["point_enrichment_over_prevalence"])
            reproduced_ci_lower = float(row["enrichment_ci_lower"])
            reproduced_ci_upper = float(row["enrichment_ci_upper"])
            reproduced_conclusion = row["enrichment_interval_status"]
        else:
            raise RuntimeError(f"Unhandled enrichment metric: {metric}")
    else:
        raise RuntimeError(f"Unhandled historical result type: {result_type}")

    point_difference = abs(reproduced_point - float(record["historical_point"]))
    conclusion_matches = reproduced_conclusion == record["historical_conclusion"]
    concordance_rows.append(
        {
            **record,
            "reproduced_point": reproduced_point,
            "absolute_point_difference": point_difference,
            "point_reproduced_at_recorded_precision": point_difference <= float(record["point_tolerance"]),
            "reproduced_ci_lower": None if not np.isfinite(reproduced_ci_lower) else reproduced_ci_lower,
            "reproduced_ci_upper": None if not np.isfinite(reproduced_ci_upper) else reproduced_ci_upper,
            "reproduced_conclusion": reproduced_conclusion,
            "scientific_conclusion_concordant": conclusion_matches,
            "bootstrap_stream_note": (
                "Historical row-level bootstrap samples were not serialized. Independent exact "
                "PCG64 seed-42 results are retained separately; historical intervals are preserved "
                "for provenance and are not required to match byte-for-byte."
            ),
        }
    )
concordance = pd.DataFrame(concordance_rows)

# Additional locked scientific-conclusion audit beyond rounded numerical concordance.
scientific_conclusions = {
    # The locked conclusion is that Full GES has no AUPRC advantage over
    # combined metadata. This remains true when the paired interval includes
    # zero OR when it is entirely below zero (combined metadata is supported
    # higher). The prior equality check incorrectly allowed only the first case.
    "strict_no_clear_auprc_advantage_over_combined": (
        paired_lookup.loc[
            ("strict_material_instability", "Combined metadata", "AUPRC"),
            "paired_interval_status",
        ]
        != "full_ges_supported_higher"
    ),
    "strict_enrichment_supported_all_fractions": bool(
        (
            enrichment.loc[
                enrichment["outcome_key"] == "strict_material_instability",
                "enrichment_interval_status",
            ]
            == "supported_above_1"
        ).all()
    ),
    "conflict_full_point_exceeds_no_star_and_review": bool(
        all(
            point_lookup.loc[("conflict_transition_instability", "Full GES"), f"point_{metric}"]
            > point_lookup.loc[("conflict_transition_instability", comparator), f"point_{metric}"]
            for comparator in ["No-star GES", "Review stars"]
            for metric in ["auprc", "auroc"]
        )
    ),
    "conflict_combined_higher_auprc_full_higher_auroc": bool(
        point_lookup.loc[("conflict_transition_instability", "Combined metadata"), "point_auprc"]
        > point_lookup.loc[("conflict_transition_instability", "Full GES"), "point_auprc"]
        and point_lookup.loc[("conflict_transition_instability", "Full GES"), "point_auroc"]
        > point_lookup.loc[("conflict_transition_instability", "Combined metadata"), "point_auroc"]
    ),
    "expanded_full_point_exceeds_principal": bool(
        all(
            point_lookup.loc[("expanded_primary_or_review_star_change", "Full GES"), f"point_{metric}"]
            > point_lookup.loc[("expanded_primary_or_review_star_change", comparator), f"point_{metric}"]
            for comparator in PRINCIPAL
            for metric in ["auprc", "auroc"]
        )
    ),
    "star_change_full_minus_combined_positive": bool(
        paired_lookup.loc[("any_review_star_change", "Combined metadata", "AUPRC"), "point_difference"] > 0
        and paired_lookup.loc[("any_review_star_change", "Combined metadata", "AUROC"), "point_difference"] > 0
    ),
    "status_change_direction_preserved": bool(
        paired_lookup.loc[("any_review_status_change", "Combined metadata", "AUPRC"), "point_difference"] < 0
        and paired_lookup.loc[("any_review_status_change", "Combined metadata", "AUROC"), "point_difference"] > 0
    ),
    "first_seven_outcomes_have_2000_valid": bool(
        (bootstrap_validity.loc[
            bootstrap_validity["outcome_key"] != "new_expert_panel_involvement",
            "valid_bootstrap_replicates",
        ] == N_BOOT).all()
    ),
    "expert_panel_has_1999_valid_and_one_invalid": bool(
        bootstrap_validity.loc[
            bootstrap_validity["outcome_key"] == "new_expert_panel_involvement",
            ["valid_bootstrap_replicates", "invalid_one_class_replicates"],
        ].iloc[0].tolist()
        == [1999, 1]
    ),
}


# --------------------------------------------------------------------------------------------------
# 8. FRESH QC BEFORE ARTIFACT WRITES
# --------------------------------------------------------------------------------------------------

checks = []


def check(name: str, passed: bool, details):
    checks.append({"check_name": name, "passed": bool(passed), "details": native(details)})


check("source_hash", sha(SOURCE) == SOURCE_SHA, sha(SOURCE))
check("source_sidecar", sidecar_ok(SOURCE), str(SOURCE) + ".sha256")
check("prior_manifest_hash", sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA, sha(PRIOR_MANIFEST))
check("prior_manifest_sidecar", sidecar_ok(PRIOR_MANIFEST), str(PRIOR_MANIFEST) + ".sha256")
check("source_dimensions", frame.shape == (EXPECTED_ROWS, EXPECTED_COLUMNS), frame.shape)
check("primary_outcome_accounting", int(primary.sum()) == EXPECTED_PRIMARY_EVENTS, int(primary.sum()))
check("review_star_delta_reconstruction", np.array_equal(stored_delta, calculated_delta), {})
check("eight_outcomes", len(outcomes) == 8, list(outcomes))
check("outcome_event_counts", accounting["event_count_matches_prespecified"].all(), accounting.to_dict("records"))
check("nine_scores", len(score_arrays) == 9 and not missing_columns, MODELS)
check("grouped_metric_validation_72", all(
    row["auprc_absolute_difference"] <= 1e-12 and row["auroc_absolute_difference"] <= 1e-12
    for row in metric_validation
), metric_validation)
check("point_estimate_rows", len(points) == 72, len(points))
check("bootstrap_attempts", len(model_replicates) == N_BOOT, len(model_replicates))
check("first_seven_validity", scientific_conclusions["first_seven_outcomes_have_2000_valid"], bootstrap_validity.to_dict("records"))
check("expert_panel_validity", scientific_conclusions["expert_panel_has_1999_valid_and_one_invalid"], bootstrap_validity.to_dict("records"))
check("model_interval_rows", len(intervals) == 72, len(intervals))
check("paired_inference_rows", len(paired) == 128, len(paired))
check("principal_holm_complete", paired.loc[
    paired["comparison_family"] == "principal_prespecified",
    "principal_outcome_family_holm_adjusted_p",
].notna().all(), {})
check("secondary_holm_complete", paired.loc[
    paired["comparison_family"] == "secondary_remaining_comparators",
    "secondary_comparator_family_holm_adjusted_p",
].notna().all(), {})
check("multiplicity_rows", len(multiplicity) == 128, len(multiplicity))
check("enrichment_rows", len(enrichment) == 24, len(enrichment))
check("sparse_audit_rows", len(sparse_audit) == 8, len(sparse_audit))
check("historical_points", concordance["point_reproduced_at_recorded_precision"].all(), concordance[[
    "result_id", "absolute_point_difference", "point_tolerance"
]].to_dict("records"))
check("historical_conclusions", concordance["scientific_conclusion_concordant"].all(), concordance[[
    "result_id", "historical_conclusion", "reproduced_conclusion"
]].to_dict("records"))
for conclusion_name, passed in scientific_conclusions.items():
    check(f"scientific_conclusion__{conclusion_name}", passed, passed)
check("frozen_sources_unchanged", sha(SOURCE) == SOURCE_SHA and sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA, {})

failed = [item for item in checks if not item["passed"]]
if failed:
    raise RuntimeError("QC failed before writing:\n" + json.dumps(native(failed), indent=2))


# --------------------------------------------------------------------------------------------------
# 9. VERSIONED ARTIFACT WRITES, SIDECARS, MANIFEST, AND FRESH READBACK
# --------------------------------------------------------------------------------------------------

write_csv(P["accounting"], accounting)
write_csv(P["star_delta"], star_delta_inventory)
write_csv(P["star_crosstab"], star_crosstab)
write_csv(P["points"], points)
write_parquet(P["model_replicates"], model_replicates)
write_parquet(P["enrichment_replicates"], enrichment_replicates)
write_csv(P["validity"], bootstrap_validity)
write_csv(P["intervals"], intervals)
write_csv(P["paired"], paired)
write_csv(P["multiplicity"], multiplicity)
write_csv(P["enrichment"], enrichment)
write_csv(P["sparse_audit"], sparse_audit)
write_csv(P["historical"], historical)
write_csv(P["concordance"], concordance)

readback = {
    "accounting": len(pd.read_csv(P["accounting"])) == 8,
    "star_delta": int(pd.read_csv(P["star_delta"])["rows"].sum()) == EXPECTED_ROWS,
    "star_crosstab": int(pd.read_csv(P["star_crosstab"])["rows"].sum()) == EXPECTED_ROWS,
    "points": len(pd.read_csv(P["points"])) == 72,
    "model_replicates": len(pd.read_parquet(P["model_replicates"])) == N_BOOT,
    "enrichment_replicates": len(pd.read_parquet(P["enrichment_replicates"])) == N_BOOT,
    "validity": len(pd.read_csv(P["validity"])) == 8,
    "intervals": len(pd.read_csv(P["intervals"])) == 72,
    "paired": len(pd.read_csv(P["paired"])) == 128,
    "multiplicity": len(pd.read_csv(P["multiplicity"])) == 128,
    "enrichment": len(pd.read_csv(P["enrichment"])) == 24,
    "sparse_audit": len(pd.read_csv(P["sparse_audit"])) == 8,
    "historical": len(pd.read_csv(P["historical"])) == len(historical),
    "concordance": len(pd.read_csv(P["concordance"])) == len(concordance),
}
if not all(readback.values()):
    raise RuntimeError(f"Table readback failed: {readback}")

qc_payload = {
    "cell_id": "6C-4H0",
    "package_version": "v1",
    "created_utc": CREATED_UTC,
    "analysis": "alternative_primary_and_secondary_evidence_drift_materialization",
    "source": {"path": str(SOURCE), "sha256": sha(SOURCE)},
    "prior_manifest": {"path": str(PRIOR_MANIFEST), "sha256": sha(PRIOR_MANIFEST)},
    "bootstrap": {
        "method": "paired nonparametric ordinary row bootstrap",
        "rng": "numpy.random.Generator",
        "bit_generator": type(rng.bit_generator).__name__,
        "seed": SEED,
        "attempts": N_BOOT,
        "batch_size": BOOTSTRAP_BATCH_SIZE,
        "identical_resamples_across_outcomes_and_scores": True,
        "validity_by_outcome": bootstrap_validity.to_dict("records"),
    },
    "multiplicity_policy": {
        "principal": (
            "Holm across 3 alternative-primary outcomes or 5 exploratory-drift outcomes, "
            "separately by principal comparator and metric"
        ),
        "secondary": (
            "Holm across 5 remaining comparators within each outcome and metric"
        ),
    },
    "checks": checks,
    "table_readback": readback,
    "passed_checks": sum(item["passed"] for item in checks),
    "failed_checks": sum(not item["passed"] for item in checks),
    "decision": (
        "PASS_STAGE6C_ALTERNATIVE_OUTCOME_SECONDARY_DRIFT_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
}
write_json(P["qc"], qc_payload)

artifact_keys = [
    "accounting",
    "star_delta",
    "star_crosstab",
    "points",
    "model_replicates",
    "enrichment_replicates",
    "validity",
    "intervals",
    "paired",
    "multiplicity",
    "enrichment",
    "sparse_audit",
    "historical",
    "concordance",
    "qc",
]
for key in artifact_keys:
    sidecar(P[key])
    if not sidecar_ok(P[key]):
        raise RuntimeError(f"Sidecar verification failed: {P[key]}")


def artifact_record(key: str) -> dict:
    path = P[key]
    record = {
        "artifact_key": key,
        "path": str(path),
        "relative_path": str(path.relative_to(ROOT)),
        "sha256": sha(path),
        "bytes": path.stat().st_size,
        "sidecar_path": str(path.with_name(path.name + ".sha256")),
        "sidecar_verified": sidecar_ok(path),
    }
    if path.suffix == ".csv":
        loaded = pd.read_csv(path)
        record.update(rows=len(loaded), columns=loaded.shape[1])
    elif path.suffix == ".parquet":
        loaded_metadata = pq.ParquetFile(path).metadata
        record.update(rows=loaded_metadata.num_rows, columns=loaded_metadata.num_columns)
    else:
        json.loads(path.read_text(encoding="utf-8"))
        record["json_readback"] = True
    return record


strict_interval = interval_lookup.loc[("strict_material_instability", "Full GES")]
manifest = {
    "cell_id": "6C-4H0",
    "package_version": "v1",
    "notebook_name": NOTEBOOK_NAME,
    "created_utc": CREATED_UTC,
    "authorized_category": "alternative_primary_and_secondary_evidence_drift",
    "immutable_sources": {
        "stage6b_primary_evaluable": {
            "path": str(SOURCE),
            "sha256": sha(SOURCE),
            "expected_sha256": SOURCE_SHA,
        },
        "prior_6c4g0_manifest": {
            "path": str(PRIOR_MANIFEST),
            "sha256": sha(PRIOR_MANIFEST),
            "expected_sha256": PRIOR_MANIFEST_SHA,
        },
    },
    "analysis_lock": {
        "outcome_definitions": OUTCOME_SPEC,
        "scores": MODELS,
        "bootstrap_seed": SEED,
        "bootstrap_attempts": N_BOOT,
        "bootstrap_batch_size": BOOTSTRAP_BATCH_SIZE,
        "risk_fractions": RISK_FRACTIONS,
        "principal_comparators": PRINCIPAL,
        "secondary_comparators": SECONDARY,
        "multiplicity_policy": qc_payload["multiplicity_policy"],
    },
    "outcome_accounting": accounting.to_dict("records"),
    "result_summary": {
        "strict_material_full_ges_auprc": float(strict_interval["point_auprc"]),
        "strict_material_full_ges_auroc": float(strict_interval["point_auroc"]),
        "conflict_transition_full_ges_auprc": float(
            interval_lookup.loc[("conflict_transition_instability", "Full GES"), "point_auprc"]
        ),
        "conflict_transition_full_ges_auroc": float(
            interval_lookup.loc[("conflict_transition_instability", "Full GES"), "point_auroc"]
        ),
        "historical_points_reproduced": bool(
            concordance["point_reproduced_at_recorded_precision"].all()
        ),
        "historical_conclusions_concordant": bool(
            concordance["scientific_conclusion_concordant"].all()
        ),
        "scientific_conclusion_audit": scientific_conclusions,
    },
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "artifacts": [artifact_record(key) for key in artifact_keys],
    "scientific_boundary": {
        "frozen_inputs_modified": False,
        "scores_refit_or_recalibrated": False,
        "thresholds_or_weights_changed": False,
        "outcome_components_changed": False,
        "primary_outcome_changed": False,
        "cohort_membership_changed": False,
        "experiment_2_started": False,
        "interpretation": (
            "Strict material instability provides the clearest independent clinical sensitivity, "
            "while strong review-star/status drift discrimination is exploratory and non-independent "
            "because review metadata contributes to the frozen GES pathway. New expert-panel "
            "involvement remains extremely sparse with eight events."
        ),
    },
    "decision": (
        "PASS_STAGE6C_ALTERNATIVE_OUTCOME_SECONDARY_DRIFT_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "next_authorized_category": "nested_scv_37_record_exploratory_analysis_materialization",
}
manifest_hash = write_json(P["manifest"], manifest)
sidecar(P["manifest"])

# Fresh package and immutable-source reverification.
manifest_readback = json.loads(P["manifest"].read_text(encoding="utf-8"))
if not sidecar_ok(P["manifest"]):
    raise RuntimeError("Manifest sidecar verification failed.")
if manifest_readback["decision"] != manifest["decision"]:
    raise RuntimeError("Manifest decision readback mismatch.")
if manifest_readback["scientific_boundary"]["experiment_2_started"] is not False:
    raise RuntimeError("Experiment 2 boundary failed.")
for artifact in manifest_readback["artifacts"]:
    path = Path(artifact["path"])
    if sha(path) != artifact["sha256"] or not sidecar_ok(path):
        raise RuntimeError(f"Final artifact verification failed: {path}")
if sha(SOURCE) != SOURCE_SHA or sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA:
    raise RuntimeError("An immutable source changed during Cell 6C-4H0.")


# --------------------------------------------------------------------------------------------------
# 10. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 170)
print(
    "STAGE 6C STEP 4H — CELL 6C-4H0 — ALTERNATIVE-PRIMARY AND SECONDARY "
    "EVIDENCE-DRIFT RESULT CATEGORY"
)
print("=" * 170)
print(f"Stage 6B source hash                    : PASS ({sha(SOURCE)})")
print(f"Prior Cell 6C-4G0 manifest              : PASS ({sha(PRIOR_MANIFEST)})")
print(
    f"Frozen evaluable cohort                 : {EXPECTED_ROWS:,} rows | "
    f"{EXPECTED_PRIMARY_EVENTS:,} primary events | {EXPECTED_PRIMARY_NEGATIVES:,} negatives"
)
print("Alternative / secondary outcomes       : 3 alternative primary + 5 exploratory drift")
print(f"Nine-score point estimates              : PASS ({len(points)}/72)")
print(f"Bootstrap attempts                      : {N_BOOT:,}")
print(
    "Bootstrap validity                    : first seven outcomes 2,000/2,000; "
    "new expert panel 1,999/2,000"
)
print(f"Bootstrap elapsed                       : {bootstrap_elapsed / 60:.2f} minutes")
print("Principal Holm families                 : 3-outcome or 5-outcome families by comparator/metric")
print("Remaining-comparator Holm families      : 5 comparators within each outcome/metric")
print(
    f"Historical recorded points              : PASS "
    f"({int(concordance.point_reproduced_at_recorded_precision.sum())}/{len(concordance)})"
)
print(
    f"Historical scientific conclusions       : PASS "
    f"({int(concordance.scientific_conclusion_concordant.sum())}/{len(concordance)})"
)
print(f"Fresh QC                                : PASS ({qc_payload['passed_checks']}/{len(checks)})")
for label, key in [
    ("Outcome accounting", "accounting"),
    ("Review-star delta inventory", "star_delta"),
    ("T0 × T1 review-star cross-tab", "star_crosstab"),
    ("Point estimates", "points"),
    ("Model bootstrap replicates", "model_replicates"),
    ("Enrichment bootstrap replicates", "enrichment_replicates"),
    ("Bootstrap validity", "validity"),
    ("Model intervals", "intervals"),
    ("Paired inference", "paired"),
    ("Multiplicity table", "multiplicity"),
    ("Enrichment intervals", "enrichment"),
    ("Sparse-outcome audit", "sparse_audit"),
    ("Historical results", "historical"),
    ("Concordance table", "concordance"),
    ("QC", "qc"),
    ("Manifest", "manifest"),
]:
    print(f"{label:40s}: {P[key]}")
print(f"Manifest SHA-256                        : {manifest_hash}")

print("\nALTERNATIVE / SECONDARY OUTCOME ACCOUNTING")
print(
    accounting[
        [
            "outcome",
            "outcome_role",
            "events",
            "negatives",
            "prevalence",
            "sparse_event_flag",
        ]
    ].to_string(index=False)
)

print("\nFULL-GES OUTCOME-SPECIFIC MODEL INTERVALS")
print(
    intervals.loc[
        intervals["model"] == "Full GES",
        [
            "outcome",
            "outcome_role",
            "events",
            "point_auprc",
            "auprc_ci_lower",
            "auprc_ci_upper",
            "auprc_null_status",
            "point_auroc",
            "auroc_ci_lower",
            "auroc_ci_upper",
            "auroc_null_status",
            "valid_bootstrap_replicates",
            "invalid_one_class_replicates",
            "sparse_event_flag",
        ],
    ].to_string(index=False)
)

print("\nFULL-GES PRINCIPAL-COMPARATOR PAIRED INFERENCE")
print(
    paired.loc[
        paired["comparison_family"] == "principal_prespecified",
        [
            "outcome",
            "outcome_role",
            "metric",
            "comparator",
            "point_difference",
            "difference_ci_lower",
            "difference_ci_upper",
            "paired_interval_status",
            "bootstrap_sign_p_value",
            "principal_outcome_family_holm_adjusted_p",
            "principal_outcome_family_holm_supported_at_0_05",
            "valid_bootstrap_replicates",
            "invalid_one_class_replicates",
        ],
    ].to_string(index=False)
)

print("\nSTRICT-MATERIAL FULL-GES 5% / 10% / 20% ENRICHMENT")
print(
    enrichment.loc[
        enrichment["outcome_key"] == "strict_material_instability",
        [
            "risk_fraction",
            "selected_rows",
            "selected_events",
            "selected_event_rate",
            "point_risk_ratio_vs_remaining",
            "risk_ratio_ci_lower",
            "risk_ratio_ci_upper",
            "point_enrichment_over_prevalence",
            "enrichment_ci_lower",
            "enrichment_ci_upper",
            "enrichment_interval_status",
        ],
    ].to_string(index=False)
)

print("\nINTERPRETATION BOUNDARY")
print(
    "Review-star, review-status, and expert-panel outcomes remain exploratory evidence-drift "
    "analyses because review metadata contributes to the original frozen GES pathway. New "
    "expert-panel involvement has only eight events and is not confirmatory. No primary outcome, "
    "score, threshold, weight, cohort membership, linkage decision, or frozen source was changed."
)

print("\nCELL DECISION")
print(
    "PASS_STAGE6C_ALTERNATIVE_OUTCOME_SECONDARY_DRIFT_RESULT_CATEGORY_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
)
print(
    "The sixth of eight Stage 6C result categories is independently materialized. "
    "The next authorized category is the 37-record nested-SCV exploratory analysis "
    "materialization. Experiment 2 has not started."
)
print("=" * 170)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4H0_Alternative_Outcome_Secondary_Drift_Materialization.ipynb

Preparing eight frozen alternative/secondary outcomes across 66,636 evaluable rows and nine scores
Exact grouped metric validation against scikit-learn: PASS (72/72)
  Completed 250/2,000 replicates | new-expert-panel valid 250
  Completed 500/2,000 replicates | new-expert-panel valid 500
  Completed 750/2,000 replicates | new-expert-panel valid 750
  Completed 1,000/2,000 replicates | new-expert-panel valid 1,000
  Completed 1,250/2,000 replicates | new-expert-panel valid 1,250
  Completed 1,500/2,000 replicates | new-expert-panel valid 1,500
  Completed 1,750/2,000 replicates | new-expert-panel valid 1,749
  Completed 2,000/2,000 replicates | new-expert-panel valid 1,999

STAGE 6C STEP 4H — CELL 6C-4H0 — ALTERNATIVE-PRIMARY AND SECONDARY EVI

In [6]:
# ==================================================================================================
# STAGE 6C STEP 4I — CELL 6C-4I0
# 37-RECORD NESTED-SCV EXPLORATORY RESULT-CATEGORY MATERIALIZATION
# ==================================================================================================
#
# Purpose
# -------
# 1. Freshly verify the immutable Stage 6B evaluable cohort, the frozen score-blind nested-SCV
#    record-level package, its manifest, and the completed Cell 6C-4H0 manifest.
# 2. Confirm that the high-rigor contradictory-submission endpoint remains non-estimable.
# 3. Isolate only the frozen 37 complete-case submitter-distribution records:
#       21 major-shift events and 16 negatives; 66,599 records censored.
# 4. Join the nine already-frozen Stage 6B scores only after the nested outcome package passes.
# 5. Reproduce the exact Cell 6C-3G1 point estimates and its 2,000 paired ordinary row-bootstrap
#    attempts using NumPy Generator(PCG64), seed 42, and rng.integers().
# 6. Materialize versioned score, replicate, interval, paired-comparison, historical-concordance,
#    QC, manifest, and SHA-256 sidecar artifacts.
#
# Scientific boundary
# -------------------
# - This is a highly exploratory 37-record complete-case sensitivity analysis.
# - The high-rigor endpoint has zero positive events and is not analyzed.
# - No score, outcome, model, threshold, weight, policy, linkage decision, row order, or cohort
#   membership is changed.
# - Experiment 2 is not started.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from collections import OrderedDict
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import re
import sys
import time

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import sklearn
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. LOCKED INPUTS, EXPECTATIONS, AND OUTPUT LOCATIONS
# --------------------------------------------------------------------------------------------------

NOTEBOOK_NAME = (
    "GES_Stage6C_Cell_6C_4I0_Nested_SCV_37_Record_"
    "Exploratory_Materialization.ipynb"
)

ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")
STAGE6_DIR = ROOT / "data_processed/stage6_temporal_validation"

EVAL = STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
EVAL_SHA256 = "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"

NESTED_DIR = STAGE6_DIR / "nested_scv_secondary_outcomes"
NESTED_PACKAGE = NESTED_DIR / "stage6c_nested_scv_secondary_outcomes_record_level_v1.parquet"
NESTED_PACKAGE_SHA256 = "18b3d75b62e8d1b891dcd88eb951b4aa767aa6c88570369004bd08b9c3de7fc2"

NESTED_MANIFEST = NESTED_DIR / "stage6c_nested_scv_secondary_outcomes_manifest_v1.json"
NESTED_MANIFEST_SHA256 = "b89626648af773b79053515f81dc1b7afee1ca3e516510a2bf52aa5110c5edc5"

PRIOR_MANIFEST = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4h0_alternative_outcome_secondary_drift_materialization_v1/"
    "stage6c_4h0_alternative_outcome_secondary_drift_manifest_v1.json"
)
PRIOR_MANIFEST_SHA256 = "207eeb09c9f5d6d0f1e46058252b27c4f4906d26f918a08bb29780b7de00d2db"

EXPECTED_POLICY_SHA256 = "0d11d2eec8a3dcc0116a0ab45862a73b46ce130b79df1d9583910232bedb93aa"

EXPECTED = {
    "total_rows": 66_636,
    "eval_columns": 79,
    "nested_columns": 28,
    "complete_rows": 37,
    "events": 21,
    "negatives": 16,
    "censored": 66_599,
    "prevalence": 21 / 37,
    "censoring_fraction": 66_599 / 66_636,
    "high_rigor_events": 0,
    "high_rigor_negatives": 54_228,
    "high_rigor_censored": 12_408,
}

SEED = 42
N_BOOT = 2_000
KEYS = ["t0_row_order", "rcv_accession"]

SCORES = OrderedDict([
    ("full_ges", ("full_ges_instability_risk_t0", "Full GES")),
    ("no_star_ges", ("no_star_ges_instability_risk_t0", "No-star GES")),
    ("review_stars", ("review_stars_instability_risk", "Review stars")),
    ("combined_metadata", ("combined_metadata_instability_risk", "Combined metadata")),
    ("conflict", ("conflict_instability_risk", "Conflict")),
    ("recency", ("recency_instability_risk", "Recency")),
    ("submitter", ("submitter_instability_risk", "Submitter support")),
    ("entropy", ("entropy_instability_risk", "Classification entropy")),
    ("additive", ("additive_instability_risk", "Additive risk")),
])
PRINCIPAL_COMPARATORS = ["no_star_ges", "review_stars", "combined_metadata"]

TABLE_DIR = ROOT / (
    "outputs/tables/stage6_temporal_validation/"
    "stage6c_4i0_nested_scv_37_record_exploratory_materialization_v1"
)
QC_DIR = ROOT / (
    "outputs/quality_checks/stage6_temporal_validation/"
    "stage6c_4i0_nested_scv_37_record_exploratory_materialization_v1"
)
MANIFEST_DIR = ROOT / (
    "configs/stage6_temporal_validation/"
    "stage6c_4i0_nested_scv_37_record_exploratory_materialization_v1"
)

for directory in (TABLE_DIR, QC_DIR, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

P = {
    "endpoint_accounting": TABLE_DIR / "stage6c_nested_scv_endpoint_accounting_v1.csv",
    "analysis_cohort": TABLE_DIR / "stage6c_nested_scv_37_record_analysis_cohort_v1.parquet",
    "point_estimates": TABLE_DIR / "stage6c_nested_scv_37_record_point_estimates_v1.csv",
    "bootstrap_replicates": TABLE_DIR / "stage6c_nested_scv_37_record_bootstrap_replicates_v1.parquet",
    "model_intervals": TABLE_DIR / "stage6c_nested_scv_37_record_model_bootstrap_intervals_v1.csv",
    "paired_inference": TABLE_DIR / "stage6c_nested_scv_37_record_paired_exploratory_inference_v1.csv",
    "historical_results": TABLE_DIR / "stage6c_nested_scv_37_record_historical_results_v1.csv",
    "concordance": TABLE_DIR / "stage6c_nested_scv_37_record_historical_vs_reproduced_concordance_v1.csv",
    "limitations": TABLE_DIR / "stage6c_nested_scv_37_record_limitations_v1.csv",
    "qc": QC_DIR / "stage6c_4i0_nested_scv_37_record_exploratory_qc_v1.json",
    "manifest": MANIFEST_DIR / "stage6c_4i0_nested_scv_37_record_exploratory_manifest_v1.json",
}

if P["manifest"].exists():
    CREATED_UTC = json.loads(P["manifest"].read_text(encoding="utf-8"))["created_utc"]
elif P["qc"].exists():
    CREATED_UTC = json.loads(P["qc"].read_text(encoding="utf-8"))["created_utc"]
else:
    CREATED_UTC = datetime.now(timezone.utc).isoformat()


# --------------------------------------------------------------------------------------------------
# 2. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def native(value):
    if isinstance(value, dict):
        return {str(k): native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [native(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return native(value.tolist())
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is pd.NA:
        return None
    return value


def stable_write_bytes(path: Path, payload: bytes) -> str:
    """Write atomically; on rerun, accept only byte-identical content."""
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    temporary.write_bytes(payload)
    new_hash = sha(temporary)

    if path.exists():
        if sha(path) != new_hash:
            temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Refusing to overwrite nonidentical artifact: {path}")
        temporary.unlink(missing_ok=True)
    else:
        os.replace(temporary, path)

    return sha(path)


def write_csv(path: Path, frame: pd.DataFrame) -> str:
    payload = frame.to_csv(
        index=False,
        lineterminator="\n",
        float_format="%.12g",
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_json(path: Path, obj) -> str:
    payload = (
        json.dumps(
            native(obj),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")
    return stable_write_bytes(path, payload)


def write_parquet(path: Path, frame: pd.DataFrame) -> str:
    path = Path(path)
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}-{time.time_ns()}")
    frame.to_parquet(temporary, index=False, compression="zstd", engine="pyarrow")

    if path.exists():
        existing = pd.read_parquet(path)
        fresh = pd.read_parquet(temporary)
        pd.testing.assert_frame_equal(
            existing,
            fresh,
            check_dtype=True,
            check_exact=True,
            check_like=False,
        )
        temporary.unlink()
    else:
        os.replace(temporary, path)

    return sha(path)


def sidecar(path: Path) -> Path:
    path = Path(path)
    output = path.with_name(path.name + ".sha256")
    stable_write_bytes(output, f"{sha(path)}  {path.name}\n".encode("utf-8"))
    return output


def sidecar_hash(path: Path) -> str:
    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        Path(path).read_text(encoding="utf-8"),
    )
    if not matches:
        raise RuntimeError(f"No SHA-256 found in sidecar: {path}")
    return matches[0].lower()


def sidecar_ok(path: Path) -> bool:
    path = Path(path)
    output = path.with_name(path.name + ".sha256")
    return output.exists() and sidecar_hash(output) == sha(path)


def normalize_rcv(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.extract(r"(RCV\d+)", expand=False)
    )


def ci(values) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lower, upper = np.percentile(values, [2.5, 97.5])
    return float(lower), float(upper)


def sign_p(values) -> float:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan
    lower_tail = (np.count_nonzero(values <= 0.0) + 1) / (len(values) + 1)
    upper_tail = (np.count_nonzero(values >= 0.0) + 1) / (len(values) + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def interval_status(lower: float, upper: float) -> str:
    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"
    if lower > 0:
        return "full_ges_supported_higher"
    if upper < 0:
        return "full_ges_supported_lower"
    return "interval_includes_null"


# --------------------------------------------------------------------------------------------------
# 3. CRYPTOGRAPHIC AND STRUCTURAL PREFLIGHT
# --------------------------------------------------------------------------------------------------

required_files = [
    EVAL,
    EVAL.with_name(EVAL.name + ".sha256"),
    NESTED_PACKAGE,
    NESTED_PACKAGE.with_name(NESTED_PACKAGE.name + ".sha256"),
    NESTED_MANIFEST,
    NESTED_MANIFEST.with_name(NESTED_MANIFEST.name + ".sha256"),
    PRIOR_MANIFEST,
    PRIOR_MANIFEST.with_name(PRIOR_MANIFEST.name + ".sha256"),
]
for path in required_files:
    if not path.exists():
        raise FileNotFoundError(path)

if sha(EVAL) != EVAL_SHA256 or not sidecar_ok(EVAL):
    raise RuntimeError("Stage 6B evaluable package verification failed.")
if sha(NESTED_PACKAGE) != NESTED_PACKAGE_SHA256 or not sidecar_ok(NESTED_PACKAGE):
    raise RuntimeError("Nested-SCV record-level package verification failed.")
if sha(NESTED_MANIFEST) != NESTED_MANIFEST_SHA256 or not sidecar_ok(NESTED_MANIFEST):
    raise RuntimeError("Nested-SCV manifest verification failed.")
if sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA256 or not sidecar_ok(PRIOR_MANIFEST):
    raise RuntimeError("Prior Cell 6C-4H0 manifest verification failed.")

eval_metadata = pq.ParquetFile(EVAL).metadata
nested_metadata = pq.ParquetFile(NESTED_PACKAGE).metadata

if (eval_metadata.num_rows, eval_metadata.num_columns) != (
    EXPECTED["total_rows"],
    EXPECTED["eval_columns"],
):
    raise RuntimeError(
        f"Unexpected Stage 6B dimensions: "
        f"{(eval_metadata.num_rows, eval_metadata.num_columns)}"
    )

if (nested_metadata.num_rows, nested_metadata.num_columns) != (
    EXPECTED["total_rows"],
    EXPECTED["nested_columns"],
):
    raise RuntimeError(
        f"Unexpected nested-SCV dimensions: "
        f"{(nested_metadata.num_rows, nested_metadata.num_columns)}"
    )

nested_manifest_readback = json.loads(NESTED_MANIFEST.read_text(encoding="utf-8"))
prior_manifest_readback = json.loads(PRIOR_MANIFEST.read_text(encoding="utf-8"))

if prior_manifest_readback.get("decision") != (
    "PASS_STAGE6C_ALTERNATIVE_OUTCOME_SECONDARY_DRIFT_RESULT_CATEGORY_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
):
    raise RuntimeError("Prior Cell 6C-4H0 decision mismatch.")


# --------------------------------------------------------------------------------------------------
# 4. VERIFY THE FROZEN NESTED OUTCOMES BEFORE LOADING ANY SCORE
# --------------------------------------------------------------------------------------------------

nested_columns = [
    "t0_row_order",
    "rcv_accession",
    "new_contradictory_high_rigor_status",
    "new_contradictory_high_rigor_submission",
    "distribution_status",
    "submitter_classification_tvd",
    "major_submitter_distribution_shift",
    "policy_version",
    "policy_sha256",
]

nested = pd.read_parquet(NESTED_PACKAGE, columns=nested_columns).copy()
nested["rcv_accession"] = normalize_rcv(nested["rcv_accession"])
nested["t0_row_order"] = pd.to_numeric(
    nested["t0_row_order"],
    errors="raise",
).astype("int64")

if len(nested) != EXPECTED["total_rows"]:
    raise RuntimeError("Nested-SCV row count mismatch.")
if nested["rcv_accession"].isna().any():
    raise RuntimeError("Nested-SCV RCV normalization failure.")
if nested["rcv_accession"].nunique() != EXPECTED["total_rows"]:
    raise RuntimeError("Nested-SCV RCV keys are not unique.")
if nested["t0_row_order"].nunique() != EXPECTED["total_rows"]:
    raise RuntimeError("Nested-SCV row-order keys are not unique.")
if not nested["policy_version"].astype("string").eq("1.1").all():
    raise RuntimeError("Nested-SCV policy-version mismatch.")
if not nested["policy_sha256"].astype("string").eq(EXPECTED_POLICY_SHA256).all():
    raise RuntimeError("Nested-SCV policy-hash mismatch.")

high_rigor = nested["new_contradictory_high_rigor_submission"]
hr_events = int(high_rigor.fillna(0).astype("int8").sum())
hr_evaluable = int(high_rigor.notna().sum())
hr_negatives = hr_evaluable - hr_events
hr_censored = len(nested) - hr_evaluable

if (
    hr_events,
    hr_negatives,
    hr_censored,
) != (
    EXPECTED["high_rigor_events"],
    EXPECTED["high_rigor_negatives"],
    EXPECTED["high_rigor_censored"],
):
    raise RuntimeError(
        "High-rigor contradiction endpoint accounting mismatch: "
        f"{(hr_events, hr_negatives, hr_censored)}"
    )

complete = nested.loc[nested["distribution_status"].eq("EVALUABLE")].copy()
distribution_censored = int(nested["distribution_status"].ne("EVALUABLE").sum())

if len(complete) != EXPECTED["complete_rows"]:
    raise RuntimeError("Complete-case nested-SCV row count mismatch.")
if distribution_censored != EXPECTED["censored"]:
    raise RuntimeError("Nested-SCV distribution censoring count mismatch.")
if complete["major_submitter_distribution_shift"].isna().any():
    raise RuntimeError("Complete-case major-shift outcome contains missing values.")

y_prejoin = complete["major_submitter_distribution_shift"].astype("int8").to_numpy()
if (
    int(y_prejoin.sum()),
    int(len(y_prejoin) - y_prejoin.sum()),
) != (
    EXPECTED["events"],
    EXPECTED["negatives"],
):
    raise RuntimeError("Nested-SCV event accounting mismatch.")

tvd_values = set(
    np.round(
        pd.to_numeric(
            complete["submitter_classification_tvd"],
            errors="raise",
        ).to_numpy(float),
        12,
    ).tolist()
)
if tvd_values != {0.0, 1.0}:
    raise RuntimeError(f"Unexpected complete-case TVD support: {tvd_values}")


# --------------------------------------------------------------------------------------------------
# 5. LOAD FROZEN SCORES ONLY AFTER THE SCORE-BLIND NESTED PACKAGE PASSES
# --------------------------------------------------------------------------------------------------

score_columns = [column for column, _ in SCORES.values()]
required_score_columns = KEYS + score_columns
eval_schema = pq.ParquetFile(EVAL).schema_arrow.names
missing_score_columns = [
    column for column in required_score_columns if column not in eval_schema
]
if missing_score_columns:
    raise RuntimeError(f"Missing Stage 6B score columns: {missing_score_columns}")

scores = pd.read_parquet(EVAL, columns=required_score_columns).copy()
scores["rcv_accession"] = normalize_rcv(scores["rcv_accession"])
scores["t0_row_order"] = pd.to_numeric(
    scores["t0_row_order"],
    errors="raise",
).astype("int64")

if len(scores) != EXPECTED["total_rows"]:
    raise RuntimeError("Stage 6B score row count mismatch.")
if scores["rcv_accession"].nunique() != EXPECTED["total_rows"]:
    raise RuntimeError("Stage 6B score RCV keys are not unique.")
if scores["t0_row_order"].nunique() != EXPECTED["total_rows"]:
    raise RuntimeError("Stage 6B score row-order keys are not unique.")

for column in score_columns:
    scores[column] = pd.to_numeric(scores[column], errors="raise").astype("float64")
    values = scores[column].to_numpy(float)
    if not np.isfinite(values).all() or values.min() < 0.0 or values.max() > 1.0:
        raise RuntimeError(f"Invalid Stage 6B score field: {column}")

analysis = complete.merge(
    scores,
    on=KEYS,
    how="left",
    validate="one_to_one",
    indicator=True,
)
if not analysis["_merge"].eq("both").all():
    raise RuntimeError("A complete-case nested-SCV row failed score linkage.")

analysis = (
    analysis.drop(columns="_merge")
    .sort_values("t0_row_order", kind="mergesort")
    .reset_index(drop=True)
)

if len(analysis) != EXPECTED["complete_rows"]:
    raise RuntimeError("Final 37-record analysis cohort size mismatch.")

y = analysis["major_submitter_distribution_shift"].astype("int8").to_numpy()
prevalence = float(y.mean())

if abs(prevalence - EXPECTED["prevalence"]) > 1e-15:
    raise RuntimeError("37-record endpoint prevalence mismatch.")


# --------------------------------------------------------------------------------------------------
# 6. ENDPOINT ACCOUNTING AND LOCKED POINT ESTIMATES
# --------------------------------------------------------------------------------------------------

endpoint_accounting = pd.DataFrame([
    {
        "endpoint_key": "new_contradictory_high_rigor_submission",
        "endpoint": "New contradictory high-rigor submission",
        "status": "NOT_ESTIMABLE_NO_POSITIVE_EVENTS",
        "evaluable_rows": hr_events + hr_negatives,
        "events": hr_events,
        "negatives": hr_negatives,
        "censored_rows": hr_censored,
        "censoring_fraction": hr_censored / EXPECTED["total_rows"],
        "analyzed_for_discrimination": False,
        "interpretation": (
            "The frozen nested source provides zero observable positive high-rigor events; "
            "AUPRC and AUROC are not estimable and the outcome is not redefined."
        ),
    },
    {
        "endpoint_key": "major_submitter_distribution_shift",
        "endpoint": "Major submitter-classification distribution shift",
        "status": "TECHNICALLY_ESTIMABLE_HIGHLY_EXPLORATORY",
        "evaluable_rows": EXPECTED["complete_rows"],
        "events": EXPECTED["events"],
        "negatives": EXPECTED["negatives"],
        "censored_rows": EXPECTED["censored"],
        "censoring_fraction": EXPECTED["censoring_fraction"],
        "analyzed_for_discrimination": True,
        "interpretation": (
            "Only 37 complete-case records are evaluable; all inference is descriptive and "
            "highly exploratory."
        ),
    },
])

point_rows = []
for model_key, (score_column, model_name) in SCORES.items():
    score = analysis[score_column].to_numpy(float)
    auprc = float(average_precision_score(y, score))
    auroc = float(roc_auc_score(y, score))
    point_rows.append({
        "endpoint_key": "major_submitter_distribution_shift",
        "model_key": model_key,
        "model": model_name,
        "score_column": score_column,
        "rows": EXPECTED["complete_rows"],
        "events": EXPECTED["events"],
        "negatives": EXPECTED["negatives"],
        "prevalence": prevalence,
        "unique_score_values": int(np.unique(score).size),
        "point_auprc": auprc,
        "auprc_minus_prevalence": auprc - prevalence,
        "auprc_lift_over_prevalence": auprc / prevalence,
        "point_auroc": auroc,
        "auroc_minus_0_50": auroc - 0.50,
        "analysis_status": "HIGHLY_EXPLORATORY_37_COMPLETE_CASE_RECORDS",
    })

point_estimates = pd.DataFrame(point_rows)
point_lookup = point_estimates.set_index("model_key")


# --------------------------------------------------------------------------------------------------
# 7. EXACT CELL 6C-3G1 2,000-ATTEMPT PAIRED ORDINARY ROW BOOTSTRAP
# --------------------------------------------------------------------------------------------------

print(f"Use this Colab notebook file name: {NOTEBOOK_NAME}")
print(
    f"\nPreparing the frozen 37-record nested-SCV complete-case endpoint: "
    f"{EXPECTED['events']} events and {EXPECTED['negatives']} negatives"
)

rng = np.random.default_rng(SEED)
replicate_rows = []
valid = 0
invalid = 0
analysis_start = time.time()

score_arrays = {
    model_key: analysis[score_column].to_numpy(float)
    for model_key, (score_column, _) in SCORES.items()
}

for attempt in range(1, N_BOOT + 1):
    # Exact historical Cell 6C-3G1 bootstrap implementation.
    indices = rng.integers(0, len(y), size=len(y))
    y_boot = y[indices]
    sampled_events = int(y_boot.sum())
    sampled_negatives = int(len(y_boot) - sampled_events)
    valid_two_class = bool(np.unique(y_boot).size == 2)

    row = {
        "replicate": attempt,
        "sampled_rows": len(y_boot),
        "sampled_events": sampled_events,
        "sampled_negatives": sampled_negatives,
        "valid_two_class_replicate": valid_two_class,
        "seed": SEED,
        "rng": "numpy.random.Generator",
        "bit_generator": type(rng.bit_generator).__name__,
    }

    if valid_two_class:
        valid += 1
        for model_key in SCORES:
            score_boot = score_arrays[model_key][indices]
            row[f"{model_key}__auprc"] = float(
                average_precision_score(y_boot, score_boot)
            )
            row[f"{model_key}__auroc"] = float(
                roc_auc_score(y_boot, score_boot)
            )
    else:
        invalid += 1
        for model_key in SCORES:
            row[f"{model_key}__auprc"] = np.nan
            row[f"{model_key}__auroc"] = np.nan

    replicate_rows.append(row)

    if attempt % 250 == 0:
        print(
            f"  Completed {attempt:,}/{N_BOOT:,} replicates | "
            f"valid {valid:,}"
        )

bootstrap_elapsed = time.time() - analysis_start
bootstrap_replicates = pd.DataFrame(replicate_rows)

if (valid, invalid) != (N_BOOT, 0):
    raise RuntimeError(
        f"Historical Cell 6C-3G1 validity mismatch: valid={valid}, invalid={invalid}"
    )


# --------------------------------------------------------------------------------------------------
# 8. MODEL INTERVALS AND PAIRED EXPLORATORY INFERENCE
# --------------------------------------------------------------------------------------------------

interval_rows = []
for model_key, (_, model_name) in SCORES.items():
    auprc_values = bootstrap_replicates[f"{model_key}__auprc"].to_numpy(float)
    auroc_values = bootstrap_replicates[f"{model_key}__auroc"].to_numpy(float)
    auprc_low, auprc_high = ci(auprc_values)
    auroc_low, auroc_high = ci(auroc_values)
    point = point_lookup.loc[model_key]

    interval_rows.append({
        "endpoint_key": "major_submitter_distribution_shift",
        "model_key": model_key,
        "model": model_name,
        "rows": EXPECTED["complete_rows"],
        "events": EXPECTED["events"],
        "negatives": EXPECTED["negatives"],
        "prevalence": prevalence,
        "unique_score_values": int(point["unique_score_values"]),
        "point_auprc": float(point["point_auprc"]),
        "auprc_ci_lower": auprc_low,
        "auprc_ci_upper": auprc_high,
        "point_auprc_minus_prevalence": float(point["auprc_minus_prevalence"]),
        "point_auroc": float(point["point_auroc"]),
        "auroc_ci_lower": auroc_low,
        "auroc_ci_upper": auroc_high,
        "point_auroc_minus_0_50": float(point["auroc_minus_0_50"]),
        "attempted_bootstrap_replicates": N_BOOT,
        "valid_bootstrap_replicates": valid,
        "invalid_one_class_replicates": invalid,
        "analysis_status": "HIGHLY_EXPLORATORY_37_COMPLETE_CASE_RECORDS",
    })

model_intervals = pd.DataFrame(interval_rows)

paired_rows = []
for comparator_key in PRINCIPAL_COMPARATORS:
    comparator_name = SCORES[comparator_key][1]
    for metric_name, metric_label, point_column in [
        ("auprc", "AUPRC", "point_auprc"),
        ("auroc", "AUROC", "point_auroc"),
    ]:
        differences = (
            bootstrap_replicates[f"full_ges__{metric_name}"].to_numpy(float)
            - bootstrap_replicates[f"{comparator_key}__{metric_name}"].to_numpy(float)
        )
        lower, upper = ci(differences)
        point_difference = float(
            point_lookup.loc["full_ges", point_column]
            - point_lookup.loc[comparator_key, point_column]
        )

        paired_rows.append({
            "endpoint_key": "major_submitter_distribution_shift",
            "metric": metric_label,
            "comparison": f"Full GES minus {comparator_name}",
            "comparator_key": comparator_key,
            "comparator": comparator_name,
            "point_difference": point_difference,
            "difference_ci_lower": lower,
            "difference_ci_upper": upper,
            "paired_interval_status": interval_status(lower, upper),
            "bootstrap_probability_full_greater": float(np.mean(differences > 0)),
            "bootstrap_sign_p_value": sign_p(differences),
            "attempted_bootstrap_replicates": N_BOOT,
            "valid_bootstrap_replicates": int(np.isfinite(differences).sum()),
            "invalid_one_class_replicates": int((~np.isfinite(differences)).sum()),
            "analysis_status": "HIGHLY_EXPLORATORY_NO_MULTIPLICITY_CLAIM",
        })

paired_inference = pd.DataFrame(paired_rows)


# --------------------------------------------------------------------------------------------------
# 9. HISTORICAL-RESULT PRESERVATION AND CONCORDANCE
# --------------------------------------------------------------------------------------------------

historical_metric_values = {
    "full_ges": {"AUPRC": 0.660749, "AUROC": 0.559524},
    "no_star_ges": {"AUPRC": 0.660749, "AUROC": 0.559524},
    "review_stars": {"AUPRC": 0.567568, "AUROC": 0.500000},
    "combined_metadata": {"AUPRC": 0.660749, "AUROC": 0.559524},
    "conflict": {"AUPRC": 0.567568, "AUROC": 0.500000},
    "recency": {"AUPRC": 0.660749, "AUROC": 0.559524},
    "submitter": {"AUPRC": 0.567568, "AUROC": 0.500000},
    "entropy": {"AUPRC": 0.567568, "AUROC": 0.500000},
    "additive": {"AUPRC": 0.567568, "AUROC": 0.500000},
}

historical_rows = [
    {
        "result_id": "endpoint__evaluable_rows",
        "result_type": "endpoint_accounting",
        "model_key": "",
        "metric": "evaluable_rows",
        "historical_value": 37.0,
        "historical_conclusion": "37_complete_case_records",
        "tolerance": 0.0,
    },
    {
        "result_id": "endpoint__events",
        "result_type": "endpoint_accounting",
        "model_key": "",
        "metric": "events",
        "historical_value": 21.0,
        "historical_conclusion": "21_major_shift_events",
        "tolerance": 0.0,
    },
    {
        "result_id": "endpoint__negatives",
        "result_type": "endpoint_accounting",
        "model_key": "",
        "metric": "negatives",
        "historical_value": 16.0,
        "historical_conclusion": "16_major_shift_negatives",
        "tolerance": 0.0,
    },
    {
        "result_id": "endpoint__censored_rows",
        "result_type": "endpoint_accounting",
        "model_key": "",
        "metric": "censored_rows",
        "historical_value": 66_599.0,
        "historical_conclusion": "66599_distribution_censored",
        "tolerance": 0.0,
    },
    {
        "result_id": "endpoint__high_rigor_events",
        "result_type": "endpoint_accounting",
        "model_key": "",
        "metric": "high_rigor_events",
        "historical_value": 0.0,
        "historical_conclusion": "high_rigor_endpoint_not_estimable",
        "tolerance": 0.0,
    },
    {
        "result_id": "bootstrap__valid_replicates",
        "result_type": "bootstrap_accounting",
        "model_key": "",
        "metric": "valid_replicates",
        "historical_value": 2_000.0,
        "historical_conclusion": "2000_valid_bootstrap_replicates",
        "tolerance": 0.0,
    },
]

for model_key, metric_values in historical_metric_values.items():
    for metric, value in metric_values.items():
        historical_rows.append({
            "result_id": f"{model_key}__{metric.lower()}",
            "result_type": "model_point",
            "model_key": model_key,
            "metric": metric,
            "historical_value": value,
            "historical_conclusion": "historical_rounded_point_reproduced",
            "tolerance": 5.1e-7,
        })

historical_results = pd.DataFrame(historical_rows)
historical_results["source"] = (
    "Technical report Version 7.0 Appendix P and original Cell 6C-3G1; "
    "historical rounded point values are preserved separately from exact reproduced values."
)

concordance_rows = []
for record in historical_results.to_dict("records"):
    result_type = record["result_type"]
    metric = record["metric"]

    if result_type == "endpoint_accounting":
        reproduced_map = {
            "evaluable_rows": float(EXPECTED["complete_rows"]),
            "events": float(EXPECTED["events"]),
            "negatives": float(EXPECTED["negatives"]),
            "censored_rows": float(EXPECTED["censored"]),
            "high_rigor_events": float(hr_events),
        }
        conclusion_map = {
            "evaluable_rows": "37_complete_case_records",
            "events": "21_major_shift_events",
            "negatives": "16_major_shift_negatives",
            "censored_rows": "66599_distribution_censored",
            "high_rigor_events": "high_rigor_endpoint_not_estimable",
        }
        reproduced_value = reproduced_map[metric]
        reproduced_conclusion = conclusion_map[metric]

    elif result_type == "bootstrap_accounting":
        reproduced_value = float(valid)
        reproduced_conclusion = "2000_valid_bootstrap_replicates"

    elif result_type == "model_point":
        model_key = record["model_key"]
        column = "point_auprc" if metric == "AUPRC" else "point_auroc"
        reproduced_value = float(point_lookup.loc[model_key, column])
        reproduced_conclusion = "historical_rounded_point_reproduced"

    else:
        raise RuntimeError(f"Unhandled historical result type: {result_type}")

    absolute_difference = abs(reproduced_value - float(record["historical_value"]))
    point_pass = absolute_difference <= float(record["tolerance"])
    conclusion_pass = reproduced_conclusion == record["historical_conclusion"]

    concordance_rows.append({
        **record,
        "reproduced_value": reproduced_value,
        "absolute_difference": absolute_difference,
        "value_reproduced_at_recorded_precision": point_pass,
        "reproduced_conclusion": reproduced_conclusion,
        "scientific_conclusion_concordant": conclusion_pass,
    })

concordance = pd.DataFrame(concordance_rows)

limitations = pd.DataFrame([
    {
        "limitation_key": "complete_case_size",
        "observed_value": "37 evaluable records",
        "interpretation": (
            "The endpoint is technically estimable but too small to establish generalizable "
            "superiority, calibration, or clinical utility."
        ),
    },
    {
        "limitation_key": "censoring",
        "observed_value": f"{EXPECTED['censored']:,}/{EXPECTED['total_rows']:,} censored",
        "interpretation": (
            "The 99.9445% censoring fraction is the dominant scientific limitation and must be "
            "reported with every result."
        ),
    },
    {
        "limitation_key": "high_rigor_endpoint",
        "observed_value": "0 positive events",
        "interpretation": (
            "The high-rigor contradictory-submission endpoint is not estimable and is not "
            "redefined using aggregate review metadata."
        ),
    },
    {
        "limitation_key": "tvd_support",
        "observed_value": "Observed TVD values are only 0 and 1",
        "interpretation": (
            "Continuous TVD analysis is not treated as a stable continuous endpoint in this tiny "
            "complete-case subset."
        ),
    },
    {
        "limitation_key": "multiplicity_and_inference",
        "observed_value": "Exploratory paired bootstrap only",
        "interpretation": (
            "Intervals describe resampling instability inside the 37-record subset and do not "
            "support confirmatory multiplicity-adjusted claims."
        ),
    },
])


# --------------------------------------------------------------------------------------------------
# 10. FRESH QC BEFORE ARTIFACT WRITES
# --------------------------------------------------------------------------------------------------

checks = []

def check(name: str, passed: bool, details):
    checks.append({
        "check_name": name,
        "passed": bool(passed),
        "details": native(details),
    })

dynamic_models = ["full_ges", "no_star_ges", "combined_metadata", "recency"]
constant_models = ["review_stars", "conflict", "submitter", "entropy", "additive"]

check("stage6b_hash", sha(EVAL) == EVAL_SHA256, sha(EVAL))
check("stage6b_sidecar", sidecar_ok(EVAL), str(EVAL) + ".sha256")
check("nested_package_hash", sha(NESTED_PACKAGE) == NESTED_PACKAGE_SHA256, sha(NESTED_PACKAGE))
check("nested_package_sidecar", sidecar_ok(NESTED_PACKAGE), str(NESTED_PACKAGE) + ".sha256")
check("nested_manifest_hash", sha(NESTED_MANIFEST) == NESTED_MANIFEST_SHA256, sha(NESTED_MANIFEST))
check("nested_manifest_sidecar", sidecar_ok(NESTED_MANIFEST), str(NESTED_MANIFEST) + ".sha256")
check("prior_4h0_manifest_hash", sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA256, sha(PRIOR_MANIFEST))
check("prior_4h0_manifest_sidecar", sidecar_ok(PRIOR_MANIFEST), str(PRIOR_MANIFEST) + ".sha256")
check("stage6b_dimensions", (eval_metadata.num_rows, eval_metadata.num_columns) == (66_636, 79), {})
check("nested_dimensions", (nested_metadata.num_rows, nested_metadata.num_columns) == (66_636, 28), {})
check("nested_policy", nested["policy_sha256"].astype("string").eq(EXPECTED_POLICY_SHA256).all(), {})
check("high_rigor_nonestimable", (hr_events, hr_negatives, hr_censored) == (0, 54_228, 12_408), {})
check("distribution_accounting", (len(analysis), int(y.sum()), int(len(y)-y.sum()), distribution_censored) == (37, 21, 16, 66_599), {})
check("distribution_censoring_fraction", abs(EXPECTED["censoring_fraction"] - 0.9994447445837085) < 1e-15, EXPECTED["censoring_fraction"])
check("tvd_support", tvd_values == {0.0, 1.0}, sorted(tvd_values))
check("score_join_complete", len(analysis) == 37 and analysis[score_columns].notna().all().all(), {})
check("nine_point_estimates", len(point_estimates) == 9, len(point_estimates))
check("bootstrap_attempts", len(bootstrap_replicates) == 2_000, len(bootstrap_replicates))
check("bootstrap_validity", (valid, invalid) == (2_000, 0), {"valid": valid, "invalid": invalid})
check("nine_model_intervals", len(model_intervals) == 9, len(model_intervals))
check("six_paired_results", len(paired_inference) == 6, len(paired_inference))
check(
    "dynamic_model_historical_metrics",
    all(
        abs(point_lookup.loc[key, "point_auprc"] - 0.660749) <= 5.1e-7
        and abs(point_lookup.loc[key, "point_auroc"] - 0.559524) <= 5.1e-7
        for key in dynamic_models
    ),
    point_estimates.loc[
        point_estimates["model_key"].isin(dynamic_models),
        ["model_key", "point_auprc", "point_auroc"],
    ].to_dict("records"),
)
check(
    "constant_comparator_metrics",
    all(
        int(point_lookup.loc[key, "unique_score_values"]) == 1
        and abs(point_lookup.loc[key, "point_auprc"] - prevalence) <= 1e-15
        and abs(point_lookup.loc[key, "point_auroc"] - 0.5) <= 1e-15
        for key in constant_models
    ),
    point_estimates.loc[
        point_estimates["model_key"].isin(constant_models),
        ["model_key", "unique_score_values", "point_auprc", "point_auroc"],
    ].to_dict("records"),
)
check(
    "full_no_star_identical_metrics",
    abs(point_lookup.loc["full_ges", "point_auprc"] - point_lookup.loc["no_star_ges", "point_auprc"]) <= 1e-15
    and abs(point_lookup.loc["full_ges", "point_auroc"] - point_lookup.loc["no_star_ges", "point_auroc"]) <= 1e-15,
    {},
)
check(
    "full_combined_identical_metrics",
    abs(point_lookup.loc["full_ges", "point_auprc"] - point_lookup.loc["combined_metadata", "point_auprc"]) <= 1e-15
    and abs(point_lookup.loc["full_ges", "point_auroc"] - point_lookup.loc["combined_metadata", "point_auroc"]) <= 1e-15,
    {},
)
check(
    "historical_values",
    concordance["value_reproduced_at_recorded_precision"].all(),
    concordance.loc[
        ~concordance["value_reproduced_at_recorded_precision"],
        ["result_id", "historical_value", "reproduced_value", "absolute_difference", "tolerance"],
    ].to_dict("records"),
)
check(
    "historical_conclusions",
    concordance["scientific_conclusion_concordant"].all(),
    concordance.loc[
        ~concordance["scientific_conclusion_concordant"],
        ["result_id", "historical_conclusion", "reproduced_conclusion"],
    ].to_dict("records"),
)
check("limitations_complete", len(limitations) == 5, len(limitations))
check(
    "frozen_sources_unchanged",
    sha(EVAL) == EVAL_SHA256
    and sha(NESTED_PACKAGE) == NESTED_PACKAGE_SHA256
    and sha(NESTED_MANIFEST) == NESTED_MANIFEST_SHA256
    and sha(PRIOR_MANIFEST) == PRIOR_MANIFEST_SHA256,
    {},
)

failed = [item for item in checks if not item["passed"]]
if failed:
    raise RuntimeError(
        "QC failed before writing:\n"
        + json.dumps(native(failed), indent=2)
    )


# --------------------------------------------------------------------------------------------------
# 11. VERSIONED WRITES, SIDECARS, MANIFEST, AND FRESH READBACK
# --------------------------------------------------------------------------------------------------

analysis_artifact_columns = (
    KEYS
    + [
        "submitter_classification_tvd",
        "major_submitter_distribution_shift",
        "policy_version",
        "policy_sha256",
    ]
    + score_columns
)
analysis_artifact = analysis[analysis_artifact_columns].copy()

write_csv(P["endpoint_accounting"], endpoint_accounting)
write_parquet(P["analysis_cohort"], analysis_artifact)
write_csv(P["point_estimates"], point_estimates)
write_parquet(P["bootstrap_replicates"], bootstrap_replicates)
write_csv(P["model_intervals"], model_intervals)
write_csv(P["paired_inference"], paired_inference)
write_csv(P["historical_results"], historical_results)
write_csv(P["concordance"], concordance)
write_csv(P["limitations"], limitations)

readback = {
    "endpoint_accounting": len(pd.read_csv(P["endpoint_accounting"])) == 2,
    "analysis_cohort": len(pd.read_parquet(P["analysis_cohort"])) == 37,
    "point_estimates": len(pd.read_csv(P["point_estimates"])) == 9,
    "bootstrap_replicates": len(pd.read_parquet(P["bootstrap_replicates"])) == 2_000,
    "model_intervals": len(pd.read_csv(P["model_intervals"])) == 9,
    "paired_inference": len(pd.read_csv(P["paired_inference"])) == 6,
    "historical_results": len(pd.read_csv(P["historical_results"])) == len(historical_results),
    "concordance": len(pd.read_csv(P["concordance"])) == len(concordance),
    "limitations": len(pd.read_csv(P["limitations"])) == 5,
}
if not all(readback.values()):
    raise RuntimeError(f"Table readback failed: {readback}")

qc_payload = {
    "cell_id": "6C-4I0",
    "package_version": "v1",
    "created_utc": CREATED_UTC,
    "analysis": "nested_scv_37_record_exploratory_materialization",
    "immutable_sources": {
        "stage6b_evaluable": {"path": str(EVAL), "sha256": sha(EVAL)},
        "nested_scv_record_package": {
            "path": str(NESTED_PACKAGE),
            "sha256": sha(NESTED_PACKAGE),
        },
        "nested_scv_manifest": {
            "path": str(NESTED_MANIFEST),
            "sha256": sha(NESTED_MANIFEST),
        },
        "prior_6c4h0_manifest": {
            "path": str(PRIOR_MANIFEST),
            "sha256": sha(PRIOR_MANIFEST),
        },
    },
    "bootstrap": {
        "method": "paired ordinary row bootstrap with replacement",
        "exact_historical_implementation": "rng.integers(0, 37, size=37)",
        "rng": "numpy.random.Generator",
        "bit_generator": type(rng.bit_generator).__name__,
        "seed": SEED,
        "attempts": N_BOOT,
        "valid_replicates": valid,
        "invalid_one_class_replicates": invalid,
        "identical_resamples_across_nine_scores": True,
    },
    "endpoint_accounting": endpoint_accounting.to_dict("records"),
    "checks": checks,
    "table_readback": readback,
    "passed_checks": sum(item["passed"] for item in checks),
    "failed_checks": sum(not item["passed"] for item in checks),
    "decision": (
        "PASS_STAGE6C_NESTED_SCV_37_RECORD_EXPLORATORY_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
}
write_json(P["qc"], qc_payload)

artifact_keys = [
    "endpoint_accounting",
    "analysis_cohort",
    "point_estimates",
    "bootstrap_replicates",
    "model_intervals",
    "paired_inference",
    "historical_results",
    "concordance",
    "limitations",
    "qc",
]

for key in artifact_keys:
    sidecar(P[key])
    if not sidecar_ok(P[key]):
        raise RuntimeError(f"Sidecar verification failed: {P[key]}")


def artifact_record(key: str) -> dict:
    path = P[key]
    record = {
        "artifact_key": key,
        "path": str(path),
        "relative_path": str(path.relative_to(ROOT)),
        "sha256": sha(path),
        "bytes": path.stat().st_size,
        "sidecar_path": str(path.with_name(path.name + ".sha256")),
        "sidecar_verified": sidecar_ok(path),
    }

    if path.suffix == ".csv":
        loaded = pd.read_csv(path)
        record.update(rows=len(loaded), columns=loaded.shape[1])
    elif path.suffix == ".parquet":
        metadata = pq.ParquetFile(path).metadata
        record.update(rows=metadata.num_rows, columns=metadata.num_columns)
    else:
        json.loads(path.read_text(encoding="utf-8"))
        record["json_readback"] = True

    return record


manifest = {
    "cell_id": "6C-4I0",
    "package_version": "v1",
    "notebook_name": NOTEBOOK_NAME,
    "created_utc": CREATED_UTC,
    "authorized_category": "nested_scv_37_record_exploratory_analysis_materialization",
    "immutable_sources": qc_payload["immutable_sources"],
    "analysis_lock": {
        "endpoint": "major_submitter_distribution_shift",
        "threshold": "submitter_classification_tvd >= 0.50",
        "complete_case_rows": EXPECTED["complete_rows"],
        "events": EXPECTED["events"],
        "negatives": EXPECTED["negatives"],
        "censored_rows": EXPECTED["censored"],
        "scores": {
            model_key: {"column": column, "display_name": display_name}
            for model_key, (column, display_name) in SCORES.items()
        },
        "bootstrap_seed": SEED,
        "bootstrap_attempts": N_BOOT,
        "bootstrap_implementation": "rng.integers(0, 37, size=37)",
        "principal_comparators": PRINCIPAL_COMPARATORS,
    },
    "result_summary": {
        "prevalence": prevalence,
        "censoring_fraction": EXPECTED["censoring_fraction"],
        "full_ges_auprc": float(point_lookup.loc["full_ges", "point_auprc"]),
        "full_ges_auroc": float(point_lookup.loc["full_ges", "point_auroc"]),
        "no_star_ges_auprc": float(point_lookup.loc["no_star_ges", "point_auprc"]),
        "no_star_ges_auroc": float(point_lookup.loc["no_star_ges", "point_auroc"]),
        "combined_metadata_auprc": float(
            point_lookup.loc["combined_metadata", "point_auprc"]
        ),
        "combined_metadata_auroc": float(
            point_lookup.loc["combined_metadata", "point_auroc"]
        ),
        "recency_auprc": float(point_lookup.loc["recency", "point_auprc"]),
        "recency_auroc": float(point_lookup.loc["recency", "point_auroc"]),
        "historical_values_reproduced": bool(
            concordance["value_reproduced_at_recorded_precision"].all()
        ),
        "historical_conclusions_concordant": bool(
            concordance["scientific_conclusion_concordant"].all()
        ),
    },
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "artifacts": [artifact_record(key) for key in artifact_keys],
    "scientific_boundary": {
        "high_rigor_endpoint_analyzed": False,
        "high_rigor_endpoint_reason": "zero_observable_positive_events",
        "frozen_inputs_modified": False,
        "scores_refit_or_recalibrated": False,
        "thresholds_or_weights_changed": False,
        "outcome_definition_changed": False,
        "policy_changed": False,
        "linkage_decisions_changed": False,
        "row_order_changed": False,
        "cohort_membership_changed": False,
        "experiment_2_started": False,
        "interpretation": (
            "The 37-record endpoint is technically estimable but highly exploratory. Full GES, "
            "no-star GES, combined metadata, and recency have identical descriptive AUPRC/AUROC. "
            "The remaining low-cardinality comparators are constant. With 99.9445% censoring, "
            "these results cannot establish generalizable superiority, calibration, clinical "
            "utility, or RAG safety."
        ),
    },
    "decision": (
        "PASS_STAGE6C_NESTED_SCV_37_RECORD_EXPLORATORY_RESULT_CATEGORY_"
        "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
    ),
    "next_authorized_category": "leave_one_gene_out_validation_materialization",
}
manifest_hash = write_json(P["manifest"], manifest)
sidecar(P["manifest"])

# Fresh package and immutable-source reverification.
manifest_readback = json.loads(P["manifest"].read_text(encoding="utf-8"))

if not sidecar_ok(P["manifest"]):
    raise RuntimeError("Manifest sidecar verification failed.")
if manifest_readback["decision"] != manifest["decision"]:
    raise RuntimeError("Manifest decision readback mismatch.")
if manifest_readback["scientific_boundary"]["experiment_2_started"] is not False:
    raise RuntimeError("Experiment 2 boundary failed.")

for artifact in manifest_readback["artifacts"]:
    path = Path(artifact["path"])
    if sha(path) != artifact["sha256"] or not sidecar_ok(path):
        raise RuntimeError(f"Final artifact verification failed: {path}")

if (
    sha(EVAL) != EVAL_SHA256
    or sha(NESTED_PACKAGE) != NESTED_PACKAGE_SHA256
    or sha(NESTED_MANIFEST) != NESTED_MANIFEST_SHA256
    or sha(PRIOR_MANIFEST) != PRIOR_MANIFEST_SHA256
):
    raise RuntimeError("An immutable source changed during Cell 6C-4I0.")


# --------------------------------------------------------------------------------------------------
# 12. CONTROLLED FINAL OUTPUT
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 170)
print(
    "STAGE 6C STEP 4I — CELL 6C-4I0 — "
    "37-RECORD NESTED-SCV EXPLORATORY RESULT CATEGORY"
)
print("=" * 170)
print(f"Stage 6B evaluable hash                 : PASS ({sha(EVAL)})")
print(f"Nested-SCV package hash                 : PASS ({sha(NESTED_PACKAGE)})")
print(f"Nested-SCV manifest hash                : PASS ({sha(NESTED_MANIFEST)})")
print(f"Prior Cell 6C-4H0 manifest              : PASS ({sha(PRIOR_MANIFEST)})")
print(
    "High-rigor contradiction endpoint      : NOT ESTIMABLE "
    f"({hr_events} events, {hr_negatives:,} negatives, {hr_censored:,} censored)"
)
print(
    f"Distribution endpoint                   : {EXPECTED['complete_rows']} rows | "
    f"{EXPECTED['events']} events | {EXPECTED['negatives']} negatives"
)
print(
    f"Distribution censoring                  : {EXPECTED['censored']:,}/"
    f"{EXPECTED['total_rows']:,} ({100 * EXPECTED['censoring_fraction']:.4f}%)"
)
print(f"Nine-score point estimates              : PASS ({len(point_estimates)}/9)")
print(f"Bootstrap attempts                      : {N_BOOT:,}")
print(f"Valid / invalid replicates              : {valid:,} / {invalid:,}")
print(f"Bootstrap elapsed                       : {bootstrap_elapsed / 60:.2f} minutes")
print(f"Model intervals                         : PASS ({len(model_intervals)}/9)")
print(f"Principal paired comparisons            : PASS ({len(paired_inference)}/6)")
print(
    f"Historical recorded values              : PASS "
    f"({int(concordance.value_reproduced_at_recorded_precision.sum())}/{len(concordance)})"
)
print(
    f"Historical scientific conclusions       : PASS "
    f"({int(concordance.scientific_conclusion_concordant.sum())}/{len(concordance)})"
)
print(f"Fresh QC                                : PASS ({qc_payload['passed_checks']}/{len(checks)})")

for label, key in [
    ("Endpoint accounting", "endpoint_accounting"),
    ("37-record analysis cohort", "analysis_cohort"),
    ("Point estimates", "point_estimates"),
    ("Bootstrap replicates", "bootstrap_replicates"),
    ("Model intervals", "model_intervals"),
    ("Paired exploratory inference", "paired_inference"),
    ("Historical results", "historical_results"),
    ("Concordance table", "concordance"),
    ("Limitations", "limitations"),
    ("QC", "qc"),
    ("Manifest", "manifest"),
]:
    print(f"{label:40s}: {P[key]}")

print(f"Manifest SHA-256                        : {manifest_hash}")

print("\nNESTED-SCV ENDPOINT ACCOUNTING")
print(endpoint_accounting.to_string(index=False))

print("\n37-RECORD NINE-SCORE MODEL INTERVALS")
print(
    model_intervals[
        [
            "model",
            "unique_score_values",
            "point_auprc",
            "auprc_ci_lower",
            "auprc_ci_upper",
            "point_auprc_minus_prevalence",
            "point_auroc",
            "auroc_ci_lower",
            "auroc_ci_upper",
            "point_auroc_minus_0_50",
            "valid_bootstrap_replicates",
            "invalid_one_class_replicates",
        ]
    ].to_string(index=False)
)

print("\nFULL-GES PRINCIPAL-COMPARATOR PAIRED EXPLORATORY INFERENCE")
print(paired_inference.to_string(index=False))

print("\nSCIENTIFIC INTERPRETATION BOUNDARY")
print(
    "Only 37 of 66,636 records are evaluable and 99.9445% are censored. "
    "Full GES, no-star GES, combined metadata, and recency have identical descriptive "
    "discrimination in this tiny subset. The high-rigor contradiction endpoint has zero "
    "positive events and was not analyzed. These estimates do not establish generalizable "
    "superiority, calibration, clinical utility, or RAG safety."
)

print("\nCELL DECISION")
print(
    "PASS_STAGE6C_NESTED_SCV_37_RECORD_EXPLORATORY_RESULT_CATEGORY_"
    "MATERIALIZED_CHECKSUM_PROTECTED_AND_FRESHLY_REVERIFIED"
)
print(
    "The seventh of eight Stage 6C result categories is independently materialized. "
    "The next authorized category is leave-one-gene-out validation materialization. "
    "Experiment 2 has not started."
)
print("=" * 170)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Use this Colab notebook file name: GES_Stage6C_Cell_6C_4I0_Nested_SCV_37_Record_Exploratory_Materialization.ipynb

Preparing the frozen 37-record nested-SCV complete-case endpoint: 21 events and 16 negatives
  Completed 250/2,000 replicates | valid 250
  Completed 500/2,000 replicates | valid 500
  Completed 750/2,000 replicates | valid 750
  Completed 1,000/2,000 replicates | valid 1,000
  Completed 1,250/2,000 replicates | valid 1,250
  Completed 1,500/2,000 replicates | valid 1,500
  Completed 1,750/2,000 replicates | valid 1,750
  Completed 2,000/2,000 replicates | valid 2,000


RuntimeError: QC failed before writing:
[
  {
    "check_name": "distribution_censoring_fraction",
    "passed": false,
    "details": 0.999444744582508
  }
]